In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
import gc
import glob
import warnings
import numpy as np
import xarray as xr
import cftime
import numba as nb  # 🚀 涡轮增压引擎

from tqdm.auto import tqdm
from multiprocessing import Pool

warnings.filterwarnings("ignore")

# ============================================================
# Paths & Config
# ============================================================
ROOT = "/mnt/soclim0/public_data/weiji"

PATHS = {
    "BWCN": os.path.join(ROOT, "BWCN"),
    "B2000WCN": os.path.join(ROOT, "B2000WCN001002"),
    "NOCOUPL": os.path.join(ROOT, "B2000WCN_NOCOUPL001002"),
    "MERRA2": os.path.join(ROOT, "MERRA2_Processed"),
}

FW_TXT_PATHS = {
    "BWCN": os.path.join(PATHS["BWCN"], "BWCN_final_warming_date.txt"),
    "B2000WCN": os.path.join(PATHS["B2000WCN"], "B2000WCN_final_warming_date.txt"),
    "NOCOUPL": os.path.join(PATHS["NOCOUPL"], "B2000WCN_NOCOUPL_final_warming_date.txt"),
    "MERRA2": os.path.join(PATHS["MERRA2"], "MERRA2_final_warming_date.txt"),
}

CLIM_NC_PATHS = {
    "BWCN": {"U": os.path.join(PATHS["BWCN"], "BWCN_U_climatology_1to100hPa.nc"), "T": os.path.join(PATHS["BWCN"], "BWCN_T_climatology_1to100hPa.nc"), "O3": os.path.join(PATHS["BWCN"], "BWCN_O3_climatology_1to100hPa.nc"), "EPFLUX": os.path.join(PATHS["BWCN"], "BWCN_EPFLUX_climatology_1to100hPa.nc")},
    "B2000WCN": {"U": os.path.join(PATHS["B2000WCN"], "B2000WCN_U_climatology_1to100hPa.nc"), "T": os.path.join(PATHS["B2000WCN"], "B2000WCN_T_climatology_1to100hPa.nc"), "O3": os.path.join(PATHS["B2000WCN"], "B2000WCN_O3_climatology_1to100hPa.nc"), "EPFLUX": os.path.join(PATHS["B2000WCN"], "B2000WCN_EPFLUX_climatology_1to100hPa.nc")},
    "NOCOUPL": {"U": os.path.join(PATHS["NOCOUPL"], "B2000WCN_NOCOUPL_U_climatology_1to100hPa.nc"), "T": os.path.join(PATHS["NOCOUPL"], "B2000WCN_NOCOUPL_T_climatology_1to100hPa.nc"), "O3": os.path.join(PATHS["NOCOUPL"], "B2000WCN_NOCOUPL_O3_climatology_1to100hPa.nc"), "EPFLUX": os.path.join(PATHS["NOCOUPL"], "B2000WCN_NOCOUPL_EPFLUX_climatology_1to100hPa.nc")},
    "MERRA2": {"U": os.path.join(PATHS["MERRA2"], "MERRA2_U_climatology_1to100hPa.nc"), "T": os.path.join(PATHS["MERRA2"], "MERRA2_T_climatology_1to100hPa.nc"), "O3": os.path.join(PATHS["MERRA2"], "MERRA2_O3_climatology_1to100hPa.nc"), "EPFLUX": os.path.join(PATHS["MERRA2"], "MERRA2_EPFLUX_climatology_1to100hPa.nc")},
}

CORES = 2
DROP_YEARS_MODEL = {104}
N_PREV = 91
TARGET_PLEV_HPA = np.array([100.0, 70.0, 50.0, 40.0, 30.0, 20.0, 10.0, 7.0, 5.0, 4.0, 3.0, 2.0, 1.0], dtype=np.float64)
TARGET_PLEV_PA = TARGET_PLEV_HPA * 100.0

FW_LAT, FW_PLEV_HPA, FW_THRESHOLD, FW_MAX_CONSEC_WESTERLY = 60.0, 50.0, 7.0, 10
MOL_MASS_RATIO = 28.9644 / 47.9982

VAR_SPECS = {
    "U": {"var_name": "U", "lat_mode": "global", "scale_factor_model": 1.0, "scale_factor_merra": 1.0, "plot_unit": "m/s", "long_name": "Zonal wind"},
    "T": {"var_name": "T", "lat_mode": "global", "scale_factor_model": 1.0, "scale_factor_merra": 1.0, "plot_unit": "K", "long_name": "Temperature"},
    "O3": {"var_name": "O3", "lat_mode": "global", "scale_factor_model": 1e6, "scale_factor_merra": MOL_MASS_RATIO * 1e6, "plot_unit": "ppmv", "long_name": "Ozone mixing ratio"},
    "EPFLUX": {"var_name": "Fz", "lat_mode": None, "scale_factor_model": 1.0, "scale_factor_merra": 1.0, "plot_unit": "native", "long_name": "Vertical EP flux (Fz)"},
}

MONTH_DAY_365 = [(m, d) for m, nd in enumerate([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31], 1) for d in range(1, nd + 1)]

# ============================================================
# Numba Fast Interp
# ============================================================
@nb.njit(parallel=True)
def fast_logp_interp_flat(p_flat, v_flat, target_p):
    n_profs = p_flat.shape[0]
    nlev = p_flat.shape[1]
    ntarg = len(target_p)
    out = np.empty((n_profs, ntarg), dtype=np.float32)
    ltp = np.log(target_p)
    
    for i in nb.prange(n_profs):
        p, v = p_flat[i], v_flat[i]
        valid = 0
        for k in range(nlev):
            if p[k] > 0 and not np.isnan(p[k]) and not np.isnan(v[k]): valid += 1
                
        if valid < 2:
            for t_idx in range(ntarg): out[i, t_idx] = np.nan
            continue
            
        p_valid = np.empty(valid, dtype=np.float64)
        v_valid = np.empty(valid, dtype=np.float64)
        idx = 0
        for k in range(nlev):
            if p[k] > 0 and not np.isnan(p[k]) and not np.isnan(v[k]):
                p_valid[idx], v_valid[idx] = p[k], v[k]
                idx += 1
                
        order = np.argsort(p_valid)
        p_sorted, v_sorted = p_valid[order], v_valid[order]
        lp = np.log(p_sorted)
        out_i = np.interp(ltp, lp, v_sorted)
        
        slope_top = (v_sorted[1] - v_sorted[0]) / (lp[1] - lp[0])
        slope_bottom = (v_sorted[-1] - v_sorted[-2]) / (lp[-1] - lp[-2])
        
        for t_idx in range(ntarg):
            if ltp[t_idx] < lp[0]: out[i, t_idx] = v_sorted[0] + slope_top * (ltp[t_idx] - lp[0])
            elif ltp[t_idx] > lp[-1]: out[i, t_idx] = v_sorted[-1] + slope_bottom * (ltp[t_idx] - lp[-1])
            else: out[i, t_idx] = out_i[t_idx]
    return out

# ============================================================
# Basic helpers
# ============================================================
def get_ymd_from_time_value(tt):
    if hasattr(tt, "year") and hasattr(tt, "month"): return int(tt.year), int(tt.month), int(tt.day)
    y, m, d = np.datetime_as_string(tt, unit="D").split("-")
    return int(y), int(m), int(d)

def parse_file_year(path):
    m_all = re.findall(r'(?<!\d)(\d{4})(?!\d)', os.path.basename(path))
    return int(m_all[-1]) if m_all else None

def detect_coord_name(ds, candidates):
    for c in candidates:
        if c in ds.coords or c in ds.dims: return c
    return None

def detect_main_var(ds, preferred=None):
    if preferred is not None and preferred in ds.data_vars: return preferred
    for v in ds.data_vars:
        if "time" in ds[v].dims: return v
    raise ValueError("Cannot detect main variable.")

def rebuild_time_from_filename_year(ds, file_year):
    new_t = [cftime.DatetimeNoLeap(file_year, *get_ymd_from_time_value(tt)[1:]) for tt in ds["time"].values]
    return ds.assign_coords(time=("time", new_t))

def noleap_doy_from_month_day(month, day):
    return sum([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31][:month - 1]) + day

def drop_feb29_and_get_time_parts(time_vals):
    keep_idx, years, months, days = [], [], [], []
    for i, tt in enumerate(time_vals):
        y, m, d = get_ymd_from_time_value(tt)
        if not (m == 2 and d == 29):
            keep_idx.append(i); years.append(y); months.append(m); days.append(d)
    return np.array(keep_idx, dtype=int), np.array(years, dtype=int), np.array(months, dtype=int), np.array(days, dtype=int)

def build_clim_dataarrays(clim_target, plev_hpa, lat_vals=None, lon_vals=None):
    if lat_vals is not None and lon_vals is not None:
        arr_365 = np.transpose(clim_target, (1, 2, 3, 0)).astype(np.float32)
        dims, dims_os = ("plev", "lat", "lon", "dayofyear"), ("plev", "lat", "lon", "time_index")
        coords_jd = {"plev": plev_hpa.astype(np.float32), "lat": lat_vals.astype(np.float32), "lon": lon_vals.astype(np.float32), "dayofyear": np.arange(1, 366)}
        coords_os = {"plev": plev_hpa.astype(np.float32), "lat": lat_vals.astype(np.float32), "lon": lon_vals.astype(np.float32), "time_index": np.arange(365)}
    else:
        arr_365 = np.transpose(clim_target, (1, 0)).astype(np.float32)
        dims, dims_os = ("plev", "dayofyear"), ("plev", "time_index")
        coords_jd = {"plev": plev_hpa.astype(np.float32), "dayofyear": np.arange(1, 366)}
        coords_os = {"plev": plev_hpa.astype(np.float32), "time_index": np.arange(365)}

    arr_os = np.concatenate([arr_365[..., -N_PREV:], arr_365[..., :-N_PREV]], axis=-1).astype(np.float32)
    clim_jd = xr.DataArray(arr_365, coords=coords_jd, dims=dims, name="clim_jandec")
    clim_os = xr.DataArray(arr_os, coords=coords_os, dims=dims_os, name="clim_octsep")
    return clim_jd, clim_os

def save_single_source_climatology(exp_name, var_key, clim_jd, clim_os, long_name, plot_unit):
    ds_out = xr.Dataset(data_vars={"clim_jandec": clim_jd, "clim_octsep": clim_os})
    ds_out.attrs.update({"experiment": exp_name, "variable": var_key, "long_name": long_name, "plot_unit": plot_unit, "pressure_range": "1-100 hPa only"})
    out_nc = CLIM_NC_PATHS[exp_name][var_key]
    ds_out.to_netcdf(out_nc)
    print(f"  Saved source climatology: {out_nc}")

def logp_interp_extrap_1d(p_prof, v_prof, target_p):
    p, v, tp = np.asarray(p_prof, dtype=np.float64), np.asarray(v_prof, dtype=np.float64), np.asarray(target_p, dtype=np.float64)
    mask = np.isfinite(p) & np.isfinite(v) & (p > 0)
    p, v = p[mask], v[mask]
    if p.size < 2: return np.full(tp.shape, np.nan, dtype=np.float64)
    order = np.argsort(p)
    p, v = p[order], v[order]
    lp, ltp = np.log(p), np.log(tp)
    out = np.interp(ltp, lp, v)
    mask_top = ltp < lp[0]
    if np.any(mask_top): out[mask_top] = v[0] + ((v[1] - v[0]) / (lp[1] - lp[0])) * (ltp[mask_top] - lp[0])
    mask_bottom = ltp > lp[-1]
    if np.any(mask_bottom): out[mask_bottom] = v[-1] + ((v[-1] - v[-2]) / (lp[-1] - lp[-2])) * (ltp[mask_bottom] - lp[-1])
    return out

def interp_fixed_pressure_clim_to_target(clim_var_native, native_plev_hpa, target_plev_hpa):
    native_pa, target_pa = np.asarray(native_plev_hpa, dtype=np.float64) * 100.0, np.asarray(target_plev_hpa, dtype=np.float64) * 100.0
    out = np.full((365, len(target_plev_hpa)), np.nan, dtype=np.float32)
    for d in range(365): out[d, :] = logp_interp_extrap_1d(native_pa, clim_var_native[d, :], target_pa).astype(np.float32)
    return out

def list_model_yearly_files(exp_name, var_key):
    files = sorted(glob.glob(os.path.join(PATHS[exp_name], var_key, "*.nc")))
    valid = [f for f in files if parse_file_year(f) is not None and parse_file_year(f) not in DROP_YEARS_MODEL]
    if not valid: raise FileNotFoundError(f"No valid files for {exp_name}/{var_key}")
    return valid

def scan_merra_files(var_key):
    subdir_map = {"U": "U", "T": "T", "O3": "O3", "EPFLUX": "EPflux_daily"}
    files = sorted(glob.glob(os.path.join(PATHS["MERRA2"], subdir_map[var_key], "*.nc")))
    if not files: raise FileNotFoundError(f"No MERRA2 files for {var_key}")
    return files

# ============================================================
# WACCM 4D climatology workers (U, T, O3)
# ============================================================
def process_model_clim_year_safely(path, var_key, spec, target_sum_var, target_count):
    year_int = parse_file_year(path)
    if year_int is None or year_int in DROP_YEARS_MODEL: return None, None

    ds = xr.open_dataset(path, decode_times=True)
    ds = rebuild_time_from_filename_year(ds, year_int)

    vname = spec["var_name"]
    da, ps = ds[vname], ds["PS"]

    lat_name = detect_coord_name(ds, ["lat", "latitude"])
    lon_name = detect_coord_name(ds, ["lon", "longitude"])
    lat_vals, lon_vals = ds[lat_name].values, ds[lon_name].values

    da = da.transpose("time", "lev", lat_name, lon_name)
    ps = ps.transpose("time", lat_name, lon_name)

    hyam, hybm, p0 = ds["hyam"].values.astype(np.float32), ds["hybm"].values.astype(np.float32), np.float32(ds["P0"].values)
    keep_idx, _, months, days = drop_feb29_and_get_time_parts(ds["time"].values)
    
    scale = np.float32(spec["scale_factor_model"])
    ntarg = len(TARGET_PLEV_PA)
    doy_kept = np.array([noleap_doy_from_month_day(m, d) for m, d in zip(months, days)], dtype=int)

    for i, t_idx in enumerate(keep_idx):
        da_day = da.isel(time=t_idx).values * scale
        ps_day = ps.isel(time=t_idx).values
        nlev, nlat, nlon = da_day.shape
        p_native_day = hyam[:, None, None] * p0 + hybm[:, None, None] * ps_day[None, :, :]
        
        var_flat = da_day.transpose(1, 2, 0).reshape(-1, nlev).astype(np.float64)
        p_flat = p_native_day.transpose(1, 2, 0).reshape(-1, nlev).astype(np.float64)
        out_flat = fast_logp_interp_flat(p_flat, var_flat, TARGET_PLEV_PA)
        
        var_interp_3d = out_flat.reshape(nlat, nlon, ntarg).transpose(2, 0, 1)
        j = doy_kept[i] - 1
        target_sum_var[j, ...] += var_interp_3d
        target_count[j] += 1

    ds.close()
    gc.collect()
    return lat_vals, lon_vals

def accumulate_model_hybrid_climatology(exp_name, var_key, spec):
    files = list_model_yearly_files(exp_name, var_key)
    print(f"  -> {exp_name} {var_key}: {len(files)} yearly files (Turbo Mode)")

    ds0 = xr.open_dataset(files[0])
    lat_name = detect_coord_name(ds0, ["lat", "latitude"])
    lon_name = detect_coord_name(ds0, ["lon", "longitude"])
    nlat, nlon = ds0.sizes[lat_name], ds0.sizes[lon_name]
    ds0.close()
    
    sum_var = np.zeros((365, len(TARGET_PLEV_HPA), nlat, nlon), dtype=np.float64)
    count = np.zeros(365, dtype=np.int32)
    lat_vals_global, lon_vals_global = None, None

    for f in tqdm(files, desc=f"{exp_name}-{var_key}", ncols=100):
        lat_v, lon_v = process_model_clim_year_safely(f, var_key, spec, sum_var, count)
        if lat_v is not None: lat_vals_global, lon_vals_global = lat_v, lon_v

    clim_target = (sum_var / count[:, None, None, None]).astype(np.float32)
    jd, os_ds = build_clim_dataarrays(clim_target, TARGET_PLEV_HPA, lat_vals=lat_vals_global, lon_vals=lon_vals_global)
    
    del sum_var, count
    gc.collect()
    return jd, os_ds

# ============================================================
# MERRA2 4D climatology workers (U, T, O3)
# ============================================================
def select_exact_target_levels(da, target_plev_hpa):
    src = da["plev"].values.astype(np.float64)
    sel_idx = [int(np.argmin(np.abs(src - t))) for t in target_plev_hpa]
    return da.isel(plev=sel_idx).assign_coords(plev=("plev", target_plev_hpa))

def process_merra_clim_file_safely(path, var_key, spec, sum_doy, count_arr):
    ds = xr.open_dataset(path, decode_times=True)
    vname = detect_main_var(ds, preferred=spec["var_name"])
    da = ds[vname]

    lat_name = detect_coord_name(ds, ["lat", "latitude"])
    lon_name = detect_coord_name(ds, ["lon", "longitude"])
    p_name = detect_coord_name(ds, ["plev", "lev", "level"])

    da = da.rename({p_name: "plev"})
    da = select_exact_target_levels(da, TARGET_PLEV_HPA)

    lat_vals = da[lat_name].values
    lon_vals = da[lon_name].values if lon_name else None

    if lon_name: da = da.transpose("time", "plev", lat_name, lon_name)
    else: da = da.transpose("time", "plev", lat_name)

    da = da.astype(np.float32) * np.float32(spec["scale_factor_merra"])
    keep_idx, years, _, _ = drop_feb29_and_get_time_parts(da["time"].values)
    da = da.isel(time=keep_idx)

    for y in np.unique(years):
        yy = (years == y)
        if yy.sum() != 365: continue
        sum_doy += da.isel(time=yy).values.astype(np.float64)
        count_arr += 1

    ds.close()
    gc.collect()
    return lat_vals, lon_vals

def accumulate_merra_climatology(var_key, spec):
    files = scan_merra_files(var_key)
    print(f"  -> MERRA2 {var_key}: {len(files)} files")

    ds0 = xr.open_dataset(files[0])
    lat_name, lon_name = detect_coord_name(ds0, ["lat", "latitude"]), detect_coord_name(ds0, ["lon", "longitude"])
    nlat, nlon = ds0.sizes[lat_name], ds0.sizes[lon_name] if lon_name else None
    ds0.close()

    sum_doy = np.zeros((365, len(TARGET_PLEV_HPA), nlat, nlon), dtype=np.float64) if nlon else np.zeros((365, len(TARGET_PLEV_HPA), nlat), dtype=np.float64)
    count = np.zeros(365, dtype=np.int32)
    lat_vals, lon_vals = None, None

    for f in tqdm(files, desc=f"MERRA2-{var_key}", ncols=100):
        lat_v, lon_v = process_merra_clim_file_safely(f, var_key, spec, sum_doy, count)
        if lat_vals is None: lat_vals, lon_vals = lat_v, lon_v

    clim_target = (sum_doy / count[:, None, None, None]).astype(np.float32) if nlon else (sum_doy / count[:, None, None]).astype(np.float32)
    return build_clim_dataarrays(clim_target, TARGET_PLEV_HPA, lat_vals=lat_vals, lon_vals=lon_vals)

# ============================================================
# 2D EPFLUX climatology workers (All Models + MERRA2)
# ============================================================
def find_epflux_file(exp_name):
    files = sorted(glob.glob(os.path.join(PATHS[exp_name], "EPflux_daily", "*.nc")))
    if not files: raise FileNotFoundError(f"No EPFLUX file for {exp_name}")
    return files[0]

def accumulate_model_epflux_climatology(exp_name, spec):
    path = find_epflux_file(exp_name)
    print(f"  -> {exp_name} EPFLUX: {os.path.basename(path)}")

    ds = xr.open_dataset(path, decode_times=True)
    vname = detect_main_var(ds, preferred=spec["var_name"])
    da = ds[vname]

    p_name = detect_coord_name(ds, ["plev", "lev", "level"])
    da = da.rename({p_name: "plev"}).transpose("time", "plev")

    src = da["plev"].values.astype(np.float64)
    if np.nanmax(src) > 2000:
        src /= 100.0
        da = da.assign_coords(plev=("plev", src))

    da = da.isel(plev=((src >= 1.0) & (src <= 100.0)))
    native_plev_hpa = da["plev"].values.astype(np.float64)

    keep_idx, years, _, _ = drop_feb29_and_get_time_parts(da["time"].values)
    da = da.isel(time=keep_idx)

    sum_doy = np.zeros((365, len(native_plev_hpa)), dtype=np.float64)
    count = np.zeros(365, dtype=np.int32)

    for y in np.unique(years):
        yy = (years == y)
        if yy.sum() == 365:
            sum_doy += da.isel(time=yy).values.astype(np.float64)
            count += 1

    ds.close()
    return sum_doy, count, native_plev_hpa

def finalize_model_epflux_climatology(sum_doy, count, native_plev_hpa):
    clim_native = sum_doy / count[:, None]
    clim_target = interp_fixed_pressure_clim_to_target(clim_native, native_plev_hpa, TARGET_PLEV_HPA)
    return build_clim_dataarrays(clim_target, TARGET_PLEV_HPA)

def accumulate_merra_epflux_climatology(spec):
    files = scan_merra_files("EPFLUX")
    print(f"  -> MERRA2 EPFLUX: {len(files)} files")

    # Read first file to get structure
    ds0 = xr.open_dataset(files[0])
    p_name = detect_coord_name(ds0, ["plev", "lev", "level"])
    src0 = ds0[p_name].values.astype(np.float64)
    if np.nanmax(src0) > 2000: src0 /= 100.0
    native_mask = (src0 >= 1.0) & (src0 <= 100.0)
    native_plev_hpa = src0[native_mask]
    ds0.close()

    sum_doy = np.zeros((365, len(native_plev_hpa)), dtype=np.float64)
    count = np.zeros(365, dtype=np.int32)

    for f in tqdm(files, desc="MERRA2-EPFLUX", ncols=100):
        ds = xr.open_dataset(f, decode_times=True)
        vname = detect_main_var(ds, preferred=spec["var_name"])
        da = ds[vname].rename({detect_coord_name(ds, ["plev", "lev", "level"]): "plev"}).transpose("time", "plev")
        
        src = da["plev"].values.astype(np.float64)
        if np.nanmax(src) > 2000:
            src /= 100.0
            da = da.assign_coords(plev=("plev", src))
            
        da = da.isel(plev=((src >= 1.0) & (src <= 100.0)))
        keep_idx, years, _, _ = drop_feb29_and_get_time_parts(da["time"].values)
        da = da.isel(time=keep_idx)
        
        for y in np.unique(years):
            yy = (years == y)
            if yy.sum() == 365:
                sum_doy += da.isel(time=yy).values.astype(np.float64)
                count += 1
                
        ds.close()
        gc.collect()

    clim_native = sum_doy / count[:, None]
    clim_target = interp_fixed_pressure_clim_to_target(clim_native, native_plev_hpa, TARGET_PLEV_HPA)
    return build_clim_dataarrays(clim_target, TARGET_PLEV_HPA)


# ============================================================
# FW helpers
# ============================================================
def find_fw_for_one_year(vals_daily):
    vals_jj = np.asarray(vals_daily[:181], dtype=np.float64)
    for day_idx in range(len(vals_jj)):
        if not np.isfinite(vals_jj[day_idx]): continue
        if vals_jj[day_idx] < FW_THRESHOLD:
            has_long_westerly = False
            for i in range(day_idx + 1, max(day_idx + 1, len(vals_jj) - FW_MAX_CONSEC_WESTERLY + 1)):
                seg = vals_jj[i:i + FW_MAX_CONSEC_WESTERLY]
                if len(seg) < FW_MAX_CONSEC_WESTERLY: break
                if np.all(seg > FW_THRESHOLD):
                    has_long_westerly = True; break
            if not has_long_westerly:
                doy = day_idx + 1
                return doy, MONTH_DAY_365[doy - 1][0], MONTH_DAY_365[doy - 1][1]
    return np.nan, np.nan, np.nan

def write_fw_txt(out_txt, fw_records):
    with open(out_txt, "w", encoding="utf-8") as f:
        f.write("# Final warming dates\n# Columns: year  fw_doy  fw_month  fw_day\n")
        for r in fw_records:
            f.write(f"{r['year']:04d} {'nan' if np.isnan(r['fw_doy']) else int(r['fw_doy'])} {'nan' if np.isnan(r['fw_month']) else int(r['fw_month'])} {'nan' if np.isnan(r['fw_day']) else int(r['fw_day'])}\n")
    print(f"  FW txt saved: {out_txt}")

def process_model_fw_year(path):
    year_int = parse_file_year(path)
    if year_int is None or year_int in DROP_YEARS_MODEL: return None
    ds = xr.open_dataset(path, decode_times=True)
    da, ps = ds["U"], ds["PS"]
    lat_name, lon_name = detect_coord_name(ds, ["lat", "latitude"]), detect_coord_name(ds, ["lon", "longitude"])
    da, ps = da.sel({lat_name: FW_LAT}, method="nearest"), ps.sel({lat_name: FW_LAT}, method="nearest")
    if lon_name and lon_name in da.dims: da, ps = da.mean(dim=lon_name), ps.mean(dim=lon_name)
    hyam, hybm, p0 = ds["hyam"].values.astype(np.float64), ds["hybm"].values.astype(np.float64), float(ds["P0"].values)
    p_native = hyam[None, :] * p0 + hybm[None, :] * ps.values[:, None]
    u50 = np.array([logp_interp_extrap_1d(p_native[i, :], da.values[i, :], np.array([FW_PLEV_HPA * 100.0], dtype=np.float64))[0] for i in range(da.sizes["time"])])
    doy, mm, dd = find_fw_for_one_year(u50)
    ds.close()
    return {"year": int(year_int), "fw_doy": doy, "fw_month": mm, "fw_day": dd}

def compute_and_write_fw_model(exp_name):
    files = list_model_yearly_files(exp_name, "U")
    with Pool(processes=CORES) as pool:
        res = list(tqdm(pool.imap(process_model_fw_year, files), total=len(files), desc=f"{exp_name}-FW", ncols=100))
    write_fw_txt(FW_TXT_PATHS[exp_name], sorted([r for r in res if r], key=lambda x: x["year"]))

def process_merra_fw_file(path):
    ds = xr.open_dataset(path, decode_times=True)
    da = ds[detect_main_var(ds, preferred="U")]
    lat_name, lon_name = detect_coord_name(ds, ["lat", "latitude"]), detect_coord_name(ds, ["lon", "longitude"])
    da = da.rename({detect_coord_name(ds, ["plev", "lev", "level"]): "plev"}).sel({lat_name: FW_LAT}, method="nearest").sel(plev=FW_PLEV_HPA)
    if lon_name and lon_name in da.dims: da = da.mean(dim=lon_name)
    keep_idx, years, _, _ = drop_feb29_and_get_time_parts(da["time"].values)
    vals = da.values[keep_idx]
    out = []
    for y in np.unique(years):
        if (years == y).sum() == 365:
            doy, mm, dd = find_fw_for_one_year(vals[years == y])
            out.append({"year": int(y), "fw_doy": doy, "fw_month": mm, "fw_day": dd})
    ds.close()
    return out

def compute_and_write_fw_merra():
    files = scan_merra_files("U")
    with Pool(processes=min(CORES, len(files))) as pool:
        res = list(tqdm(pool.imap(process_merra_fw_file, files), total=len(files), desc="MERRA2-FW", ncols=100))
    fw_records = sorted([r for recs in res for r in recs], key=lambda x: x["year"])
    write_fw_txt(FW_TXT_PATHS["MERRA2"], fw_records)

# ============================================================
# Build one variable
# ============================================================
def build_and_save_one_variable(var_key):
    spec = VAR_SPECS[var_key]
    print("\n" + "=" * 90 + f"\n[BUILD] {var_key}\n" + "=" * 90)

    if var_key == "EPFLUX":
        # 🚨 EPFLUX (2D) 专属通道，所有模型和 MERRA2 均走此路线
        m_jd, m_os = accumulate_merra_epflux_climatology(spec)
        save_single_source_climatology("MERRA2", var_key, m_jd, m_os, spec["long_name"], spec["plot_unit"])

        for exp in ["BWCN", "B2000WCN", "NOCOUPL"]:
            e_sum, e_count, e_plev = accumulate_model_epflux_climatology(exp, spec)
            e_jd, e_os = finalize_model_epflux_climatology(e_sum, e_count, e_plev)
            save_single_source_climatology(exp, var_key, e_jd, e_os, spec["long_name"], spec["plot_unit"])
    else:
        # 🚨 U, T, O3 (4D) 专属通道
        clim_merra_jd, clim_merra_os = accumulate_merra_climatology(var_key, spec)
        save_single_source_climatology("MERRA2", var_key, clim_merra_jd, clim_merra_os, spec["long_name"], spec["plot_unit"])

        for exp in ["BWCN", "B2000WCN", "NOCOUPL"]:
            clim_jd, clim_os = accumulate_model_hybrid_climatology(exp, var_key, spec)
            save_single_source_climatology(exp, var_key, clim_jd, clim_os, spec["long_name"], spec["plot_unit"])
            
    gc.collect()

# ============================================================
# Run directly
# ============================================================
if __name__ == "__main__":
    # 你可以自由决定跑哪几个，现在它是完全稳定、安全的。
    for var_key in ["U", "T", "O3", "EPFLUX"]:
        build_and_save_one_variable(var_key)

    print("\n" + "=" * 90 + "\n[BUILD] Final Warming date txt files\n" + "=" * 90)
    compute_and_write_fw_merra()
    compute_and_write_fw_model("BWCN")
    compute_and_write_fw_model("B2000WCN")
    compute_and_write_fw_model("NOCOUPL")
    print("\nAll tasks completed successfully.")

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import glob
import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

# ============================================================
# Paths
# ============================================================
ROOT = "/mnt/soclim0/public_data/weiji"
OUT_ROOT = "/home/weiji/restart_exam/code/20260226B2000WCN_BWCN_MERRA2_COMPARE/RESULT"
PLOT_OUT = os.path.join(OUT_ROOT, "plots")
os.makedirs(PLOT_OUT, exist_ok=True)

PATHS = {
    "BWCN": os.path.join(ROOT, "BWCN"),
    "B2000WCN": os.path.join(ROOT, "B2000WCN001002"),
    "NOCOUPL": os.path.join(ROOT, "B2000WCN_NOCOUPL001002"),
    "MERRA2": os.path.join(ROOT, "MERRA2_Processed"),
    "COUPLED": os.path.join(OUT_ROOT, "data"),
}

XTICKS = [0, 31, 61, 92, 123, 151, 182, 212, 243, 273, 304, 335]
XTICK_LABELS = ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep"]

mpl.rc("xtick", direction="out", labelsize=9)
mpl.rc("ytick", direction="out", labelsize=9)
mpl.rc("axes", titlesize=11, labelsize=10)
mpl.rc("font", family="sans-serif")

EPFLUX_LAT = 60.0
O3_NH_BOUNDS = (60.0, 90.0)
#MICRO_KGKG_TO_PPMV = 28.9644 / 48.0   

PLOT_CFG = {
    "U": {"cmap": "RdBu_r", "cbar_label": "U (m/s)", "title": "Climatological Zonal Wind at 60°N (1–100 hPa)", "extend": "both", "symmetric": True},
    "T": {"cmap": "Spectral_r", "cbar_label": "T (K)", "title": "Climatological Polar-Cap Temperature (60–90°N, 1–100 hPa)", "extend": "both", "symmetric": False},
    "O3": {"cmap": "viridis", "cbar_label": "O$_3$ (ppmv)", "title": "Climatological Polar-Cap Ozone (60–90°N, 1–100 hPa)", "extend": "both", "symmetric": False},
    "EPFLUX": {"cmap": "PuOr_r", "cbar_label": "Fz (native)", "title": "Climatological EP Flux Fz (1–100 hPa)", "extend": "both", "symmetric": True},
}

DISPLAY_NAMES = {"MERRA2": "MERRA2", "COUPLED": "Coupled", "NOCOUPL": "NOCOUPL"}

# ============================================================
# Helpers
# ============================================================
def robust_levels(values, symmetric=False, nlev=31):
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    if x.size == 0: return np.linspace(-1, 1, nlev)

    if symmetric:
        vmax = np.nanpercentile(np.abs(x), 98)
        if not np.isfinite(vmax) or vmax == 0: vmax = np.nanmax(np.abs(x))
        if not np.isfinite(vmax) or vmax == 0: vmax = 1.0
        return np.linspace(-vmax, vmax, nlev)

    vmin, vmax = np.nanpercentile(x, 2), np.nanpercentile(x, 98)
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax: vmin, vmax = np.nanmin(x), np.nanmax(x)
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax: vmin, vmax = -1, 1
    return np.linspace(vmin, vmax, nlev)

def find_clim_file(exp_name, var_key):
    # 🚨 修复Bug：彻底纠正了 NOCOUPL 文件前缀找不到的问题
    if exp_name == "NOCOUPL":
        exact_file = os.path.join(PATHS["NOCOUPL"], f"B2000WCN_NOCOUPL_{var_key}_climatology_1to100hPa.nc")
    elif exp_name == "COUPLED":
        exact_file = os.path.join(PATHS["COUPLED"], f"{var_key}_compare_climatology_1to100hPa.nc")
    else:
        exact_file = os.path.join(PATHS[exp_name], f"{exp_name}_{var_key}_climatology_1to100hPa.nc")
        
    if os.path.exists(exact_file):
        return exact_file
    return None

def reduce_to_old_style_profile(da, var_key):
    if "lon" in da.dims: da = da.mean(dim="lon")

    if var_key == "U" and "lat" in da.dims:
        da = da.sel(lat=60.0, method="nearest")
    elif var_key in ["T", "O3"] and "lat" in da.dims:
        lat_vals = da["lat"].values
        ascending = lat_vals[0] < lat_vals[-1]
        band = da.sel(lat=slice(60.0, 90.0)) if ascending else da.sel(lat=slice(90.0, 60.0))
        w = np.cos(np.deg2rad(band["lat"]))
        da = band.weighted(w).mean(dim="lat")
    elif var_key == "EPFLUX" and "lat" in da.dims:
        da = da.sel(lat=EPFLUX_LAT, method="nearest")

    if "dataset" in da.dims: da = da.squeeze("dataset", drop=True)
    elif "dataset" in da.coords: da = da.drop_vars("dataset")

    return da.transpose("plev", "dayofyear")

def load_one_dataset_profile(exp_name, var_key):
    path = find_clim_file(exp_name, var_key)
    if path is None:
        print(f"[Skip] File not found for {exp_name}-{var_key}")
        return None, None

    ds = xr.open_dataset(path)
    
    if f"{exp_name}_clim_jandec" in ds:
        da = ds[f"{exp_name}_clim_jandec"]
    elif "clim_jandec" in ds:
        da = ds["clim_jandec"]
    else:
        ds.close()
        return None, None

    if var_key == "O3":
        factor =  1.0
        da = da * factor

    da2 = reduce_to_old_style_profile(da, var_key).load()
    ds.close()

    idx = np.r_[np.arange(273, 365), np.arange(0, 273)]
    da2 = da2.isel(dayofyear=idx).rename({"dayofyear": "time_index"}).assign_coords(time_index=np.arange(365))
    return da2, path

# ============================================================
# Main plot function
# ============================================================
def plot_one_variable(var_key):
    base_datasets = ["MERRA2", "BWCN", "B2000WCN", "NOCOUPL"]
    da_dict = {}
    
    for dname in base_datasets:
        da, path = load_one_dataset_profile(dname, var_key)
        if da is not None:
            da_dict[dname] = da
            print(f"[Load] {dname}: {path}")

    # 🚨 科学修复：现场加权合成 COUPLED！(BWCN约24年，B2000WCN约209年)
    if "BWCN" in da_dict and "B2000WCN" in da_dict:
        w_bwcn, w_b2k = 24.0, 209.0
        da_dict["COUPLED"] = (da_dict["BWCN"] * w_bwcn + da_dict["B2000WCN"] * w_b2k) / (w_bwcn + w_b2k)
        print(f"[Compute] COUPLED generated with weighted average (BWCN*24 + B2000WCN*209)")

    plot_targets = ["MERRA2", "COUPLED", "NOCOUPL"]
    used_names = [d for d in plot_targets if d in da_dict]

    if not used_names:
        print(f"\n[Skip] 缺少足够数据来画图: {var_key}")
        return

    arr_list = [da_dict[d] for d in used_names]
    arr = xr.concat(arr_list, dim="dataset")
    arr = arr.assign_coords(dataset=("dataset", used_names))
    arr = arr.transpose("dataset", "plev", "time_index")

    plev, x = arr["plev"].values, arr["time_index"].values
    levels = robust_levels(arr.values, symmetric=PLOT_CFG[var_key]["symmetric"], nlev=31)

    fig, axes = plt.subplots(1, len(used_names), figsize=(5.0 * len(used_names), 4.8), sharex=True, sharey=True, constrained_layout=True)
    if len(used_names) == 1: axes = [axes]

    cf = None
    for i, dname in enumerate(used_names):
        ax = axes[i]
        z = arr.sel(dataset=dname).values
        if var_key == "EPFLUX": z = -z

        cf = ax.contourf(x, plev, z, levels=levels, cmap=PLOT_CFG[var_key]["cmap"], extend=PLOT_CFG[var_key]["extend"], alpha=0.9)
        cs = ax.contour(x, plev, z, levels=10, colors="k", linewidths=0.45, alpha=0.65)
        fmt_str = "%.2g" if var_key == "EPFLUX" else "%.1f"
        ax.clabel(cs, inline=True, fontsize=6, fmt=fmt_str)

        if var_key in ["U", "EPFLUX"]:
            try: ax.contour(x, plev, z, levels=[0], colors="k", linewidths=1.5)
            except: pass

        ax.set_yscale("log")
        ax.invert_yaxis()
        ax.set_ylim(100, 1)
        ax.set_yticks(plev)
        ax.yaxis.set_major_formatter(ScalarFormatter())

        ax.set_xlim(x[0], x[-1])
        ax.set_xticks(XTICKS)
        ax.set_xticklabels(XTICK_LABELS)
        ax.grid(True, which="major", linestyle="--", linewidth=0.35, alpha=0.25)
        ax.set_title(DISPLAY_NAMES.get(dname, dname))
        if i == 0: ax.set_ylabel("Pressure (hPa)")

    cbar = fig.colorbar(cf, ax=axes, pad=0.02, shrink=0.95)
    cbar.set_label(PLOT_CFG[var_key]["cbar_label"])
    fig.suptitle(PLOT_CFG[var_key]["title"], fontsize=13)

    out_png = os.path.join(PLOT_OUT, f"{var_key}_compare_climatology_1to100hPa_OctSep.png")
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"\n✅ 成功画图保存至: {out_png}")

if __name__ == "__main__":
    for var_key in ["U","T","O3","EPFLUX"]: 
        plot_one_variable(var_key)

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import glob
import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

# ============================================================
# Paths
# ============================================================
ROOT = "/mnt/soclim0/public_data/weiji"
OUT_ROOT = "/home/weiji/restart_exam/code/20260226B2000WCN_BWCN_MERRA2_COMPARE/RESULT"
PLOT_OUT = os.path.join(OUT_ROOT, "plots")
os.makedirs(PLOT_OUT, exist_ok=True)

PATHS = {
    "BWCN": os.path.join(ROOT, "BWCN"),
    "B2000WCN": os.path.join(ROOT, "B2000WCN001002"),
    "NOCOUPL": os.path.join(ROOT, "B2000WCN_NOCOUPL001002"),
    "MERRA2": os.path.join(ROOT, "MERRA2_Processed"),
    "COUPLED": os.path.join(OUT_ROOT, "data"),
}

# ============================================================
# X-axis (Oct -> Sep)
# ============================================================
XTICKS = [0, 31, 61, 92, 123, 151, 182, 212, 243, 273, 304, 335]
XTICK_LABELS = ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar",
                "Apr", "May", "Jun", "Jul", "Aug", "Sep"]

# ============================================================
# Matplotlib style
# ============================================================
mpl.rc("xtick", direction="out", labelsize=9)
mpl.rc("ytick", direction="out", labelsize=9)
mpl.rc("axes", titlesize=11, labelsize=10)
mpl.rc("font", family="sans-serif")

# ============================================================
# Plot config
# ============================================================
EPFLUX_LAT = 60.0
O3_NH_BOUNDS = (60.0, 90.0)
O3_SH_BOUNDS = (-90.0, -60.0)
O3_DEBUG = True

# ============================================================
# O3 unit conversion
# ============================================================
# 你的当前 climatology 文件大概率来自：
# raw O3 (kg/kg) * 1e6
# 所以还需要乘 (M_air / M_O3) 才是真正 ppmv
MW_AIR = 28.9644
MW_O3 = 48.0
MICRO_KGKG_TO_PPMV = MW_AIR / MW_O3   # ~= 0.603425

# 如果后面发现某个数据集其实已经是 ppmv，就把它改成 "ppmv"
O3_UNIT_MODE = {
    "MERRA2": "micro_kgkg",
    "COUPLED": "ppmv",
    "NOCOUPL": "ppmv",
}
PLOT_CFG = {
    "U": {
        "cmap": "RdBu_r",
        "cbar_label": "U (m/s)",
        "title": "Climatological Zonal Wind at 60°N (1–100 hPa)",
        "extend": "both",
        "symmetric": True,
    },
    "T": {
        "cmap": "Spectral_r",
        "cbar_label": "T (K)",
        "title": "Climatological Polar-Cap Temperature (60–90°N, 1–100 hPa)", # 👈 修改了这里
        "extend": "both",
        "symmetric": False,
    },
    "O3": {
        "cmap": "viridis",
        "cbar_label": "O$_3$ (ppmv)",
        "title": "Climatological Polar-Cap Ozone (60–90°N, 1–100 hPa)",
        "extend": "both",
        "symmetric": False,
    },
    "EPFLUX": {
        "cmap": "PuOr_r",
        "cbar_label": "Fz (native)",
        "title": "Climatological EP Flux Fz (1–100 hPa)",
        "extend": "both",
        "symmetric": True,
    },
}

DISPLAY_NAMES = {
    "MERRA2": "MERRA2",
    "COUPLED": "Coupled",
    "NOCOUPL": "NOCOUPL",
}

# ============================================================
# Helpers
# ============================================================
def robust_levels(values, symmetric=False, nlev=31):
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]

    if x.size == 0:
        return np.linspace(-1, 1, nlev)

    if symmetric:
        vmax = np.nanpercentile(np.abs(x), 98)
        if not np.isfinite(vmax) or vmax == 0:
            vmax = np.nanmax(np.abs(x))
        if not np.isfinite(vmax) or vmax == 0:
            vmax = 1.0
        return np.linspace(-vmax, vmax, nlev)

    vmin = np.nanpercentile(x, 2)
    vmax = np.nanpercentile(x, 98)
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
        vmin = np.nanmin(x)
        vmax = np.nanmax(x)
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
        vmin, vmax = -1, 1
    return np.linspace(vmin, vmax, nlev)

def find_clim_file(exp_name, var_key):
    folder = PATHS[exp_name]
    pattern = os.path.join(folder, f"{exp_name}_{var_key}_climatology_*_1to100hPa.nc")
    files = sorted(glob.glob(pattern))
    if len(files) == 0:
        return None
    if len(files) > 1:
        print(f"[Warn] multiple files found for {exp_name}-{var_key}, using: {files[0]}")
    return files[0]

def select_lat_band_robust(da, lat_min, lat_max):
    if "lat" not in da.dims:
        raise ValueError("No lat dimension found.")

    lat_vals = da["lat"].values
    ascending = lat_vals[0] < lat_vals[-1]

    if ascending:
        out = da.sel(lat=slice(lat_min, lat_max))
    else:
        out = da.sel(lat=slice(lat_max, lat_min))

    if out.sizes.get("lat", 0) == 0:
        raise ValueError(
            f"Lat-band selection failed for bounds ({lat_min}, {lat_max}); "
            f"lat order={'ascending' if ascending else 'descending'}; "
            f"lat range=({float(lat_vals.min())}, {float(lat_vals.max())})"
        )
    return out

def weighted_cap_mean(da_band):
    w = np.cos(np.deg2rad(da_band["lat"]))
    return da_band.weighted(w).mean(dim="lat")

def reorder_dayofyear_to_octsep(da_plev_day):
    if "dayofyear" not in da_plev_day.dims:
        raise ValueError("dayofyear dim not found.")

    idx = np.r_[np.arange(273, 365), np.arange(0, 273)]
    da_os = da_plev_day.isel(dayofyear=idx)
    da_os = da_os.rename({"dayofyear": "time_index"})
    da_os = da_os.assign_coords(time_index=np.arange(365))
    return da_os

def convert_o3_to_ppmv(da, exp_name):
    """
    把当前 climatology 文件中的 O3 数值统一转成 ppmv。
    假设:
      - 'micro_kgkg' : 上游已经做过 *1e6，但还没做分子量换算
      - 'kgkg'       : 上游完全没做换算
      - 'ppmv'       : 已经是 ppmv，不改
    """
    mode = O3_UNIT_MODE.get(exp_name, "ppmv")

    if mode == "ppmv":
        factor = 1.0
    elif mode == "micro_kgkg":
        factor = MICRO_KGKG_TO_PPMV
    elif mode == "kgkg":
        factor = MICRO_KGKG_TO_PPMV * 1e6
    else:
        raise ValueError(f"Unknown O3_UNIT_MODE for {exp_name}: {mode}")

    if O3_DEBUG:
        v0 = float(np.nanmean(da.values))
        v1 = float(np.nanmean((da * factor).values))
        print(
            f"[O3 unit] {exp_name}: mode={mode}, factor={factor:.6f}, "
            f"mean_before={v0:.6g}, mean_after={v1:.6g}"
        )

    return da * factor

def diagnose_o3_nh_vs_sh(da_lonmean, exp_name):
    nh_band = select_lat_band_robust(da_lonmean, *O3_NH_BOUNDS)
    sh_band = select_lat_band_robust(da_lonmean, *O3_SH_BOUNDS)

    nh_lats = nh_band["lat"].values
    sh_lats = sh_band["lat"].values

    nh_pc = weighted_cap_mean(nh_band)
    sh_pc = weighted_cap_mean(sh_band)

    nh_mean = float(nh_pc.mean().values)
    sh_mean = float(sh_pc.mean().values)

    print(
        f"[O3 check] {exp_name}: "
        f"lat order={'ascending' if nh_lats[0] < nh_lats[-1] else 'descending'} | "
        f"NH selected=({float(nh_lats.min()):.1f}, {float(nh_lats.max()):.1f}) | "
        f"SH selected=({float(sh_lats.min()):.1f}, {float(sh_lats.max()):.1f}) | "
        f"annual mean NH={nh_mean:.3f}, SH={sh_mean:.3f}"
    )

def reduce_to_old_style_profile(da, var_key, exp_name):
    if "dayofyear" not in da.dims or "plev" not in da.dims:
        raise ValueError(f"Unexpected dims for plotting: {da.dims}")

    if "lon" in da.dims:
        da = da.mean(dim="lon")

    # 仅对纬向风 U 提取 60°N 单点
    if var_key == "U":
        if "lat" in da.dims:
            da = da.sel(lat=60.0, method="nearest")

    # 对 T 和 O3 都计算 60-90°N 的面积加权极区平均
    elif var_key in ["T", "O3"]:
        if "lat" in da.dims:
            if var_key == "O3" and O3_DEBUG:
                diagnose_o3_nh_vs_sh(da, exp_name)

            nh_band = select_lat_band_robust(da, *O3_NH_BOUNDS)
            nh_lats = nh_band["lat"].values

            if np.nanmax(nh_lats) < 0 or np.nanmin(nh_lats) < 0:
                raise ValueError(
                    f"O3 NH selection failed for {exp_name}: selected lat band is not in Northern Hemisphere. "
                    f"Selected lat range=({float(np.nanmin(nh_lats))}, {float(np.nanmax(nh_lats))})"
                )

            da = weighted_cap_mean(nh_band)

    elif var_key == "EPFLUX":
        if "lat" in da.dims:
            da = da.sel(lat=EPFLUX_LAT, method="nearest")

    extra_dims = [d for d in da.dims if d not in ["dayofyear", "plev"]]
    if len(extra_dims) > 0:
        raise ValueError(f"Still has unexpected dims after reduction: {extra_dims}")

    return da.transpose("plev", "dayofyear")

def load_one_dataset_profile(exp_name, var_key):
    path = find_clim_file(exp_name, var_key)
    if path is None:
        print(f"[Skip] file not found for {exp_name}-{var_key}")
        return None, None

    ds = xr.open_dataset(path)

    if "clim_jandec" not in ds:
        print(f"[Skip] clim_jandec not found in: {path}")
        ds.close()
        return None, None

    da = ds["clim_jandec"]

    # 关键修改：O3 先把当前文件中的数值统一换成 ppmv
    if var_key == "O3":
        da = convert_o3_to_ppmv(da, exp_name)

    da2 = reduce_to_old_style_profile(da, var_key, exp_name).load()
    ds.close()

    da2 = reorder_dayofyear_to_octsep(da2)

    return da2, path

# ============================================================
# Main plot function
# ============================================================
def plot_one_variable(var_key):
    datasets = ["MERRA2", "COUPLED", "NOCOUPL"]

    arr_list = []
    used_names = []

    for dname in datasets:
        da, path = load_one_dataset_profile(dname, var_key)
        if da is None:
            continue
        arr_list.append(da)
        used_names.append(dname)
        print(f"[Load] {dname}: {path}")

    if len(arr_list) == 0:
        print(f"[Skip] no valid data found for {var_key}")
        return

    arr = xr.concat(arr_list, dim="dataset")
    arr = arr.assign_coords(dataset=("dataset", used_names))
    arr = arr.transpose("dataset", "plev", "time_index")

    plev = arr["plev"].values
    x = arr["time_index"].values

    levels = robust_levels(arr.values, symmetric=PLOT_CFG[var_key]["symmetric"], nlev=31)

    fig, axes = plt.subplots(
        1, len(used_names),
        figsize=(5.0 * len(used_names), 4.8),
        sharex=True,
        sharey=True,
        constrained_layout=True
    )

    if len(used_names) == 1:
        axes = [axes]

    cf = None
    for i, dname in enumerate(used_names):
        ax = axes[i]
        z = arr.sel(dataset=dname).values

        if var_key == "EPFLUX":
            z = -z

        cf = ax.contourf(
            x, plev, z,
            levels=levels,
            cmap=PLOT_CFG[var_key]["cmap"],
            extend=PLOT_CFG[var_key]["extend"],
            alpha=0.9
        )

        cs = ax.contour(
            x, plev, z,
            levels=10,
            colors="k",
            linewidths=0.45,
            alpha=0.65
        )
        ax.clabel(cs, inline=True, fontsize=6, fmt="%.2g")

        if var_key in ["U", "EPFLUX"]:
            try:
                ax.contour(
                    x, plev, z,
                    levels=[0],
                    colors="k",
                    linewidths=1.2
                )
            except Exception:
                pass

        ax.set_yscale("log")
        ax.invert_yaxis()
        ax.set_ylim(100, 1)

        ax.set_yticks(plev)
        ax.yaxis.set_major_formatter(ScalarFormatter())

        ax.set_xlim(x[0], x[-1])
        ax.set_xticks(XTICKS)
        ax.set_xticklabels(XTICK_LABELS)

        ax.grid(True, which="major", linestyle="--", linewidth=0.35, alpha=0.25)
        ax.set_title(DISPLAY_NAMES.get(dname, dname))

        if i == 0:
            ax.set_ylabel("Pressure (hPa)")

    cbar = fig.colorbar(cf, ax=axes, pad=0.02, shrink=0.95)
    cbar.set_label(PLOT_CFG[var_key]["cbar_label"])

    fig.suptitle(PLOT_CFG[var_key]["title"], fontsize=13)

    out_png = os.path.join(PLOT_OUT, f"{var_key}_compare_climatology_1to100hPa_OctSep.png")
    out_pdf = os.path.join(PLOT_OUT, f"{var_key}_compare_climatology_1to100hPa_OctSep.pdf")

    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.savefig(out_pdf, bbox_inches="tight")
    plt.show()

    print(f"Saved: {out_png}")
    print(f"Saved: {out_pdf}")

# ============================================================
# Run
# ============================================================
for var_key in ["U", "T", "O3", "EPFLUX"]:
    plot_one_variable(var_key)

print("\nAll plots saved to:")
print(PLOT_OUT)

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import glob
import logging
import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

# ============================================================
# 终极修复：压制字体报错 & 强制使用内置安全字体
# ============================================================
# 彻底屏蔽 font_manager 的警告轰炸
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)

plt.rcParams.update({
    "font.family": "DejaVu Sans",  # 使用 Matplotlib 100% 包含的内置字体
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.linewidth": 0.8,
    "figure.dpi": 300
})

# ============================================================
# Paths & Global Config
# ============================================================
ROOT = "/mnt/soclim0/public_data/weiji"
OUT_ROOT = "/home/weiji/restart_exam/code/20260226B2000WCN_BWCN_MERRA2_COMPARE/RESULT"
PLOT_OUT = os.path.join(OUT_ROOT, "plots_nature")
os.makedirs(PLOT_OUT, exist_ok=True)

PATHS = {"MERRA2": os.path.join(ROOT, "MERRA2_Processed"),
         "COUPLED": os.path.join(OUT_ROOT, "data")}

XTICKS = [0, 31, 61, 92, 123, 151, 182, 212, 243, 273, 304, 335]
XTICK_LABELS = ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep"]

# 变量特定配置
PLOT_CFG = {
    "U": {
        "cmap": "RdBu_r", "cmap_bias": "RdBu_r", "label": "U (m s$^{-1}$)", 
        "title": "Zonal Wind at 60°N", "sym": True, "fmt": "%.0f"
    },
    "T": {
        "cmap": "Spectral_r", "cmap_bias": "RdBu_r", "label": "T (K)", 
        "title": "Polar-Cap Temp (60–90°N)", "sym": False, "fmt": "%.0f"
    },
    "O3": {
        "cmap": "viridis", "cmap_bias": "PiYG", "label": "O$_3$ (ppmv)", 
        "title": "Polar-Cap Ozone (60–90°N)", "sym": False, "fmt": "%.1f"
    },
    "EPFLUX": {
        "cmap": "PuOr_r", "cmap_bias": "PuOr_r", "label": "F$_z$ ($10^{-3}$ native)", 
        "title": "EP Flux F$_z$", "sym": True, "scale": 1e3, "fmt": "%.1f"
    },
}

# ============================================================
# Core Functions
# ============================================================
def get_nice_levels(data, cfg, is_bias=False):
    """创建美观的离散刻度，增加对小数值(如O3 Bias)的智能自适应"""
    x = data[np.isfinite(data)]
    if is_bias or cfg.get("sym"):
        vmax = np.nanpercentile(np.abs(x), 98)
        if vmax == 0: 
            return np.linspace(-1, 1, 13)
        
        # 智能色标边界限制
        if vmax > 10:
            limit = np.ceil(vmax / 2) * 2  # 例如 13.5 -> 14
        elif vmax > 1:
            limit = np.ceil(vmax)          # 例如 3.2 -> 4.0
        elif vmax > 0.1:
            limit = np.ceil(vmax * 10) / 10.0  # 例如 0.37 -> 0.40 (解决 O3 留白问题)
        else:
            limit = np.ceil(vmax * 100) / 100.0 # 例如 0.037 -> 0.04
            
        return np.linspace(-limit, limit, 15) # 使用15层，过渡更平滑
    else:
        vmin, vmax = np.nanpercentile(x, 2), np.nanpercentile(x, 98)
        return np.linspace(vmin, vmax, 13)

def load_and_process(exp_name, var_key):
    pattern = os.path.join(PATHS[exp_name], f"{exp_name}_{var_key}_climatology_*_1to100hPa.nc")
    files = sorted(glob.glob(pattern))
    if not files: return None
    ds = xr.open_dataset(files[0])
    da = ds["clim_jandec"]
    
    if var_key == "O3" and exp_name == "MERRA2":
        da = da * (28.9644 / 48.0)
    
    if "lon" in da.dims: da = da.mean("lon")
    if "lat" in da.dims:
        if var_key in ["U", "EPFLUX"]: da = da.sel(lat=60.0, method="nearest")
        else:
            lat_m = (60, 90) if da.lat[0] < da.lat[-1] else (90, 60)
            band = da.sel(lat=slice(*lat_m))
            da = band.weighted(np.cos(np.deg2rad(band.lat))).mean("lat")
            
    idx = np.r_[np.arange(273, 365), np.arange(0, 273)]
    da = da.isel(dayofyear=idx).transpose("plev", "dayofyear")
    val = da.values
    if var_key == "EPFLUX": val = -val
    
    # 处理 EPFLUX 数量级
    if "scale" in PLOT_CFG[var_key]:
        val = val * PLOT_CFG[var_key]["scale"]
        
    return val, da.plev.values

# ============================================================
# Plotting
# ============================================================
def plot_3panel_nature(var_key):
    m2_res = load_and_process("MERRA2", var_key)
    cpl_res = load_and_process("COUPLED", var_key)
    if not m2_res or not cpl_res: return

    m2_v, plev = m2_res
    cpl_v, _ = cpl_res
    bias = cpl_v - m2_v
    days = np.arange(365)
    cfg = PLOT_CFG[var_key]

    fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.8), sharey=True, constrained_layout=True)

    levs_clim = get_nice_levels(np.concatenate([m2_v, cpl_v]), cfg)
    levs_bias = get_nice_levels(bias, cfg, is_bias=True)

    titles = ["MERRA2", "Interactive WACCM4", "Bias (Model - Obs)"]
    data_list = [m2_v, cpl_v, bias]
    cf_clim = None
    cf_bias = None

    # --- Plot Panels ---
    for i in range(3):
        ax = axes[i]
        data = data_list[i]
        curr_levs = levs_bias if i == 2 else levs_clim
        curr_cmap = cfg["cmap_bias"] if i == 2 else cfg["cmap"]
        
        # 1. 填色
        cf = ax.contourf(days, plev, data, levels=curr_levs, cmap=curr_cmap, extend="both")
        if i == 1: cf_clim = cf
        if i == 2: cf_bias = cf
        
        # 2. 等值线叠加与数值标注
        # 无论在哪张图，用来画线的底层数据统一使用 MERRA2 (i=2时) 或自身的原始场 (i<2)
        line_data = m2_v if i == 2 else data
        
        # 对气候态等值线抽样，避免线太密导致数字重叠
        contour_levels = levs_clim[::2] 
        
        contour_alpha = 0.85 if i == 2 else 0.5
        contour_lw = 0.6 if i == 2 else 0.4
        
        # 绘制黑线
        cs = ax.contour(days, plev, line_data, levels=contour_levels, colors="k", linewidths=contour_lw, alpha=contour_alpha)
        
        # 标上数值 (inline=True 表示切断线标数字)
        ax.clabel(cs, inline=True, fontsize=7, fmt=cfg["fmt"])
        
        # 3. 零线加粗实线
        if var_key == "U":
            # 如果是 Bias 图 (i==2)，叠加 MERRA2 的 0 线
            # 如果是气候态图 (i<2)，画各自的 0 线
            zero_line_data = m2_v if i == 2 else data
            ax.contour(days, plev, zero_line_data, levels=[0], colors="k", linewidths=1.5)

        # 坐标轴样式美化
        ax.set_yscale("log")
        ax.invert_yaxis()
        ax.set_ylim(100, 1)
        ax.set_yticks([100, 50, 20, 10, 5, 2, 1])
        ax.yaxis.set_major_formatter(ScalarFormatter())
        
        ax.set_xticks(XTICKS)
        ax.set_xticklabels(XTICK_LABELS, rotation=45, ha='right')
        
        ax.set_title(titles[i], pad=10, fontweight="bold")
        if i == 0: ax.set_ylabel("Pressure (hPa)")
        
    # 添加色标
    cb1 = fig.colorbar(cf_clim, ax=axes[:2], location='right', shrink=0.85, pad=0.02)
    cb1.set_label(cfg["label"])
    cb1.ax.yaxis.set_major_formatter(plt.FormatStrFormatter(cfg["fmt"]))

    cb2 = fig.colorbar(cf_bias, ax=axes[2], location='right', shrink=0.85, pad=0.02)
    cb2.set_label(f"$\Delta$ {cfg['label']}")
    cb2.ax.yaxis.set_major_formatter(plt.FormatStrFormatter(cfg["fmt"]))

    fig.suptitle(f"{cfg['title']}", fontsize=14, fontweight="bold")
    
    out_path = os.path.join(PLOT_OUT, f"{var_key}_3panel_NatureStyle.png")
    plt.savefig(out_path, bbox_inches="tight")
    plt.show() # 画完释放内存，替代 plt.show() 防止卡顿
    print(f"Saved: {out_path}")

if __name__ == "__main__":
    for vk in ["U", "T", "O3", "EPFLUX"]:
        plot_3panel_nature(vk)

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

# ============================================================
# 用户配置
# ============================================================
FILE = "/mnt/backup_ETH/Marina/MERRA2/MERRA2_3D_UVTO3_r144x96_dm.nc"
OUT_DIR = "./debug_merra2_o3"
os.makedirs(OUT_DIR, exist_ok=True)

NH_BOUNDS = (60.0, 90.0)
SH_BOUNDS = (-90.0, -60.0)

# 定义 Y 轴常用稀疏刻度 (hPa)
STRAT_TICKS = [1, 2, 5, 10, 20, 30, 50, 70, 100, 200, 500, 1000]

# 定义固定色标 (根据你提供的图片数值)
CUSTOM_LEVELS = [1.095, 1.802, 2.510, 3.217, 3.925, 4.632, 5.340, 6.047]

# ============================================================
# 数据处理工具
# ============================================================
def standardize_names(da):
    rename_map = {}
    for cand in ["plev", "lev", "level"]:
        if cand in da.coords or cand in da.dims:
            if cand != "plev": rename_map[cand] = "plev"
            break
    for cand in ["lat", "latitude"]:
        if cand in da.coords or cand in da.dims:
            if cand != "lat": rename_map[cand] = "lat"
            break
    for cand in ["lon", "longitude"]:
        if cand in da.coords or cand in da.dims:
            if cand != "lon": rename_map[cand] = "lon"
            break
    return da.rename(rename_map) if rename_map else da

def convert_o3_to_ppmv(da):
    units = str(da.attrs.get("units", "")).lower().replace(" ", "")
    if units in ["kg/kg", "kgkg-1", "kgkg^-1", "kgkg**-1"]:
        mol_mass_ratio = 28.9644 / 47.9982
        return da * (mol_mass_ratio * 1e6)
    if units in ["mol/mol", "molmol-1", "molmol^-1", "vmr", "1", ""]:
        return da * 1e6
    return da * 1e6

def normalize_plev_to_hpa(da):
    if "plev" not in da.coords: return da
    p = da["plev"].values.astype(float)
    if np.nanmax(p) > 2000:
        da = da.assign_coords(plev=("plev", p / 100.0))
    return da

def select_lat_band_robust(da, lat_min, lat_max):
    lat_vals = da["lat"].values
    ascending = lat_vals[0] < lat_vals[-1]
    out = da.sel(lat=slice(lat_min, lat_max)) if ascending else da.sel(lat=slice(lat_max, lat_min))
    return out

def weighted_cap_mean(da):
    w = np.cos(np.deg2rad(da["lat"]))
    return da.weighted(w).mean(dim="lat")

def build_o3_climatology(o3):
    o3_zm = o3.mean(dim="lon") if "lon" in o3.dims else o3
    nh_band = select_lat_band_robust(o3_zm, *NH_BOUNDS)
    sh_band = select_lat_band_robust(o3_zm, *SH_BOUNDS)
    nh_pc = weighted_cap_mean(nh_band)
    sh_pc = weighted_cap_mean(sh_band)
    nh_clim = nh_pc.groupby("time.dayofyear").mean("time")
    sh_clim = sh_pc.groupby("time.dayofyear").mean("time")
    return nh_clim, sh_clim

def reorder_dayofyear_to_octsep(da):
    doy = da["dayofyear"].values
    if len(doy) == 366:
        da = da.sel(dayofyear=doy[doy != 60])
        doy = da["dayofyear"].values
    # Oct 1st is index 273 for 365-day year
    idx = np.r_[np.arange(273, 365), np.arange(0, 273)]
    da_os = da.isel(dayofyear=idx)
    da_os = da_os.rename({"dayofyear": "time_index"}).assign_coords(time_index=np.arange(365))
    return da_os

# ============================================================
# 绘图函数
# ============================================================
def set_common_axes(ax):
    """统一设置 X 轴月份和 Y 轴 Log 格式"""
    ax.set_yscale("log")
    ax.invert_yaxis()
    ax.yaxis.set_major_formatter(ScalarFormatter())
    ax.set_xticks([0, 31, 61, 92, 123, 151, 182, 212, 243, 273, 304, 335])
    ax.set_xticklabels(["Oct", "Nov", "Dec", "Jan", "Feb", "Mar",
                        "Apr", "May", "Jun", "Jul", "Aug", "Sep"])
    ax.grid(True, linestyle="--", linewidth=0.3, alpha=0.3)

def plot_nh_sh_comparison(nh_clim, sh_clim):
    """绘制南北半球对比图 (1-1000 hPa)"""
    nh_os = reorder_dayofyear_to_octsep(nh_clim.transpose("plev", "dayofyear"))
    sh_os = reorder_dayofyear_to_octsep(sh_clim.transpose("plev", "dayofyear"))

    fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True, constrained_layout=True)
    
    for ax, z, title in zip(axes, [nh_os, sh_os], ["NH (60–90N)", "SH (60–90S)"]):
        cf = ax.contourf(z.time_index, z.plev, z.values, levels=CUSTOM_LEVELS, cmap="viridis", extend="both")
        ax.contour(z.time_index, z.plev, z.values, levels=CUSTOM_LEVELS, colors="white", linewidths=0.2, alpha=0.3)
        
        set_common_axes(ax)
        ax.set_yticks(STRAT_TICKS) # 使用稀疏刻度
        ax.set_ylim(1000, 1)
        ax.set_title(title)

    axes[0].set_ylabel("Pressure (hPa)")
    cbar = fig.colorbar(cf, ax=axes, pad=0.02, shrink=0.8)
    cbar.set_label("O$_3$ (ppmv)")
    
    out_name = os.path.join(OUT_DIR, "O3_NH_SH_Log_1-1000hPa.png")
    plt.savefig(out_name, dpi=300)
    print(f"Saved: {out_name}")

def plot_nh_stratosphere_only(nh_clim):
    """专门绘制北半球 1-100 hPa 专项图"""
    
    # --- 健壮的切片逻辑 ---
    # 获取当前 plev 的最小值和最大值
    p_min, p_max = nh_clim.plev.min().item(), nh_clim.plev.max().item()
    
    # 确定我们要截取的实际边界（确保在文件范围内）
    target_low = max(1.0, p_min)
    target_high = min(100.0, p_max)
    
    # 自动识别坐标轴方向进行切片
    if nh_clim.plev[0] < nh_clim.plev[-1]:
        # 升序 (1 -> 1000)
        nh_strat = nh_clim.sel(plev=slice(target_low, target_high))
    else:
        # 降序 (1000 -> 1)
        nh_strat = nh_clim.sel(plev=slice(target_high, target_low))
    
    # 检查是否真的切到了数据
    if nh_strat.plev.size < 2:
        print(f"WARNING: No data found in range 1-100 hPa. Actual plev range: {p_min:.2f} to {p_max:.2f}")
        return

    nh_os = reorder_dayofyear_to_octsep(nh_strat.transpose("plev", "dayofyear"))
    
    plt.figure(figsize=(10, 6))
    ax = plt.gca()
    
    # 使用 .values 确保 matplotlib 处理的是 numpy 数组
    cf = ax.contourf(nh_os.time_index.values, nh_os.plev.values, nh_os.values, 
                     levels=CUSTOM_LEVELS, cmap="viridis", extend="both")
    
    cs = ax.contour(nh_os.time_index.values, nh_os.plev.values, nh_os.values, 
                    levels=CUSTOM_LEVELS, colors="white", linewidths=0.4, alpha=0.4)
    
    set_common_axes(ax)
    
    # 针对平流层的刻度
    strat_only_ticks = [1, 2, 5, 10, 20, 30, 50, 70, 100]
    # 过滤掉不在当前数据范围内的刻度，防止报错
    valid_ticks = [t for t in strat_only_ticks if target_low <= t <= target_high]
    
    ax.set_yticks(valid_ticks)
    ax.set_ylim(target_high, target_low) # 限制范围
    
    ax.set_title("NH Polar Cap O$_3$ (60–90N, 1–100 hPa)", fontsize=13, pad=15)
    ax.set_ylabel("Pressure (hPa)")
    
    cbar = plt.colorbar(cf, pad=0.03)
    cbar.set_label("O$_3$ Mixing Ratio (ppmv)")
    
    out_name = os.path.join(OUT_DIR, "NH_O3_Stratosphere_1-100hPa.png")
    plt.savefig(out_name, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_name}")
# ============================================================
# 主程序
# ============================================================
def main():
    if not os.path.exists(FILE):
        raise FileNotFoundError(f"File not found: {FILE}")

    # 使用 dask 加载
    ds = xr.open_dataset(FILE, chunks={"time": 24})
    
    # 自动探测 O3 变量
    o3_var = next((v for v in ds.data_vars if v.lower() in ["o3", "ozone"]), None)
    if not o3_var: raise ValueError("Could not find O3 variable.")
    
    o3 = ds[o3_var]
    o3 = standardize_names(o3)
    o3 = normalize_plev_to_hpa(o3)
    o3 = convert_o3_to_ppmv(o3)
    
    print(f"Processing {o3_var}...")
    
    # 计算气候态
    nh_clim, sh_clim = build_o3_climatology(o3)
    
    # 执行绘图
    plot_nh_sh_comparison(nh_clim, sh_clim)
    plot_nh_stratosphere_only(nh_clim)
    
    ds.close()
    print("\nAll tasks completed.")

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

# ============================================================
# Paths
# ============================================================
OUT_ROOT = "/home/weiji/restart_exam/code/20260226B2000WCN_BWCN_MERRA2_COMPARE/RESULT"
COMPARE_OUT = os.path.join(OUT_ROOT, "data")
PLOT_OUT = os.path.join(OUT_ROOT, "plots")
os.makedirs(PLOT_OUT, exist_ok=True)

# ============================================================
# X-axis (Oct -> Sep)
# ============================================================
XTICKS = [0, 31, 61, 92, 123, 151, 182, 212, 243, 273, 304, 335]
XTICK_LABELS = ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar",
                "Apr", "May", "Jun", "Jul", "Aug", "Sep"]

# ============================================================
# Matplotlib style
# ============================================================
mpl.rc("xtick", direction="out", labelsize=9)
mpl.rc("ytick", direction="out", labelsize=9)
mpl.rc("axes", titlesize=11, labelsize=10)
mpl.rc("font", family="sans-serif")

# ============================================================
# Plot config for BIAS
# ============================================================
PLOT_CFG_BIAS = {
    "U": {
        "cmap": "RdBu_r",
        "cbar_label": "Bias: U (m/s)",
        "title": "Zonal Wind Bias at 60°N (Model - MERRA2)",
    },
    "T": {
        "cmap": "RdBu_r",  
        "cbar_label": "Bias: T (K)",
        "title": "Polar-Cap Temperature Bias (60–90°N, Model - MERRA2)",
    },
    "O3": {
        "cmap": "PiYG",    
        "cbar_label": "Bias: O$_3$ (ppmv)",
        "title": "Polar-Cap Ozone Bias (60–90°N, Model - MERRA2)",
    },
    "EPFLUX": {
        "cmap": "PuOr_r",
        "cbar_label": "Bias: Fz (native)",
        "title": "EP Flux Fz Bias (Model - MERRA2)",
    },
}

# ============================================================
# Helpers
# ============================================================
def robust_symmetric_levels(values, nlev=31):
    """为 Difference 图表生成以 0 为中心的对称 levels"""
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.linspace(-1, 1, nlev)
    vmax = np.nanpercentile(np.abs(x), 98)
    if not np.isfinite(vmax) or vmax == 0:
        vmax = np.nanmax(np.abs(x))
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0
    return np.linspace(-vmax, vmax, nlev)

# ============================================================
# Main Plot Function
# ============================================================
def plot_bias_with_merra_contours(var_key):
    # 读取第一步脚本生成的比较缓存文件
    cache_file = os.path.join(COMPARE_OUT, f"{var_key}_compare_climatology_1to100hPa.nc")
    if not os.path.exists(cache_file):
        print(f"[Skip] Cache file not found: {cache_file}")
        return

    print(f"[Load] Reading cache for {var_key}...")
    ds = xr.open_dataset(cache_file)
    
    # 获取 Oct-Sep 排列的数据
    da_os = ds["clim_octsep"]

    # 提取三大数据集的值
    # ⚠️ 此时所有数据集（包括 MERRA2 O3）都已经是 ppmv 且空间处理完毕
    merra = da_os.sel(dataset="MERRA2").values
    cpl = da_os.sel(dataset="COUPLED").values
    noc = da_os.sel(dataset="NOCOUPL").values

    # 如果是 EPFLUX，按照原有逻辑进行反转
    if var_key == "EPFLUX":
        merra = -merra
        cpl = -cpl
        noc = -noc

    # 计算偏差 Bias = Model - Obs
    diff_cpl = cpl - merra
    diff_noc = noc - merra

    plev = da_os["plev"].values
    x = da_os["time_index"].values

    # 统一 levels，中心点为 0
    all_diffs = np.concatenate([diff_cpl.ravel(), diff_noc.ravel()])
    levels_diff = robust_symmetric_levels(all_diffs, nlev=31)

    fig, axes = plt.subplots(
        1, 2, 
        figsize=(10.5, 4.8), 
        sharex=True, sharey=True, 
        constrained_layout=True
    )

    plot_data = [
        {"ax": axes[0], "diff": diff_cpl, "title": "COUPLED Bias"},
        {"ax": axes[1], "diff": diff_noc, "title": "NOCOUPL Bias"}
    ]

    cf = None
    for p in plot_data:
        ax = p["ax"]
        z_diff = p["diff"]

        # 1. 填色图：Bias (填色层级以 0 为中心对称)
        cf = ax.contourf(
            x, plev, z_diff,
            levels=levels_diff,
            cmap=PLOT_CFG_BIAS[var_key]["cmap"],
            extend="both",
            alpha=0.95
        )

        # 2. 等值线图：叠加 MERRA2 原始气候态作为背景参考
        cs = ax.contour(
            x, plev, merra,
            levels=10, 
            colors="k", 
            linewidths=0.6, 
            alpha=0.8
        )
        ax.clabel(cs, inline=True, fontsize=7, fmt="%.2g")

        # U 和 EPFLUX 加粗 0 线
        if var_key in ["U", "EPFLUX"]:
            try:
                ax.contour(
                    x, plev, merra,
                    levels=[0],
                    colors="k",
                    linewidths=1.5
                )
            except Exception:
                pass

        # 坐标轴与网格设置
        ax.set_yscale("log")
        ax.invert_yaxis()
        ax.set_ylim(100, 1)
        ax.set_yticks(plev)
        ax.yaxis.set_major_formatter(ScalarFormatter())
        ax.set_xlim(x[0], x[-1])
        ax.set_xticks(XTICKS)
        ax.set_xticklabels(XTICK_LABELS)
        ax.grid(True, which="major", linestyle="--", linewidth=0.35, alpha=0.3)
        ax.set_title(p["title"])

    axes[0].set_ylabel("Pressure (hPa)")
    cbar = fig.colorbar(cf, ax=axes, pad=0.02, shrink=0.95)
    cbar.set_label(PLOT_CFG_BIAS[var_key]["cbar_label"])

    fig.suptitle(f"{PLOT_CFG_BIAS[var_key]['title']}\n(Contours: MERRA2 Climatology)", fontsize=11, y=1.05)

    out_png = os.path.join(PLOT_OUT, f"{var_key}_Bias_with_MERRA2_contours_OctSep.png")
    out_pdf = os.path.join(PLOT_OUT, f"{var_key}_Bias_with_MERRA2_contours_OctSep.pdf")
    
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.savefig(out_pdf, bbox_inches="tight")
    plt.show()
    
    print(f"Saved: {out_png}")
    ds.close()

if __name__ == "__main__":
    print("=" * 60)
    print("Generating Bias Plots: T/O3=60-90N, MERRA2 already in ppmv")
    print("=" * 60)
    
    for var_key in ["U", "T", "O3", "EPFLUX"]:
        plot_bias_with_merra_contours(var_key)

    print("\nAll bias plots saved to:")
    print(PLOT_OUT)

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
import gc
import glob
import warnings
import numpy as np
import xarray as xr
import numba as nb

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# ============================================================
# Paths & Config
# ============================================================
ROOT = "/mnt/soclim0/public_data/weiji"
TXT_ROOT = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"

PATHS = {
    "BWCN": os.path.join(ROOT, "BWCN"),
    "B2000WCN": os.path.join(ROOT, "B2000WCN001002"),
    "NOCOUPL": os.path.join(ROOT, "B2000WCN_NOCOUPL001002"),
    "MERRA2": os.path.join(ROOT, "MERRA2_Processed"),
}

# 动态生成输出路径的辅助函数，匹配你要求的各个子目录和命名规则
def get_out_path(exp, var_key):
    prefix_map = {
        "BWCN": "BWCN",
        "B2000WCN": "B2000WCN",
        "NOCOUPL": "B2000WCN_NOCOUPL",
        "MERRA2": "MERRA2"
    }
    prefix = prefix_map[exp]
    return os.path.join(PATHS[exp], f"{prefix}_{var_key}_LOW25_climatology_1to100hPa.nc")

TARGET_PLEV_HPA = np.array(
    [100.0, 70.0, 50.0, 40.0, 30.0, 20.0, 10.0, 7.0, 5.0, 4.0, 3.0, 2.0, 1.0],
    dtype=np.float64
)
TARGET_PLEV_PA = TARGET_PLEV_HPA * 100.0

MW_AIR = 28.9644
MW_O3 = 47.9982
MOL_MASS_RATIO = MW_AIR / MW_O3

VAR_SPECS = {
    "U": {"var_name": "U", "lat_mode": "global", "scale_factor_model": 1.0, "scale_factor_merra": 1.0, "plot_unit": "m/s", "long_name": "Zonal wind"},
    "T": {"var_name": "T", "lat_mode": "global", "scale_factor_model": 1.0, "scale_factor_merra": 1.0, "plot_unit": "K", "long_name": "Temperature"},
    "O3": {"var_name": "O3", "lat_mode": "global", "scale_factor_model": 1e6, "scale_factor_merra": MOL_MASS_RATIO * 1e6, "plot_unit": "ppmv", "long_name": "Ozone mixing ratio"},
    "EPFLUX": {"var_name": "Fz", "lat_mode": None, "scale_factor_model": 1.0, "scale_factor_merra": 1.0, "plot_unit": "native", "long_name": "Vertical EP flux (Fz)"},
}

# ============================================================
# Numba Fast Interp
# ============================================================
@nb.njit(parallel=True)
def fast_logp_interp_flat(p_flat, v_flat, target_p):
    n_profs = p_flat.shape[0]
    nlev = p_flat.shape[1]
    ntarg = len(target_p)
    out = np.empty((n_profs, ntarg), dtype=np.float32)
    ltp = np.log(target_p)
    
    for i in nb.prange(n_profs):
        p, v = p_flat[i], v_flat[i]
        valid = 0
        for k in range(nlev):
            if p[k] > 0 and not np.isnan(p[k]) and not np.isnan(v[k]): valid += 1
                
        if valid < 2:
            for t_idx in range(ntarg): out[i, t_idx] = np.nan
            continue
            
        p_valid = np.empty(valid, dtype=np.float64)
        v_valid = np.empty(valid, dtype=np.float64)
        idx = 0
        for k in range(nlev):
            if p[k] > 0 and not np.isnan(p[k]) and not np.isnan(v[k]):
                p_valid[idx], v_valid[idx] = p[k], v[k]
                idx += 1
                
        order = np.argsort(p_valid)
        p_sorted, v_sorted = p_valid[order], v_valid[order]
        lp = np.log(p_sorted)
        out_i = np.interp(ltp, lp, v_sorted)
        
        slope_top = (v_sorted[1] - v_sorted[0]) / (lp[1] - lp[0])
        slope_bottom = (v_sorted[-1] - v_sorted[-2]) / (lp[-1] - lp[-2])
        
        for t_idx in range(ntarg):
            if ltp[t_idx] < lp[0]: out[i, t_idx] = v_sorted[0] + slope_top * (ltp[t_idx] - lp[0])
            elif ltp[t_idx] > lp[-1]: out[i, t_idx] = v_sorted[-1] + slope_bottom * (ltp[t_idx] - lp[-1])
            else: out[i, t_idx] = out_i[t_idx]
    return out

# ============================================================
# Core Helpers
# ============================================================
def get_ymd_from_time_value(tt):
    if hasattr(tt, "year") and hasattr(tt, "month"): return int(tt.year), int(tt.month), int(tt.day)
    y, m, d = np.datetime_as_string(tt, unit="D").split("-")
    return int(y), int(m), int(d)

def parse_file_year(path):
    m_all = re.findall(r'(?<!\d)(\d{4})(?!\d)', os.path.basename(path))
    return int(m_all[-1]) if m_all else None

def detect_coord_name(ds, candidates):
    for c in candidates:
        if c in ds.coords or c in ds.dims: return c
    return None

def detect_main_var(ds, preferred=None):
    if preferred is not None and preferred in ds.data_vars: return preferred
    for v in ds.data_vars:
        if "time" in ds[v].dims: return v
    raise ValueError("Cannot detect main variable.")

def drop_feb29_and_get_time_parts(time_vals):
    keep_idx, years, months, days = [], [], [], []
    for i, tt in enumerate(time_vals):
        y, m, d = get_ymd_from_time_value(tt)
        if not (m == 2 and d == 29):
            keep_idx.append(i); years.append(y); months.append(m); days.append(d)
    return np.array(keep_idx), np.array(years), np.array(months), np.array(days)

def logp_interp_extrap_1d(p_prof, v_prof, target_p):
    p, v, tp = np.asarray(p_prof, dtype=np.float64), np.asarray(v_prof, dtype=np.float64), np.asarray(target_p, dtype=np.float64)
    mask = np.isfinite(p) & np.isfinite(v) & (p > 0)
    p, v = p[mask], v[mask]
    if p.size < 2: return np.full(tp.shape, np.nan, dtype=np.float64)
    order = np.argsort(p)
    p, v = p[order], v[order]
    lp, ltp = np.log(p), np.log(tp)
    out = np.interp(ltp, lp, v)
    mask_top = ltp < lp[0]
    if np.any(mask_top): out[mask_top] = v[0] + ((v[1] - v[0]) / (lp[1] - lp[0])) * (ltp[mask_top] - lp[0])
    mask_bottom = ltp > lp[-1]
    if np.any(mask_bottom): out[mask_bottom] = v[-1] + ((v[-1] - v[-2]) / (lp[-1] - lp[-2])) * (ltp[mask_bottom] - lp[-1])
    return out

def interp_fixed_pressure_clim_to_target(clim_var_native, native_plev_hpa, target_plev_hpa):
    native_pa, target_pa = np.asarray(native_plev_hpa, dtype=np.float64) * 100.0, np.asarray(target_plev_hpa, dtype=np.float64) * 100.0
    out = np.full((365, len(target_plev_hpa)), np.nan, dtype=np.float32)
    for d in range(365): out[d, :] = logp_interp_extrap_1d(native_pa, clim_var_native[d, :], target_pa).astype(np.float32)
    return out

def build_clim_dataarrays_from_octsep(clim_os_target, plev_hpa, lat_vals=None, lon_vals=None):
    if lat_vals is not None and lon_vals is not None:
        arr_os = np.transpose(clim_os_target, (1, 2, 3, 0)).astype(np.float32)
        jd_data = np.concatenate([clim_os_target[92:, ...], clim_os_target[:92, ...]], axis=0)
        arr_jd = np.transpose(jd_data, (1, 2, 3, 0)).astype(np.float32)
        coords_jd = {"plev": plev_hpa.astype(np.float32), "lat": lat_vals.astype(np.float32), "lon": lon_vals.astype(np.float32), "dayofyear": np.arange(1, 366)}
        coords_os = {"plev": plev_hpa.astype(np.float32), "lat": lat_vals.astype(np.float32), "lon": lon_vals.astype(np.float32), "time_index": np.arange(365)}
        dims, dims_os = ("plev", "lat", "lon", "dayofyear"), ("plev", "lat", "lon", "time_index")
    else:
        arr_os = np.transpose(clim_os_target, (1, 0)).astype(np.float32)
        jd_data = np.concatenate([clim_os_target[92:, :], clim_os_target[:92, :]], axis=0)
        arr_jd = np.transpose(jd_data, (1, 0)).astype(np.float32)
        coords_jd = {"plev": plev_hpa.astype(np.float32), "dayofyear": np.arange(1, 366)}
        coords_os = {"plev": plev_hpa.astype(np.float32), "time_index": np.arange(365)}
        dims, dims_os = ("plev", "dayofyear"), ("plev", "time_index")

    clim_jd = xr.DataArray(arr_jd, coords=coords_jd, dims=dims, name="clim_jandec")
    clim_os = xr.DataArray(arr_os, coords=coords_os, dims=dims_os, name="clim_octsep")
    return clim_jd, clim_os

def save_targeted_climatology(exp_name, var_key, clim_jd, clim_os, long_name, plot_unit):
    ds_out = xr.Dataset(data_vars={"clim_jandec": clim_jd, "clim_octsep": clim_os})
    ds_out.attrs.update({
        "experiment": exp_name, 
        "description": "Composite of lowest 25% extreme years (Physically accurate Oct[n-1] to Sep[n])", 
        "variable": var_key, 
        "long_name": long_name, 
        "plot_unit": plot_unit, 
        "target_levels_hPa": ",".join([str(x) for x in TARGET_PLEV_HPA.tolist()])
    })
    out_nc = get_out_path(exp_name, var_key)
    ds_out.to_netcdf(out_nc)
    print(f"  [SAVED] {out_nc}")

def scan_merra_files(var_key):
    subdir_map = {"U": "U", "T": "T", "O3": "O3", "EPFLUX": "EPflux_daily"}
    files = sorted(glob.glob(os.path.join(PATHS["MERRA2"], subdir_map[var_key], "*.nc")))
    if not files: raise FileNotFoundError(f"No MERRA2 files for {var_key}")
    return files

# ============================================================
# STEP 1: Direct Load from TXT
# ============================================================
def load_low25_years_from_txt():
    exp_folder_map = {
        "MERRA2": "MERRA2",
        "BWCN": "BWCN",
        "B2000WCN": "B2000WCN",
        "NOCOUPL": "B2000WCN_NOCOUPL"
    }
    low_years_dict = {}
    for exp, folder in exp_folder_map.items():
        txt_path = os.path.join(TXT_ROOT, folder, "low25pct_years_30_70hPa.txt")
        if os.path.exists(txt_path):
            years = np.loadtxt(txt_path, dtype=int)
            years_list = [years] if years.ndim == 0 else years.tolist()
            low_years_dict[exp] = years_list
            print(f"[✔] 成功读取 {exp}: 包含 {len(years_list)} 年 -> {txt_path}")
            # 👇 新增这一行，把具体年份打印出来
            print(f"    -> 具体年份: {years_list}")
        else:
            print(f"[✘] 找不到文件 {exp}: {txt_path}")
            low_years_dict[exp] = []
    return low_years_dict

# ============================================================
# STEP 2: Targeted Climatology Builders
# ============================================================
def get_model_hybrid_part(exp_name, var_key, spec, y, part):
    files = glob.glob(os.path.join(PATHS[exp_name], var_key, f"*{y:04d}*.nc"))
    if not files: return None, None, None
    
    ds = xr.open_dataset(files[0], decode_times=True)
    phys_year = parse_file_year(files[0])
    
    if exp_name in ["B2000WCN", "NOCOUPL"] and phys_year >= 105:
        old_times = ds.time.values
        new_times = [t.replace(year=t.year + 103) for t in old_times]
        ds = ds.assign_coords(time=new_times)

    da, ps = ds[spec["var_name"]], ds["PS"]
    lat_name, lon_name = detect_coord_name(ds, ["lat", "latitude"]), detect_coord_name(ds, ["lon", "longitude"])
    lat_vals, lon_vals = ds[lat_name].values, ds[lon_name].values

    da, ps = da.transpose("time", "lev", lat_name, lon_name), ps.transpose("time", lat_name, lon_name)
    hyam, hybm, p0 = ds["hyam"].values.astype(np.float32), ds["hybm"].values.astype(np.float32), np.float32(ds["P0"].values)

    keep_idx, _, _, _ = drop_feb29_and_get_time_parts(ds["time"].values)
    if len(keep_idx) != 365:
        ds.close()
        return None, None, None
        
    keep_idx = keep_idx[273:] if part == 'prev' else keep_idx[:273]

    da_kept = da.isel(time=keep_idx).values.astype(np.float32) * np.float32(spec["scale_factor_model"])
    ps_kept = ps.isel(time=keep_idx).values.astype(np.float32)
    
    nt, nlev, nlat, nlon = da_kept.shape
    p_native = hyam[None, :, None, None] * p0 + hybm[None, :, None, None] * ps_kept[:, None, :, :]
    
    var_flat = da_kept.transpose(0, 2, 3, 1).reshape(-1, nlev).astype(np.float64)
    p_flat = p_native.transpose(0, 2, 3, 1).reshape(-1, nlev).astype(np.float64)
    out_flat = fast_logp_interp_flat(p_flat, var_flat, TARGET_PLEV_PA)
    var_interp_4d = out_flat.reshape(nt, nlat, nlon, len(TARGET_PLEV_PA)).transpose(0, 3, 1, 2)
    
    ds.close()
    return var_interp_4d, lat_vals, lon_vals

def accumulate_model_hybrid_targeted_safe(exp_name, var_key, spec, target_years):
    if not target_years: return None, None, None
    lat_vals, lon_vals, sum_var, count = None, None, None, 0
    
    for y in tqdm(target_years, desc=f"{exp_name}-{var_key} (Serial Safe)", ncols=100):
        v_prev, lat, lon = get_model_hybrid_part(exp_name, var_key, spec, y - 1, 'prev')
        v_curr, _, _ = get_model_hybrid_part(exp_name, var_key, spec, y, 'curr')
        
        if v_prev is not None and v_curr is not None:
            v_stitched = np.concatenate([v_prev, v_curr], axis=0)
            if sum_var is None:
                lat_vals, lon_vals = lat, lon
                sum_var = np.zeros_like(v_stitched, dtype=np.float64)
            sum_var += v_stitched
            count += 1
            
        del v_prev, v_curr
        gc.collect()

    if count == 0: return None, None, None
    return (sum_var / count).astype(np.float32), lat_vals, lon_vals

def get_merra_clim_part(var_key, spec, y, part):
    subdir_map = {"U": "U", "T": "T", "O3": "O3", "EPFLUX": "EPflux_daily"}
    files = glob.glob(os.path.join(PATHS["MERRA2"], subdir_map[var_key], f"*{y}*.nc"))
    if not files: return None, None, None
    
    ds = xr.open_dataset(files[0], decode_times=True)
    da = ds[detect_main_var(ds, preferred=spec["var_name"])]
    lat_name, lon_name = detect_coord_name(ds, ["lat", "latitude"]), detect_coord_name(ds, ["lon", "longitude"])
    da = da.rename({detect_coord_name(ds, ["plev", "lev", "level"]): "plev"})
    
    src = da["plev"].values.astype(np.float64)
    sel_idx = [int(np.argmin(np.abs(src - t))) for t in TARGET_PLEV_HPA]
    da = da.isel(plev=sel_idx).assign_coords(plev=("plev", TARGET_PLEV_HPA))

    lat_vals = da[lat_name].values
    lon_vals = da[lon_name].values if lon_name else None
    da = da.transpose("time", "plev", lat_name, lon_name) if lon_name else da.transpose("time", "plev", lat_name)

    keep_idx, _, _, _ = drop_feb29_and_get_time_parts(da["time"].values)
    keep_idx = keep_idx[273:] if part == 'prev' else keep_idx[:273]
    
    da = da.isel(time=keep_idx)
    arr = da.values.astype(np.float32) * np.float32(spec["scale_factor_merra"])
    ds.close()
    return arr, lat_vals, lon_vals

def accumulate_merra_targeted_safe(var_key, spec, target_years):
    lat_vals, lon_vals, sum_doy, count = None, None, None, 0
    for y in tqdm(target_years, desc=f"MERRA2-{var_key} (Serial Safe)", ncols=100):
        v_prev, lat, lon = get_merra_clim_part(var_key, spec, y - 1, 'prev')
        v_curr, _, _ = get_merra_clim_part(var_key, spec, y, 'curr')
        
        if v_prev is not None and v_curr is not None:
            v_stitched = np.concatenate([v_prev, v_curr], axis=0)
            if sum_doy is None:
                lat_vals, lon_vals = lat, lon
                sum_doy = np.zeros_like(v_stitched, dtype=np.float64)
            sum_doy += v_stitched
            count += 1
            
        del v_prev, v_curr
        gc.collect()

    if count == 0: return None, None, None
    return (sum_doy / count).astype(np.float32), lat_vals, lon_vals

# ---------------- EPFLUX 专用 2D 通道 ----------------
# ---------------- EPFLUX 专用 2D 通道 ----------------
def process_epflux_targeted_safe(exp_name, spec, target_years):
    if not target_years: return None
    
    if exp_name == "MERRA2":
        # ========================================================
        # MERRA2 路线：多文件处理
        # ========================================================
        files = scan_merra_files("EPFLUX")
        
        # 先读取第一个文件获取 native plev 的结构
        ds0 = xr.open_dataset(files[0])
        p_name = detect_coord_name(ds0, ["plev", "lev", "level"])
        src0 = ds0[p_name].values.astype(np.float64)
        if np.nanmax(src0) > 2000: src0 /= 100.0
        native_mask = (src0 >= 1.0) & (src0 <= 100.0)
        native_plev_hpa = src0[native_mask]
        ds0.close()
        
        # 为了高效，只读取 target_years 及其前一年 (y-1) 的文件
        needed_years = set(target_years).union(set([y - 1 for y in target_years]))
        year_data = {}
        
        for f in tqdm(files, desc="MERRA2-EPFLUX (Loading)", ncols=100):
            y = parse_file_year(f)
            if y in needed_years:
                ds = xr.open_dataset(f, decode_times=True)
                vname = detect_main_var(ds, preferred=spec["var_name"])
                da = ds[vname].rename({detect_coord_name(ds, ["plev", "lev", "level"]): "plev"}).transpose("time", "plev")
                
                src = da["plev"].values.astype(np.float64)
                if np.nanmax(src) > 2000:
                    src /= 100.0
                    da = da.assign_coords(plev=("plev", src))
                da = da.isel(plev=((src >= 1.0) & (src <= 100.0)))
                
                keep_idx, _, _, _ = drop_feb29_and_get_time_parts(da["time"].values)
                da = da.isel(time=keep_idx)
                
                if len(da) == 365:
                    year_data[y] = da.values.astype(np.float32)
                ds.close()
                
        sum_doy = np.zeros((365, len(native_plev_hpa)), dtype=np.float64)
        count = 0
        for y in target_years:
            if (y - 1) in year_data and y in year_data:
                v_prev = year_data[y - 1]
                v_curr = year_data[y]
                sum_doy += np.concatenate([v_prev[273:], v_curr[:273]], axis=0)
                count += 1
                
        if count == 0: return None
        clim_native = sum_doy / count
        return interp_fixed_pressure_clim_to_target(clim_native, native_plev_hpa, TARGET_PLEV_HPA)

    else:
        # ========================================================
        # 模式路线：单文件处理 (自动修补 CFTime 断层)
        # ========================================================
        epflux_files = sorted(glob.glob(os.path.join(PATHS[exp_name], "EPflux_daily", "*.nc")))
        if not epflux_files: return None
        path = epflux_files[0] # 唯一文件
        
        ds = xr.open_dataset(path, decode_times=True)
        
        # 智能时间轴修补：针对 B2000WCN 和 NOCOUPL 从 103 年回滚到 1 年的情况
        if exp_name in ["B2000WCN", "NOCOUPL"]:
            old_times = ds.time.values
            new_times = []
            offset = 0
            prev_y = old_times[0].year
            for t in old_times:
                # 如果年份突然掉落超过 50 年，说明进入了 run002，追加 103 年偏移
                if t.year < prev_y - 50: 
                    offset += 103
                new_times.append(t.replace(year=t.year + offset))
                prev_y = t.year
            ds = ds.assign_coords(time=new_times)
            
        vname = detect_main_var(ds, preferred=spec["var_name"])
        da = ds[vname].rename({detect_coord_name(ds, ["plev", "lev", "level"]): "plev"}).transpose("time", "plev")
        
        src = da["plev"].values.astype(np.float64)
        if np.nanmax(src) > 2000:
            src /= 100.0
            da = da.assign_coords(plev=("plev", src))
        da = da.isel(plev=((src >= 1.0) & (src <= 100.0)))
        native_plev_hpa = da["plev"].values.astype(np.float64)
        
        keep_idx, years, _, _ = drop_feb29_and_get_time_parts(da["time"].values)
        da = da.isel(time=keep_idx)
        years = years
        
        sum_doy = np.zeros((365, len(native_plev_hpa)), dtype=np.float64)
        count = 0
        
        for y in tqdm(target_years, desc=f"{exp_name}-EPFLUX (Stitching)", ncols=100):
            mask_prev = (years == y - 1)
            mask_curr = (years == y)
            if mask_prev.sum() == 365 and mask_curr.sum() == 365:
                v_prev = da.isel(time=mask_prev).values.astype(np.float32)
                v_curr = da.isel(time=mask_curr).values.astype(np.float32)
                # 拼接：前一年 10-12 月 (273:) + 当年 1-9 月 (:273)
                sum_doy += np.concatenate([v_prev[273:], v_curr[:273]], axis=0)
                count += 1
                
        ds.close()
        if count == 0: return None
        clim_native = sum_doy / count
        return interp_fixed_pressure_clim_to_target(clim_native, native_plev_hpa, TARGET_PLEV_HPA)

# ============================================================
# Main Execution
# ============================================================
if __name__ == "__main__":
    print("=" * 80)
    print(" STEP 1: Fast Loading Lowest 25% Years from existing TXT")
    print("=" * 80)
    
    low_years_dict = load_low25_years_from_txt()
    
    print("\n" + "=" * 80)
    print(" STEP 2: Building and Saving Physically-Accurate Low 25% Climatologies")
    print("=" * 80)

    for var_key in ["U", "T", "O3", "EPFLUX"]:
        spec = VAR_SPECS[var_key]
        print(f"\n---> Processing {var_key}")
        
        for exp in ["MERRA2", "BWCN", "B2000WCN", "NOCOUPL"]:
            low_years = low_years_dict.get(exp, [])
            
            if not low_years: 
                continue

            if var_key == "EPFLUX":
                clim = process_epflux_targeted_safe(exp, spec, low_years)
                if clim is not None:
                    jd, os_ds = build_clim_dataarrays_from_octsep(clim, TARGET_PLEV_HPA)
                    save_targeted_climatology(exp, var_key, jd, os_ds, spec["long_name"], spec["plot_unit"])
            else:
                if exp == "MERRA2":
                    clim, lat, lon = accumulate_merra_targeted_safe(var_key, spec, low_years)
                else:
                    clim, lat, lon = accumulate_model_hybrid_targeted_safe(exp, var_key, spec, low_years)
                    
                if clim is not None:
                    jd, os_ds = build_clim_dataarrays_from_octsep(clim, TARGET_PLEV_HPA, lat, lon)
                    save_targeted_climatology(exp, var_key, jd, os_ds, spec["long_name"], spec["plot_unit"])

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import numpy as np
import xarray as xr

ROOT = "/mnt/soclim0/public_data/weiji"
TXT_ROOT = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"

FILES = {
    "B2000WCN": os.path.join(ROOT, "B2000WCN001002", "EPflux_daily", "EPFLUX_206yr_Daily_Series_Full.nc"),
    "NOCOUPL": os.path.join(ROOT, "B2000WCN_NOCOUPL001002", "EPflux_daily", "EPFLUX_205yr_Daily_Series_Full.nc"),
}

TXT_FILES = {
    "B2000WCN": os.path.join(TXT_ROOT, "B2000WCN", "low25pct_years_30_70hPa.txt"),
    "NOCOUPL": os.path.join(TXT_ROOT, "B2000WCN_NOCOUPL", "low25pct_years_30_70hPa.txt"),
}


def get_ymd(tt):
    return int(tt.year), int(tt.month), int(tt.day)


def load_low25_years(txt_path):
    if not os.path.exists(txt_path):
        return []
    arr = np.loadtxt(txt_path, dtype=int)
    if np.ndim(arr) == 0:
        return [int(arr)]
    return [int(x) for x in arr.tolist()]


def inspect_epflux_file(exp_name, nc_path, txt_path):
    print("=" * 120)
    print(f"[INSPECT] {exp_name}")
    print("=" * 120)
    print(f"FILE: {nc_path}")

    if not os.path.exists(nc_path):
        print("❌ File not found.")
        return

    low25_years = load_low25_years(txt_path)
    print(f"LOW25 TXT: {txt_path}")
    print(f"LOW25 years ({len(low25_years)}): {low25_years}")

    with xr.open_dataset(nc_path, decode_times=True) as ds:
        print("\n[Dataset summary]")
        print(ds)

        if "Fz" not in ds:
            print("❌ Variable 'Fz' not found.")
            return

        time_vals = ds["time"].values
        plev_vals = ds["plev"].values if "plev" in ds.coords else None

        print("\n[Basic info]")
        print(f"time length = {len(time_vals)}")
        if plev_vals is not None:
            print(f"plev length = {len(plev_vals)}")
            print(f"plev sample = {plev_vals[:10]}")

        # time object type
        t0 = time_vals[0]
        print(f"time[0] type = {type(t0)}")
        if hasattr(ds["time"], "encoding"):
            cal = ds["time"].encoding.get("calendar", None)
            units = ds["time"].encoding.get("units", None)
            print(f"time encoding calendar = {cal}")
            print(f"time encoding units    = {units}")

        # sort/duplicate checks
        years = np.array([get_ymd(t)[0] for t in time_vals], dtype=int)
        months = np.array([get_ymd(t)[1] for t in time_vals], dtype=int)
        days = np.array([get_ymd(t)[2] for t in time_vals], dtype=int)

        unique_years = np.unique(years)

        print("\n[Year coverage]")
        print(f"min year = {unique_years.min()}")
        print(f"max year = {unique_years.max()}")
        print(f"n unique years = {len(unique_years)}")

        # missing years
        full_range = np.arange(unique_years.min(), unique_years.max() + 1)
        missing_years = sorted(set(full_range.tolist()) - set(unique_years.tolist()))
        print(f"missing years in continuous range = {missing_years}")

        # duplicates
        time_str = np.array([f"{y:04d}-{m:02d}-{d:02d}" for y, m, d in zip(years, months, days)])
        uniq_time, uniq_idx, uniq_counts = np.unique(time_str, return_index=True, return_counts=True)
        dup_mask = uniq_counts > 1
        dup_times = uniq_time[dup_mask]
        print(f"\n[Duplicate time check]")
        print(f"n duplicated timestamps = {dup_mask.sum()}")
        if dup_mask.sum() > 0:
            print("First duplicated timestamps:")
            for s, c in zip(dup_times[:20], uniq_counts[dup_mask][:20]):
                print(f"  {s}  count={c}")

        # monotonic check
        is_sorted = np.all(time_str[:-1] <= time_str[1:])
        print(f"\n[Monotonic time] sorted = {is_sorted}")
        if not is_sorted:
            bad_idx = np.where(time_str[:-1] > time_str[1:])[0]
            print("First unsorted pairs:")
            for i in bad_idx[:10]:
                print(f"  idx {i}: {time_str[i]}  >  {time_str[i+1]}")

        # count by year (raw)
        print("\n[Counts by year: raw]")
        year_counts_raw = {}
        for y in unique_years:
            year_counts_raw[int(y)] = int((years == y).sum())

        # count by year excluding Feb29
        keep_mask = ~((months == 2) & (days == 29))
        years_noleap = years[keep_mask]

        print("\n[Counts by year: after dropping Feb29]")
        year_counts_noleap = {}
        for y in unique_years:
            year_counts_noleap[int(y)] = int((years_noleap == y).sum())

        # Print full table near boundaries and abnormal years
        abnormal = [y for y, c in year_counts_noleap.items() if c != 365]
        print(f"years with noleap count != 365: {abnormal}")

        def print_year_table(year_list, title):
            print(f"\n[{title}]")
            print(" year | raw_count | noleap_count | first_date   | last_date")
            print("-" * 68)
            for y in year_list:
                mask_y = (years == y)
                if mask_y.sum() == 0:
                    print(f"{y:5d} | {'MISSING':>9} | {'MISSING':>12} | {'-':>10} | {'-':>10}")
                    continue
                idx = np.where(mask_y)[0]
                first_date = time_str[idx[0]]
                last_date  = time_str[idx[-1]]
                print(f"{y:5d} | {year_counts_raw[y]:9d} | {year_counts_noleap[y]:12d} | {first_date:10s} | {last_date:10s}")

        # boundary windows
        boundary_years = [y for y in range(max(unique_years.min(), 100), min(unique_years.max(), 110) + 1)]
        print_year_table(boundary_years, "Years around 100-110")

        tail_years = [y for y in range(max(unique_years.max() - 5, unique_years.min()), unique_years.max() + 1)]
        print_year_table(tail_years, "Last years")

        if abnormal:
            print_year_table(abnormal[:30], "Abnormal noleap-count years (first 30)")

        # show dates around 103/104/105
        print("\n[Sample timestamps around boundary 103/104/105]")
        boundary_idx = np.where((years >= 103) & (years <= 105))[0]
        if len(boundary_idx) == 0:
            print("No timestamps with year 103-105 found.")
        else:
            i0 = max(boundary_idx[0] - 5, 0)
            i1 = min(boundary_idx[-1] + 6, len(time_str))
            for i in range(i0, i1):
                print(f"{i:8d}  {time_str[i]}")

        # low25 availability in EPFLUX file
        print("\n[LOW25 year availability in EPFLUX file]")
        for y in low25_years:
            raw_c = year_counts_raw.get(y, 0)
            nl_c = year_counts_noleap.get(y, 0)
            ok = (nl_c == 365)
            print(f"  y={y:4d} | raw_count={raw_c:3d} | noleap_count={nl_c:3d} | usable={ok}")

        # also check y-1 and y pairs for Oct(y-1)-Sep(y) composites
        print("\n[LOW25 pair availability for Oct(y-1)-Sep(y)]")
        for y in low25_years:
            c_prev = year_counts_noleap.get(y - 1, 0)
            c_curr = year_counts_noleap.get(y, 0)
            ok_pair = (c_prev == 365 and c_curr == 365)
            print(f"  target_y={y:4d} | prev={y-1:4d}:{c_prev:3d} | curr={y:4d}:{c_curr:3d} | pair_ok={ok_pair}")


def main():
    for exp in ["B2000WCN", "NOCOUPL"]:
        inspect_epflux_file(exp, FILES[exp], TXT_FILES[exp])


if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
import gc
import glob
import warnings
import numpy as np
import xarray as xr
import numba as nb

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# ============================================================
# Paths & Config
# ============================================================
ROOT = "/mnt/soclim0/public_data/weiji"
TXT_ROOT = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"

PATHS = {
    "BWCN": os.path.join(ROOT, "BWCN"),
    "B2000WCN": os.path.join(ROOT, "B2000WCN001002"),
    "NOCOUPL": os.path.join(ROOT, "B2000WCN_NOCOUPL001002"),
    "MERRA2": os.path.join(ROOT, "MERRA2_Processed"),
}

# ---------------- user-facing debug switches ----------------
UTO3_DEBUG = True       # U/T/O3 文件年 <-> 物理年调试
EPFLUX_DEBUG = True     # EPFLUX 单文件物理年拼接调试
DEBUG_ONLY_BOUNDARY = False
BOUNDARY_MIN = 94
BOUNDARY_MAX = 110

# ---------------- output path ----------------
def get_out_path(exp, var_key):
    prefix_map = {
        "BWCN": "BWCN",
        "B2000WCN": "B2000WCN",
        "NOCOUPL": "B2000WCN_NOCOUPL",
        "MERRA2": "MERRA2"
    }
    prefix = prefix_map[exp]
    return os.path.join(PATHS[exp], f"{prefix}_{var_key}_LOW25_climatology_1to100hPa.nc")

# ---------------- target pressure ----------------
TARGET_PLEV_HPA = np.array(
    [100.0, 70.0, 50.0, 40.0, 30.0, 20.0, 10.0, 7.0, 5.0, 4.0, 3.0, 2.0, 1.0],
    dtype=np.float64
)
TARGET_PLEV_PA = TARGET_PLEV_HPA * 100.0

# ---------------- constants ----------------
MW_AIR = 28.9644
MW_O3 = 47.9982
MOL_MASS_RATIO = MW_AIR / MW_O3

# ---------------- variable specs ----------------
VAR_SPECS = {
    "U": {
        "var_name": "U",
        "lat_mode": "global",
        "scale_factor_model": 1.0,
        "scale_factor_merra": 1.0,
        "plot_unit": "m/s",
        "long_name": "Zonal wind",
    },
    "T": {
        "var_name": "T",
        "lat_mode": "global",
        "scale_factor_model": 1.0,
        "scale_factor_merra": 1.0,
        "plot_unit": "K",
        "long_name": "Temperature",
    },
    "O3": {
        "var_name": "O3",
        "lat_mode": "global",
        "scale_factor_model": 1e6,
        "scale_factor_merra": MOL_MASS_RATIO * 1e6,
        "plot_unit": "ppmv",
        "long_name": "Ozone mixing ratio",
    },
    "EPFLUX": {
        "var_name": "Fz",
        "lat_mode": None,
        "scale_factor_model": 1.0,
        "scale_factor_merra": 1.0,
        "plot_unit": "native",
        "long_name": "Vertical EP flux (Fz)",
    },
}

# ============================================================
# Numba Fast Interp
# ============================================================
@nb.njit(parallel=True)
def fast_logp_interp_flat(p_flat, v_flat, target_p):
    n_profs = p_flat.shape[0]
    nlev = p_flat.shape[1]
    ntarg = len(target_p)
    out = np.empty((n_profs, ntarg), dtype=np.float32)
    ltp = np.log(target_p)

    for i in nb.prange(n_profs):
        p, v = p_flat[i], v_flat[i]
        valid = 0
        for k in range(nlev):
            if p[k] > 0 and not np.isnan(p[k]) and not np.isnan(v[k]):
                valid += 1

        if valid < 2:
            for t_idx in range(ntarg):
                out[i, t_idx] = np.nan
            continue

        p_valid = np.empty(valid, dtype=np.float64)
        v_valid = np.empty(valid, dtype=np.float64)
        idx = 0
        for k in range(nlev):
            if p[k] > 0 and not np.isnan(p[k]) and not np.isnan(v[k]):
                p_valid[idx], v_valid[idx] = p[k], v[k]
                idx += 1

        order = np.argsort(p_valid)
        p_sorted, v_sorted = p_valid[order], v_valid[order]
        lp = np.log(p_sorted)
        out_i = np.interp(ltp, lp, v_sorted)

        slope_top = (v_sorted[1] - v_sorted[0]) / (lp[1] - lp[0])
        slope_bottom = (v_sorted[-1] - v_sorted[-2]) / (lp[-1] - lp[-2])

        for t_idx in range(ntarg):
            if ltp[t_idx] < lp[0]:
                out[i, t_idx] = v_sorted[0] + slope_top * (ltp[t_idx] - lp[0])
            elif ltp[t_idx] > lp[-1]:
                out[i, t_idx] = v_sorted[-1] + slope_bottom * (ltp[t_idx] - lp[-1])
            else:
                out[i, t_idx] = out_i[t_idx]
    return out

# ============================================================
# Core Helpers
# ============================================================
def debug_year_enabled(y):
    if not DEBUG_ONLY_BOUNDARY:
        return True
    return (BOUNDARY_MIN <= int(y) <= BOUNDARY_MAX)

def get_ymd_from_time_value(tt):
    if hasattr(tt, "year") and hasattr(tt, "month"):
        return int(tt.year), int(tt.month), int(tt.day)
    y, m, d = np.datetime_as_string(tt, unit="D").split("-")
    return int(y), int(m), int(d)

def parse_file_year(path):
    m_all = re.findall(r'(?<!\d)(\d{4})(?!\d)', os.path.basename(path))
    return int(m_all[-1]) if m_all else None

def detect_coord_name(ds, candidates):
    for c in candidates:
        if c in ds.coords or c in ds.dims:
            return c
    return None

def detect_main_var(ds, preferred=None):
    if preferred is not None and preferred in ds.data_vars:
        return preferred
    for v in ds.data_vars:
        if "time" in ds[v].dims:
            return v
    raise ValueError("Cannot detect main variable.")

def drop_feb29_and_get_time_parts(time_vals):
    keep_idx, years, months, days = [], [], [], []
    for i, tt in enumerate(time_vals):
        y, m, d = get_ymd_from_time_value(tt)
        if not (m == 2 and d == 29):
            keep_idx.append(i)
            years.append(y)
            months.append(m)
            days.append(d)
    return np.array(keep_idx), np.array(years), np.array(months), np.array(days)

def logp_interp_extrap_1d(p_prof, v_prof, target_p):
    p = np.asarray(p_prof, dtype=np.float64)
    v = np.asarray(v_prof, dtype=np.float64)
    tp = np.asarray(target_p, dtype=np.float64)
    mask = np.isfinite(p) & np.isfinite(v) & (p > 0)
    p, v = p[mask], v[mask]
    if p.size < 2:
        return np.full(tp.shape, np.nan, dtype=np.float64)
    order = np.argsort(p)
    p, v = p[order], v[order]
    lp, ltp = np.log(p), np.log(tp)
    out = np.interp(ltp, lp, v)

    mask_top = ltp < lp[0]
    if np.any(mask_top):
        out[mask_top] = v[0] + ((v[1] - v[0]) / (lp[1] - lp[0])) * (ltp[mask_top] - lp[0])

    mask_bottom = ltp > lp[-1]
    if np.any(mask_bottom):
        out[mask_bottom] = v[-1] + ((v[-1] - v[-2]) / (lp[-1] - lp[-2])) * (ltp[mask_bottom] - lp[-1])

    return out

def interp_fixed_pressure_clim_to_target(clim_var_native, native_plev_hpa, target_plev_hpa):
    native_pa = np.asarray(native_plev_hpa, dtype=np.float64) * 100.0
    target_pa = np.asarray(target_plev_hpa, dtype=np.float64) * 100.0
    out = np.full((365, len(target_plev_hpa)), np.nan, dtype=np.float32)
    for d in range(365):
        out[d, :] = logp_interp_extrap_1d(native_pa, clim_var_native[d, :], target_pa).astype(np.float32)
    return out

def build_clim_dataarrays_from_octsep(clim_os_target, plev_hpa, lat_vals=None, lon_vals=None):
    if lat_vals is not None and lon_vals is not None:
        arr_os = np.transpose(clim_os_target, (1, 2, 3, 0)).astype(np.float32)
        jd_data = np.concatenate([clim_os_target[92:, ...], clim_os_target[:92, ...]], axis=0)
        arr_jd = np.transpose(jd_data, (1, 2, 3, 0)).astype(np.float32)
        coords_jd = {
            "plev": plev_hpa.astype(np.float32),
            "lat": lat_vals.astype(np.float32),
            "lon": lon_vals.astype(np.float32),
            "dayofyear": np.arange(1, 366),
        }
        coords_os = {
            "plev": plev_hpa.astype(np.float32),
            "lat": lat_vals.astype(np.float32),
            "lon": lon_vals.astype(np.float32),
            "time_index": np.arange(365),
        }
        dims, dims_os = ("plev", "lat", "lon", "dayofyear"), ("plev", "lat", "lon", "time_index")
    else:
        arr_os = np.transpose(clim_os_target, (1, 0)).astype(np.float32)
        jd_data = np.concatenate([clim_os_target[92:, :], clim_os_target[:92, :]], axis=0)
        arr_jd = np.transpose(jd_data, (1, 0)).astype(np.float32)
        coords_jd = {"plev": plev_hpa.astype(np.float32), "dayofyear": np.arange(1, 366)}
        coords_os = {"plev": plev_hpa.astype(np.float32), "time_index": np.arange(365)}
        dims, dims_os = ("plev", "dayofyear"), ("plev", "time_index")

    clim_jd = xr.DataArray(arr_jd, coords=coords_jd, dims=dims, name="clim_jandec")
    clim_os = xr.DataArray(arr_os, coords=coords_os, dims=dims_os, name="clim_octsep")
    return clim_jd, clim_os

def save_targeted_climatology(exp_name, var_key, clim_jd, clim_os, long_name, plot_unit):
    ds_out = xr.Dataset(data_vars={"clim_jandec": clim_jd, "clim_octsep": clim_os})
    ds_out.attrs.update({
        "experiment": exp_name,
        "description": "Composite of lowest 25% extreme years (Physically accurate Oct[n-1] to Sep[n])",
        "variable": var_key,
        "long_name": long_name,
        "plot_unit": plot_unit,
        "target_levels_hPa": ",".join([str(x) for x in TARGET_PLEV_HPA.tolist()]),
    })
    out_nc = get_out_path(exp_name, var_key)
    ds_out.to_netcdf(out_nc)
    print(f"  [SAVED] {out_nc}")

def scan_merra_files(var_key):
    subdir_map = {"U": "U", "T": "T", "O3": "O3", "EPFLUX": "EPflux_daily"}
    files = sorted(glob.glob(os.path.join(PATHS["MERRA2"], subdir_map[var_key], "*.nc")))
    if not files:
        raise FileNotFoundError(f"No MERRA2 files for {var_key}")
    return files

# ============================================================
# U/T/O3: WACCM physical-year <-> file-year mapping
# ============================================================
def physical_year_to_file_year(exp_name, y_phys):
    """
    输入 low25 txt 中的物理年，返回 U/T/O3 单年文件真正应读取的文件年。
    """
    y_phys = int(y_phys)

    if exp_name == "BWCN":
        return y_phys

    if exp_name == "B2000WCN":
        if 1 <= y_phys <= 103:
            return y_phys
        elif 104 <= y_phys <= 209:
            return y_phys + 1
        else:
            return None

    if exp_name == "NOCOUPL":
        if 1 <= y_phys <= 103:
            return y_phys
        elif 104 <= y_phys <= 204:
            return y_phys + 1
        else:
            return None

    return y_phys

def is_valid_waccm_file_year(exp_name, file_year):
    if file_year is None:
        return False

    if exp_name == "B2000WCN":
        return (1 <= file_year <= 103) or (105 <= file_year <= 210)

    if exp_name == "NOCOUPL":
        return (1 <= file_year <= 103) or (105 <= file_year <= 205)

    if exp_name == "BWCN":
        return True

    return True

def find_exact_year_file(exp_name, var_key, file_year):
    if file_year is None:
        return None

    pattern = os.path.join(PATHS[exp_name], var_key, f"*{file_year:04d}*.nc")
    matches = sorted(glob.glob(pattern))
    matches = [f for f in matches if parse_file_year(f) == file_year]

    if len(matches) == 0:
        return None
    if len(matches) > 1:
        print(f"[WARN] Multiple matches for {exp_name}-{var_key} file_year={file_year}:")
        for m in matches:
            print("   ", m)
        print(f"[WARN] Using first match: {matches[0]}")
    return matches[0]

# ============================================================
# STEP 1: Direct Load from TXT
# ============================================================
def load_low25_years_from_txt():
    exp_folder_map = {
        "MERRA2": "MERRA2",
        "BWCN": "BWCN",
        "B2000WCN": "B2000WCN",
        "NOCOUPL": "B2000WCN_NOCOUPL",
    }
    low_years_dict = {}
    for exp, folder in exp_folder_map.items():
        txt_path = os.path.join(TXT_ROOT, folder, "low25pct_years_30_70hPa.txt")
        if os.path.exists(txt_path):
            years = np.loadtxt(txt_path, dtype=int)
            years_list = [int(years)] if years.ndim == 0 else [int(x) for x in years.tolist()]
            low_years_dict[exp] = years_list
            print(f"[✔] 成功读取 {exp}: 包含 {len(years_list)} 年 -> {txt_path}")
            print(f"    -> 具体年份: {years_list}")
        else:
            print(f"[✘] 找不到文件 {exp}: {txt_path}")
            low_years_dict[exp] = []
    return low_years_dict

# ============================================================
# STEP 2A: U/T/O3 Targeted Climatology Builders
# ============================================================
def get_model_hybrid_part(exp_name, var_key, spec, y_phys, part):
    """
    这里的 y_phys 一律解释为“物理年”，与 low25 txt 保持一致。
    对 B2000WCN / NOCOUPL 会自动映射到正确的文件年。
    """
    file_year = physical_year_to_file_year(exp_name, y_phys)

    if not is_valid_waccm_file_year(exp_name, file_year):
        if UTO3_DEBUG and debug_year_enabled(y_phys):
            print(f"[UTO3-DEBUG][INVALID-FILE-YEAR] {exp_name}-{var_key} y_phys={y_phys} -> file_year={file_year}")
        return None, None, None

    file_path = find_exact_year_file(exp_name, var_key, file_year)
    if file_path is None:
        print(f"[WARN] Missing file for {exp_name}-{var_key}: physical_year={y_phys}, file_year={file_year}")
        return None, None, None

    ds = xr.open_dataset(file_path, decode_times=True)
    parsed_file_year = parse_file_year(file_path)

    # run002 -> 偏移到物理年
    if exp_name in ["B2000WCN", "NOCOUPL"] and parsed_file_year >= 105:
        old_times = ds.time.values
        new_times = [t.replace(year=t.year + 103) for t in old_times]
        ds = ds.assign_coords(time=new_times)

    da = ds[spec["var_name"]]
    ps = ds["PS"]

    lat_name = detect_coord_name(ds, ["lat", "latitude"])
    lon_name = detect_coord_name(ds, ["lon", "longitude"])
    lat_vals, lon_vals = ds[lat_name].values, ds[lon_name].values

    da = da.transpose("time", "lev", lat_name, lon_name)
    ps = ps.transpose("time", lat_name, lon_name)

    hyam = ds["hyam"].values.astype(np.float32)
    hybm = ds["hybm"].values.astype(np.float32)
    p0 = np.float32(ds["P0"].values)

    keep_idx, years_kept, months_kept, days_kept = drop_feb29_and_get_time_parts(ds["time"].values)
    if len(keep_idx) != 365:
        print(f"[WARN] Bad day count for {exp_name}-{var_key}: file={os.path.basename(file_path)}, "
              f"physical_year={y_phys}, file_year={file_year}, len_keep={len(keep_idx)}")
        ds.close()
        return None, None, None

    unique_years = np.unique(years_kept)
    if not (len(unique_years) == 1 and int(unique_years[0]) == int(y_phys)):
        print(f"[WARN] Physical-year mismatch for {exp_name}-{var_key}: "
              f"target_phys={y_phys}, file={os.path.basename(file_path)}, "
              f"kept_unique_years={unique_years.tolist()}")
        ds.close()
        return None, None, None

    if UTO3_DEBUG and debug_year_enabled(y_phys):
        sel0 = 273 if part == "prev" else 0
        sel1 = 364 if part == "prev" else 272
        print(
            f"[UTO3-DEBUG] {exp_name}-{var_key} "
            f"target_phys={y_phys:04d} part={part} "
            f"file_year={file_year:04d} file={os.path.basename(file_path)} "
            f"shifted={'YES' if (exp_name in ['B2000WCN','NOCOUPL'] and parsed_file_year >= 105) else 'NO'} "
            f"slice={years_kept[sel0]:04d}-{months_kept[sel0]:02d}-{days_kept[sel0]:02d}"
            f" -> {years_kept[sel1]:04d}-{months_kept[sel1]:02d}-{days_kept[sel1]:02d}"
        )

    sel = np.arange(273, 365) if part == "prev" else np.arange(0, 273)
    keep_idx_sel = keep_idx[sel]

    da_kept = da.isel(time=keep_idx_sel).values.astype(np.float32) * np.float32(spec["scale_factor_model"])
    ps_kept = ps.isel(time=keep_idx_sel).values.astype(np.float32)

    nt, nlev, nlat, nlon = da_kept.shape
    p_native = hyam[None, :, None, None] * p0 + hybm[None, :, None, None] * ps_kept[:, None, :, :]

    var_flat = da_kept.transpose(0, 2, 3, 1).reshape(-1, nlev).astype(np.float64)
    p_flat = p_native.transpose(0, 2, 3, 1).reshape(-1, nlev).astype(np.float64)
    out_flat = fast_logp_interp_flat(p_flat, var_flat, TARGET_PLEV_PA)
    var_interp_4d = out_flat.reshape(nt, nlat, nlon, len(TARGET_PLEV_PA)).transpose(0, 3, 1, 2)

    ds.close()
    return var_interp_4d, lat_vals, lon_vals

def accumulate_model_hybrid_targeted_safe(exp_name, var_key, spec, target_years):
    if not target_years:
        return None, None, None

    lat_vals, lon_vals, sum_var, count = None, None, None, 0

    for y in tqdm(target_years, desc=f"{exp_name}-{var_key} (Serial Safe)", ncols=100):
        v_prev, lat, lon = get_model_hybrid_part(exp_name, var_key, spec, y - 1, 'prev')
        v_curr, _, _ = get_model_hybrid_part(exp_name, var_key, spec, y, 'curr')

        if v_prev is not None and v_curr is not None:
            v_stitched = np.concatenate([v_prev, v_curr], axis=0)
            if sum_var is None:
                lat_vals, lon_vals = lat, lon
                sum_var = np.zeros_like(v_stitched, dtype=np.float64)
            sum_var += v_stitched
            count += 1

            if UTO3_DEBUG and debug_year_enabled(y):
                print(f"[UTO3-DEBUG][OK-COMPOSITE] {exp_name}-{var_key} target_y={y:04d} built Oct({y-1})-Sep({y})")
        else:
            print(f"[SKIP] {exp_name}-{var_key}: cannot build Oct({y-1})-Sep({y})")

        del v_prev, v_curr
        gc.collect()

    if count == 0:
        return None, None, None

    return (sum_var / count).astype(np.float32), lat_vals, lon_vals

def get_merra_clim_part(var_key, spec, y, part):
    subdir_map = {"U": "U", "T": "T", "O3": "O3", "EPFLUX": "EPflux_daily"}
    files = glob.glob(os.path.join(PATHS["MERRA2"], subdir_map[var_key], f"*{y}*.nc"))
    if not files:
        return None, None, None

    ds = xr.open_dataset(files[0], decode_times=True)
    da = ds[detect_main_var(ds, preferred=spec["var_name"])]
    lat_name = detect_coord_name(ds, ["lat", "latitude"])
    lon_name = detect_coord_name(ds, ["lon", "longitude"])
    da = da.rename({detect_coord_name(ds, ["plev", "lev", "level"]): "plev"})

    src = da["plev"].values.astype(np.float64)
    sel_idx = [int(np.argmin(np.abs(src - t))) for t in TARGET_PLEV_HPA]
    da = da.isel(plev=sel_idx).assign_coords(plev=("plev", TARGET_PLEV_HPA))

    lat_vals = da[lat_name].values
    lon_vals = da[lon_name].values if lon_name else None
    da = da.transpose("time", "plev", lat_name, lon_name) if lon_name else da.transpose("time", "plev", lat_name)

    keep_idx, years_kept, months_kept, days_kept = drop_feb29_and_get_time_parts(da["time"].values)
    keep_idx = keep_idx[273:] if part == 'prev' else keep_idx[:273]

    if UTO3_DEBUG and debug_year_enabled(y):
        if len(keep_idx) > 0:
            i0, i1 = keep_idx[0], keep_idx[-1]
            print(
                f"[UTO3-DEBUG][MERRA2] {var_key} y={y:04d} part={part} "
                f"slice={years_kept[0]:04d}-{months_kept[0]:02d}-{days_kept[0]:02d}"
                f" -> {years_kept[len(keep_idx)-1]:04d}-{months_kept[len(keep_idx)-1]:02d}-{days_kept[len(keep_idx)-1]:02d}"
            )

    da = da.isel(time=keep_idx)
    arr = da.values.astype(np.float32) * np.float32(spec["scale_factor_merra"])
    ds.close()
    return arr, lat_vals, lon_vals

def accumulate_merra_targeted_safe(var_key, spec, target_years):
    lat_vals, lon_vals, sum_doy, count = None, None, None, 0
    for y in tqdm(target_years, desc=f"MERRA2-{var_key} (Serial Safe)", ncols=100):
        v_prev, lat, lon = get_merra_clim_part(var_key, spec, y - 1, 'prev')
        v_curr, _, _ = get_merra_clim_part(var_key, spec, y, 'curr')

        if v_prev is not None and v_curr is not None:
            v_stitched = np.concatenate([v_prev, v_curr], axis=0)
            if sum_doy is None:
                lat_vals, lon_vals = lat, lon
                sum_doy = np.zeros_like(v_stitched, dtype=np.float64)
            sum_doy += v_stitched
            count += 1
            if UTO3_DEBUG and debug_year_enabled(y):
                print(f"[UTO3-DEBUG][OK-COMPOSITE] MERRA2-{var_key} target_y={y:04d} built Oct({y-1})-Sep({y})")

        del v_prev, v_curr
        gc.collect()

    if count == 0:
        return None, None, None

    return (sum_doy / count).astype(np.float32), lat_vals, lon_vals

# ============================================================
# STEP 2B: EPFLUX single-file targeted climatology (FINAL FIX)
# ============================================================
def get_epflux_singlefile_path(exp_name):
    ep_dir = os.path.join(PATHS[exp_name], "EPflux_daily")
    if not os.path.isdir(ep_dir):
        print(f"[WARN] EPflux directory not found for {exp_name}: {ep_dir}")
        return None

    files = sorted(glob.glob(os.path.join(ep_dir, "*.nc")))
    if len(files) == 0:
        print(f"[WARN] No EPFLUX nc file found for {exp_name} in {ep_dir}")
        return None
    if len(files) > 1:
        print(f"[WARN] Multiple EPFLUX nc files found for {exp_name} in {ep_dir}:")
        for f in files:
            print("   ", f)
        print(f"[WARN] Using first file: {files[0]}")
    return files[0]


def load_epflux_singlefile(exp_name, spec):
    """
    读取 EPFLUX 单文件，并统一输出：
      da            : (time, plev)
      native_plev_hpa
      year_counts   : 去掉 Feb29 后每个物理年的天数
      years, months, days : 去掉 Feb29 后的时间分量
    """
    path = get_epflux_singlefile_path(exp_name)
    if path is None or (not os.path.exists(path)):
        print(f"[WARN] EPFLUX file not found for {exp_name}: {path}")
        return None, None, None, None, None, None, None

    ds = xr.open_dataset(path, decode_times=True)

    vname = detect_main_var(ds, preferred=spec["var_name"])
    da = ds[vname]

    p_name = detect_coord_name(ds, ["plev", "lev", "level"])
    if p_name is None:
        print(f"[WARN] Cannot detect pressure coordinate in EPFLUX file for {exp_name}: {path}")
        ds.close()
        return None, None, None, None, None, None, None

    if p_name != "plev":
        da = da.rename({p_name: "plev"})

    da = da.transpose("time", "plev")

    src = da["plev"].values.astype(np.float64)

    # pressure normalize to hPa
    if np.nanmax(src) > 2000:
        src_hpa = src / 100.0
        da = da.assign_coords(plev=("plev", src_hpa))
    else:
        src_hpa = src.copy()

    native_mask = (src_hpa >= 1.0) & (src_hpa <= 100.0)
    da = da.isel(plev=native_mask)
    native_plev_hpa = da["plev"].values.astype(np.float64)

    # ---- robust time extraction: use xarray .dt directly ----
    try:
        years_all = da["time"].dt.year.values.astype(int)
        months_all = da["time"].dt.month.values.astype(int)
        days_all = da["time"].dt.day.values.astype(int)
    except Exception as e:
        print(f"[WARN] EPFLUX dt accessor failed for {exp_name}: {type(e).__name__}: {e}")
        ds.close()
        return None, None, None, None, None, None, None

    keep_mask = ~((months_all == 2) & (days_all == 29))

    da = da.isel(time=keep_mask)
    years = years_all[keep_mask]
    months = months_all[keep_mask]
    days = days_all[keep_mask]

    year_counts = {}
    for y in np.unique(years):
        year_counts[int(y)] = int((years == y).sum())

    if EPFLUX_DEBUG:
        uniq_years = np.unique(years)
        print(f"[EPFLUX-DEBUG][LOAD] {exp_name}")
        print(f"  file = {path}")
        print(f"  time type = {type(da['time'].values[0])}")
        print(f"  year range = {int(uniq_years.min())} -> {int(uniq_years.max())}")
        print(f"  n_years = {len(uniq_years)}")
        bad_years = [(int(y), int(year_counts[int(y)])) for y in uniq_years if year_counts[int(y)] != 365]
        print(f"  bad years (count != 365) = {bad_years}")

    return ds, da, native_plev_hpa, year_counts, years, months, days


def process_epflux_targeted_safe(exp_name, spec, target_years):
    """
    EPFLUX 单文件版本：
    直接使用单文件中已经构造好的“物理年 time.year”。
    不做 WACCM file-year 映射。
    """
    if not target_years:
        return None

    ds, da, native_plev_hpa, year_counts, years, months, days = load_epflux_singlefile(exp_name, spec)
    if da is None:
        return None

    sum_doy = np.zeros((365, len(native_plev_hpa)), dtype=np.float64)
    count = 0

    for y in tqdm(target_years, desc=f"{exp_name}-EPFLUX (SingleFile)", ncols=100):
        c_prev = year_counts.get(int(y - 1), 0)
        c_curr = year_counts.get(int(y), 0)

        if EPFLUX_DEBUG and debug_year_enabled(y):
            print(
                f"[EPFLUX-DEBUG][CHECK] {exp_name} "
                f"target_y={y:04d} prev={y-1:04d}:{c_prev} curr={y:04d}:{c_curr}"
            )

        if not (c_prev == 365 and c_curr == 365):
            print(
                f"[SKIP][EPFLUX] {exp_name}: cannot build Oct({y-1})-Sep({y}) "
                f"because prev_count={c_prev}, curr_count={c_curr}"
            )
            continue

        mask_prev = (years == (y - 1))
        mask_curr = (years == y)

        v_prev = da.isel(time=mask_prev).values.astype(np.float32)
        v_curr = da.isel(time=mask_curr).values.astype(np.float32)

        if v_prev.shape[0] != 365 or v_curr.shape[0] != 365:
            print(
                f"[SKIP][EPFLUX] {exp_name}: bad year length after mask "
                f"for target_y={y}, prev={v_prev.shape[0]}, curr={v_curr.shape[0]}"
            )
            continue

        # Oct-Dec(prev) + Jan-Sep(curr)
        stitched = np.concatenate([v_prev[273:], v_curr[:273]], axis=0)

        if stitched.shape[0] != 365:
            print(f"[SKIP][EPFLUX] {exp_name}: stitched length != 365 for target_y={y}, got={stitched.shape[0]}")
            continue

        if EPFLUX_DEBUG and debug_year_enabled(y):
            print(
                f"[EPFLUX-DEBUG][OK] {exp_name} target_y={y:04d} "
                f"prev_slice={y-1:04d}-10-01 -> {y-1:04d}-12-31 ; "
                f"curr_slice={y:04d}-01-01 -> {y:04d}-09-30 ; "
                f"stitched_shape={stitched.shape}"
            )

        sum_doy += stitched
        count += 1

    ds.close()

    if count == 0:
        return None

    clim_native = sum_doy / count
    return interp_fixed_pressure_clim_to_target(clim_native, native_plev_hpa, TARGET_PLEV_HPA)

# ============================================================
# Main Execution
# ============================================================
if __name__ == "__main__":
    print("=" * 80)
    print(" STEP 1: Fast Loading Lowest 25% Years from existing TXT")
    print("=" * 80)

    low_years_dict = load_low25_years_from_txt()

    print("\n" + "=" * 80)
    print(" STEP 2: Building and Saving Physically-Accurate Low 25% Climatologies")
    print("=" * 80)

    # --------------------------------------------------------
    # Part A: U / T / O3
    # --------------------------------------------------------
    for var_key in ["U", "T", "O3"]:
        spec = VAR_SPECS[var_key]
        print(f"\n---> Processing {var_key}")

        for exp in ["MERRA2", "BWCN", "B2000WCN", "NOCOUPL"]:
            low_years = low_years_dict.get(exp, [])
            if not low_years:
                continue

            if exp == "MERRA2":
                clim, lat, lon = accumulate_merra_targeted_safe(var_key, spec, low_years)
            else:
                clim, lat, lon = accumulate_model_hybrid_targeted_safe(exp, var_key, spec, low_years)

            if clim is not None:
                jd, os_ds = build_clim_dataarrays_from_octsep(clim, TARGET_PLEV_HPA, lat, lon)
                save_targeted_climatology(exp, var_key, jd, os_ds, spec["long_name"], spec["plot_unit"])

    # --------------------------------------------------------
    # Part B: EPFLUX
    # --------------------------------------------------------
    print(f"\n---> Processing EPFLUX")
    spec = VAR_SPECS["EPFLUX"]

    for exp in ["MERRA2", "BWCN", "B2000WCN", "NOCOUPL"]:
        low_years = low_years_dict.get(exp, [])
        if not low_years:
            continue

        clim = process_epflux_targeted_safe(exp, spec, low_years)
        if clim is not None:
            jd, os_ds = build_clim_dataarrays_from_octsep(clim, TARGET_PLEV_HPA)
            save_targeted_climatology(exp, "EPFLUX", jd, os_ds, spec["long_name"], spec["plot_unit"])

    print("\n[INFO] Full low25 climatology script finished.")
    print("[INFO] U/T/O3: WACCM single-year files use physical-year -> file-year mapping.")
    print("[INFO] EPFLUX: all experiments use single-file physical-year timeline directly.")

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

# ============================================================
# Paths & Prefixes (已适配新的输出路径)
# ============================================================
ROOT = "/mnt/soclim0/public_data/weiji"

PATHS = {
    "BWCN": os.path.join(ROOT, "BWCN"),
    "B2000WCN": os.path.join(ROOT, "B2000WCN001002"),
    "NOCOUPL": os.path.join(ROOT, "B2000WCN_NOCOUPL001002"),
    "MERRA2": os.path.join(ROOT, "MERRA2_Processed"),
}

PREFIX_MAP = {
    "BWCN": "BWCN",
    "B2000WCN": "B2000WCN",
    "NOCOUPL": "B2000WCN_NOCOUPL",
    "MERRA2": "MERRA2"
}

OUT_ROOT = "/home/weiji/restart_exam/code/20260226B2000WCN_BWCN_MERRA2_COMPARE/RESULT"
PLOT_OUT = os.path.join(OUT_ROOT, "plots")
os.makedirs(PLOT_OUT, exist_ok=True)

# ============================================================
# X-axis (Oct -> Sep)
# ============================================================
XTICKS = [0, 31, 61, 92, 123, 151, 182, 212, 243, 273, 304, 335]
XTICK_LABELS = ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar",
                "Apr", "May", "Jun", "Jul", "Aug", "Sep"]

# ============================================================
# Matplotlib style
# ============================================================
mpl.rc("xtick", direction="out", labelsize=9)
mpl.rc("ytick", direction="out", labelsize=9)
mpl.rc("axes", titlesize=11, labelsize=10)
mpl.rc("font", family="sans-serif")

# ============================================================
# Plot config
# ============================================================
PLOT_CFG = {
    "U": {
        "cmap": "RdBu_r",
        "cbar_label": "U (m/s)",
        "title": "Low 25% Climatology: Zonal Wind at 60°N",
        "extend": "both",
        "symmetric": True,
    },
    "T": {
        "cmap": "Spectral_r",
        "cbar_label": "T (K)",
        "title": "Low 25% Climatology: Polar-Cap Temperature (60–90°N)",
        "extend": "both",
        "symmetric": False,
    },
    "O3": {
        "cmap": "viridis",
        "cbar_label": "O$_3$ (ppmv)",
        "title": "Low 25% Climatology: Polar-Cap Ozone (60–90°N)",
        "extend": "both",
        "symmetric": False,
    },
    "EPFLUX": {
        "cmap": "PuOr_r",
        "cbar_label": "Fz (native)",
        "title": "Low 25% Climatology: EP Flux Fz",
        "extend": "both",
        "symmetric": True,
    },
}

DISPLAY_NAMES = {
    "MERRA2": "MERRA2",
    "COUPLED": "Coupled",
    "NOCOUPL": "NOCOUPL",
}

# ============================================================
# Helpers
# ============================================================
def robust_levels(values, symmetric=False, nlev=31):
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    if x.size == 0: return np.linspace(-1, 1, nlev)

    if symmetric:
        vmax = np.nanpercentile(np.abs(x), 98)
        if not np.isfinite(vmax) or vmax == 0: vmax = np.nanmax(np.abs(x))
        return np.linspace(-vmax, vmax, nlev)

    vmin, vmax = np.nanpercentile(x, 2), np.nanpercentile(x, 98)
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
        vmin, vmax = np.nanmin(x), np.nanmax(x)
    return np.linspace(vmin, vmax, nlev)

def process_spatial_reduction(da, var_key):
    """处理 4D 数据的空间降维"""
    if "lat" not in da.dims:
        return da # EPFLUX 已经是 2D

    if "lon" in da.dims:
        da = da.mean(dim="lon")

    if var_key == "U":
        return da.sel(lat=60.0, method="nearest")
    elif var_key in ["T", "O3"]:
        # 60-90°N 面积加权平均
        lat_min, lat_max = 60.0, 90.0
        lat_vals = da["lat"].values
        if lat_vals[0] > lat_vals[-1]: # 处理纬度倒序
            lat_min, lat_max = lat_max, lat_min
        
        da_band = da.sel(lat=slice(lat_min, lat_max))
        weights = np.cos(np.deg2rad(da_band["lat"]))
        return da_band.weighted(weights).mean(dim="lat")
    
    return da

def load_one_dataset_profile(dname, var_key):
    """读取并处理 Low 25 数据，支持 COUPLED 的物理加权合并逻辑"""
    
    if dname == "COUPLED":
        # 🚨🚨🚨 重要：请在这里填入上一步处理时打印出的各自提取到的 25% 的年数 🚨🚨🚨
        N_BWCN = 5   # 替换为你终端打印出的真实年数
        N_B2K = 51    # 替换为你终端打印出的真实年数
        
        da_bwcn, da_b2k = None, None
        
        path_bwcn = os.path.join(PATHS["BWCN"], f"BWCN_{var_key}_LOW25_climatology_1to100hPa.nc")
        if os.path.exists(path_bwcn):
            with xr.open_dataset(path_bwcn) as ds:
                da_bwcn = process_spatial_reduction(ds["clim_octsep"], var_key).load()
                
        path_b2k = os.path.join(PATHS["B2000WCN"], f"B2000WCN_{var_key}_LOW25_climatology_1to100hPa.nc")
        if os.path.exists(path_b2k):
            with xr.open_dataset(path_b2k) as ds:
                da_b2k = process_spatial_reduction(ds["clim_octsep"], var_key).load()

        if da_bwcn is not None and da_b2k is not None:
            # 科学加权平均
            da_os = (da_bwcn * N_BWCN + da_b2k * N_B2K) / (N_BWCN + N_B2K)
            path_info = f"Weighted Merge (BWCN:{N_BWCN}, B2000WCN:{N_B2K})"
        else:
            return None, None
            
    else:
        prefix = PREFIX_MAP[dname]
        file_path = os.path.join(PATHS[dname], f"{prefix}_{var_key}_LOW25_climatology_1to100hPa.nc")
        if not os.path.exists(file_path):
            print(f"[Skip] File not found: {file_path}")
            return None, None
        with xr.open_dataset(file_path) as ds:
            da_os = process_spatial_reduction(ds["clim_octsep"], var_key).load()
        path_info = file_path

    return da_os, path_info

# ============================================================
# Main Plotting
# ============================================================
def plot_low25_variable(var_key):
    datasets = ["MERRA2", "COUPLED", "NOCOUPL"]
    arr_list, used_names = [], []

    for dname in datasets:
        da, info = load_one_dataset_profile(dname, var_key)
        if da is None: continue
        arr_list.append(da)
        used_names.append(dname)
        print(f"[Load] {dname}: {info}")

    if not arr_list: return

    arr = xr.concat(arr_list, dim="dataset").assign_coords(dataset=used_names)
    plev, x = arr["plev"].values, arr["time_index"].values

    # EPFLUX 反转绘图
    plot_data = arr.values if var_key != "EPFLUX" else -arr.values
    levels = robust_levels(plot_data, symmetric=PLOT_CFG[var_key]["symmetric"])

    fig, axes = plt.subplots(1, len(used_names), figsize=(5.2 * len(used_names), 4.8),
                             sharex=True, sharey=True, constrained_layout=True)
    if len(used_names) == 1: axes = [axes]

    for i, dname in enumerate(used_names):
        ax, z = axes[i], arr.sel(dataset=dname).values
        if var_key == "EPFLUX": z = -z

        cf = ax.contourf(x, plev, z, levels=levels, cmap=PLOT_CFG[var_key]["cmap"],
                         extend=PLOT_CFG[var_key]["extend"], alpha=0.9)
        
        cs = ax.contour(x, plev, z, levels=10, colors="k", linewidths=0.4, alpha=0.6)
        ax.clabel(cs, inline=True, fontsize=6, fmt="%.2g")

        if var_key in ["U", "EPFLUX"]:
            try: ax.contour(x, plev, z, levels=[0], colors="k", linewidths=1.2)
            except: pass

        ax.set_yscale("log")
        ax.invert_yaxis()
        ax.set_ylim(100, 1)
        ax.set_yticks(plev)
        ax.yaxis.set_major_formatter(ScalarFormatter())
        ax.set_xticks(XTICKS)
        ax.set_xticklabels(XTICK_LABELS)
        ax.grid(True, linestyle="--", alpha=0.2)
        ax.set_title(DISPLAY_NAMES.get(dname, dname), fontweight="bold")
        if i == 0: ax.set_ylabel("Pressure (hPa)")

    cbar = fig.colorbar(cf, ax=axes, pad=0.02, shrink=0.9)
    cbar.set_label(PLOT_CFG[var_key]["cbar_label"])
    fig.suptitle(PLOT_CFG[var_key]["title"], fontsize=14, y=1.02)

    out_name = f"{var_key}_LOW25_climatology_OctSep"
    plt.savefig(os.path.join(PLOT_OUT, f"{out_name}.png"), dpi=300, bbox_inches="tight")
    plt.show() # 避免在后台积攒过多的图表导致内存占用

if __name__ == "__main__":
    for var_key in ["U", "T", "O3", "EPFLUX"]:
        plot_low25_variable(var_key)
    print("\n✅ 所有图片已生成并保存至:", PLOT_OUT)

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

# ============================================================
# Paths
# ============================================================
OUT_ROOT = "/home/weiji/restart_exam/code/20260226B2000WCN_BWCN_MERRA2_COMPARE/RESULT"
COMPARE_OUT = os.path.join(OUT_ROOT, "data")
PLOT_OUT = os.path.join(OUT_ROOT, "plots")
os.makedirs(PLOT_OUT, exist_ok=True)

XTICKS = [0, 31, 61, 92, 123, 151, 182, 212, 243, 273, 304, 335]
XTICK_LABELS = ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep"]

mpl.rc("xtick", direction="out", labelsize=9)
mpl.rc("ytick", direction="out", labelsize=9)
mpl.rc("axes", titlesize=11, labelsize=10)
mpl.rc("font", family="sans-serif")

PLOT_CFG_DIFF = {
    "U": {"cmap": "RdBu_r", "cbar_label": "Diff: U (m/s)", "title": "Anomaly (Low 25 - Climatology): Zonal Wind at 60°N"},
    "T": {"cmap": "RdBu_r", "cbar_label": "Diff: T (K)", "title": "Anomaly (Low 25 - Climatology): Temperature at 60°N"},
    "O3": {"cmap": "PiYG", "cbar_label": "Diff: O$_3$ (ppmv)", "title": "Anomaly (Low 25 - Climatology): Polar-Cap O$_3$"},
    "EPFLUX": {"cmap": "PuOr_r", "cbar_label": "Diff: Fz (native)", "title": "Anomaly (Low 25 - Climatology): EP Flux Fz"},
}

MW_AIR = 28.9644
MW_O3 = 48.0
MICRO_KGKG_TO_PPMV = MW_AIR / MW_O3

# ============================================================
# Helpers
# ============================================================
def robust_symmetric_levels(values, nlev=31):
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    if x.size == 0: return np.linspace(-1, 1, nlev)
    vmax = np.nanpercentile(np.abs(x), 98)
    if not np.isfinite(vmax) or vmax == 0: vmax = np.nanmax(np.abs(x))
    if not np.isfinite(vmax) or vmax == 0: vmax = 1.0
    return np.linspace(-vmax, vmax, nlev)

def load_data(var_key):
    """加载并对齐 Full Clim 和 Low 25，统一处理单位和正负号"""
    datasets = ["MERRA2", "COUPLED", "NOCOUPL"]
    full_dict, low25_dict = {}, {}
    
    # 1. Load Full Climatology
    full_file = os.path.join(COMPARE_OUT, f"{var_key}_compare_climatology_1to100hPa.nc")
    with xr.open_dataset(full_file) as ds_full:
        da_full = ds_full["clim_octsep"].load()
        
    for dname in datasets:
        val = da_full.sel(dataset=dname).values
        # Full Clim 中的 MERRA2 O3 还是 mg/kg，需要转换
        if var_key == "O3" and dname == "MERRA2":
            val = val * MICRO_KGKG_TO_PPMV
        if var_key == "EPFLUX": val = -val
        full_dict[dname] = val

    # 2. Load Low 25 Climatology
    for dname in datasets:
        low_file = os.path.join(COMPARE_OUT, f"{dname}_{var_key}_LOW25_climatology_1to100hPa.nc")
        with xr.open_dataset(low_file) as ds_low:
            val = ds_low["clim_octsep"].values
            # Low 25 文件在生成时已经全部做过 ppmv 转换了，无需再乘系数
            if var_key == "EPFLUX": val = -val
            low25_dict[dname] = val
            
    plev = da_full["plev"].values
    x = da_full["time_index"].values
    return full_dict, low25_dict, plev, x

# ============================================================
# Main Plot Routine
# ============================================================
def plot_diff_low25_vs_clim(var_key):
    full_dict, low25_dict, plev, x = load_data(var_key)
    datasets = ["MERRA2", "COUPLED", "NOCOUPL"]
    
    # 计算 Diff (Low 25 - Full)
    diff_dict = {d: low25_dict[d] - full_dict[d] for d in datasets}
    
    # 获取统一的 Colorbar Levels
    all_diffs = np.concatenate([diff_dict[d].ravel() for d in datasets])
    levels_diff = robust_symmetric_levels(all_diffs, nlev=31)

    fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.8), sharex=True, sharey=True, constrained_layout=True)

    cf = None
    for i, dname in enumerate(datasets):
        ax = axes[i]
        z_diff = diff_dict[dname]
        z_clim = full_dict[dname] # 背景等值线用自身的 Climate

        # 填色图: Anomaly
        cf = ax.contourf(x, plev, z_diff, levels=levels_diff, cmap=PLOT_CFG_DIFF[var_key]["cmap"], extend="both", alpha=0.95)

        # 等值线: Full Climatology
        cs = ax.contour(x, plev, z_clim, levels=10, colors="k", linewidths=0.6, alpha=0.8)
        ax.clabel(cs, inline=True, fontsize=7, fmt="%.2g")
        if var_key in ["U", "EPFLUX"]:
            try: ax.contour(x, plev, z_clim, levels=[0], colors="k", linewidths=1.5)
            except Exception: pass

        ax.set_yscale("log")
        ax.invert_yaxis()
        ax.set_ylim(100, 1)
        ax.set_yticks(plev)
        ax.yaxis.set_major_formatter(ScalarFormatter())
        ax.set_xlim(x[0], x[-1])
        ax.set_xticks(XTICKS)
        ax.set_xticklabels(XTICK_LABELS)
        ax.grid(True, which="major", linestyle="--", linewidth=0.35, alpha=0.3)
        ax.set_title(f"{dname}\n(Contours: {dname} Climatology)")

    axes[0].set_ylabel("Pressure (hPa)")
    cbar = fig.colorbar(cf, ax=axes, pad=0.02, shrink=0.95)
    cbar.set_label(PLOT_CFG_DIFF[var_key]["cbar_label"])
    fig.suptitle(PLOT_CFG_DIFF[var_key]["title"], fontsize=14, y=1.05)

    out_png = os.path.join(PLOT_OUT, f"{var_key}_Diff_Low25_minus_Clim_OctSep.png")
    out_pdf = os.path.join(PLOT_OUT, f"{var_key}_Diff_Low25_minus_Clim_OctSep.pdf")
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.savefig(out_pdf, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_png}")

if __name__ == "__main__":
    print("=" * 60)
    print(" Plotting Low 25 Anomaly (Low25 - Climatology)")
    print("=" * 60)
    for var in ["U", "T", "O3", "EPFLUX"]:
        plot_diff_low25_vs_clim(var)

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

# ============================================================
# Paths
# ============================================================
ROOT = "/mnt/soclim0/public_data/weiji"

PATHS = {
    "BWCN": os.path.join(ROOT, "BWCN"),
    "B2000WCN": os.path.join(ROOT, "B2000WCN001002"),
    "NOCOUPL": os.path.join(ROOT, "B2000WCN_NOCOUPL001002"),
    "MERRA2": os.path.join(ROOT, "MERRA2_Processed"),
}

PREFIX_MAP = {
    "BWCN": "BWCN",
    "B2000WCN": "B2000WCN",
    "NOCOUPL": "B2000WCN_NOCOUPL",
    "MERRA2": "MERRA2",
}

OUT_ROOT = "/home/weiji/restart_exam/code/20260226B2000WCN_BWCN_MERRA2_COMPARE/RESULT"
PLOT_OUT = os.path.join(OUT_ROOT, "plots_raw_direct")
os.makedirs(PLOT_OUT, exist_ok=True)

XTICKS = [0, 31, 61, 92, 123, 151, 182, 212, 243, 273, 304, 335]
XTICK_LABELS = ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep"]

mpl.rc("xtick", direction="out", labelsize=9)
mpl.rc("ytick", direction="out", labelsize=9)
mpl.rc("axes", titlesize=11, labelsize=10)
mpl.rc("font", family="sans-serif")

PLOT_CFG_DIFF = {
    "U": {
        "cmap": "RdBu_r",
        "cbar_label": "Diff: U (m/s)",
        "title": "Anomaly (Low 25% - Climatology): Zonal Wind at 60°N",
    },
    "T": {
        "cmap": "RdBu_r",
        "cbar_label": "Diff: T (K)",
        "title": "Anomaly (Low 25% - Climatology): Polar-Cap Temp (60–90°N)",
    },
    "O3": {
        "cmap": "PiYG",
        "cbar_label": "Diff: O$_3$ (ppmv)",
        "title": "Anomaly (Low 25% - Climatology): Polar-Cap O$_3$ (60–90°N)",
    },
    "EPFLUX": {
        "cmap": "PuOr_r",
        "cbar_label": "Diff: Fz (native)",
        "title": "Anomaly (Low 25% - Climatology): EP Flux Fz",
    },
}

# ============================================================
# Weights
# ============================================================
# 这些请按你的真实有效样本数再确认
# full climatology 的 coupled 权重：应使用 full 的有效总年数
N_FULL_BWCN = 24
N_FULL_B2K = 209

# low25 climatology 的 coupled 权重：使用 low25 的真实年数
N_LOW25_BWCN = 5
N_LOW25_B2K = 51

# ============================================================
# Helpers
# ============================================================
def robust_symmetric_levels(values, nlev=31):
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.linspace(-1, 1, nlev)
    vmax = np.nanpercentile(np.abs(x), 98)
    if not np.isfinite(vmax) or vmax == 0:
        vmax = np.nanmax(np.abs(x))
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0
    return np.linspace(-vmax, vmax, nlev)


def process_spatial_reduction(da, var_key):
    """
    U      -> 60N 单点
    T / O3 -> 60-90N 余弦加权平均
    EPFLUX -> 保持 pressure x time
    """
    if "lat" not in da.dims:
        return da

    if "lon" in da.dims:
        da = da.mean(dim="lon")

    if var_key == "U":
        return da.sel(lat=60.0, method="nearest")

    if var_key in ["T", "O3"]:
        lat_min, lat_max = 60.0, 90.0
        lat_vals = da["lat"].values
        if lat_vals[0] > lat_vals[-1]:
            lat_min, lat_max = lat_max, lat_min
        da_band = da.sel(lat=slice(lat_min, lat_max))
        weights = np.cos(np.deg2rad(da_band["lat"]))
        return da_band.weighted(weights).mean(dim="lat")

    return da


def get_full_file_path(exp_name, var_key):
    prefix = PREFIX_MAP[exp_name]
    return os.path.join(PATHS[exp_name], f"{prefix}_{var_key}_climatology_1to100hPa.nc")


def get_low25_file_path(exp_name, var_key):
    prefix = PREFIX_MAP[exp_name]
    return os.path.join(PATHS[exp_name], f"{prefix}_{var_key}_LOW25_climatology_1to100hPa.nc")


def load_one_field(exp_name, var_key, low25=False):
    path = get_low25_file_path(exp_name, var_key) if low25 else get_full_file_path(exp_name, var_key)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")

    with xr.open_dataset(path) as ds:
        da = ds["clim_octsep"].load()

    da = process_spatial_reduction(da, var_key)

    if var_key == "EPFLUX":
        da = -da

    return da


def weighted_merge(da1, da2, w1, w2, out_name="merged"):
    if da1.shape != da2.shape:
        raise ValueError(f"Shape mismatch: {da1.shape} vs {da2.shape}")
    out = (da1 * w1 + da2 * w2) / (w1 + w2)
    out.name = out_name
    return out


def load_data(var_key):
    datasets = ["MERRA2", "COUPLED", "NOCOUPL"]

    # -------------------------
    # Full climatology
    # -------------------------
    full_merra = load_one_field("MERRA2", var_key, low25=False)
    full_bwcn = load_one_field("BWCN", var_key, low25=False)
    full_b2k = load_one_field("B2000WCN", var_key, low25=False)
    full_nocoupl = load_one_field("NOCOUPL", var_key, low25=False)

    full_coupled = weighted_merge(
        full_bwcn,
        full_b2k,
        N_FULL_BWCN,
        N_FULL_B2K,
        out_name="full_coupled"
    )

    # -------------------------
    # Low25 climatology
    # -------------------------
    low_merra = load_one_field("MERRA2", var_key, low25=True)
    low_bwcn = load_one_field("BWCN", var_key, low25=True)
    low_b2k = load_one_field("B2000WCN", var_key, low25=True)
    low_nocoupl = load_one_field("NOCOUPL", var_key, low25=True)

    low_coupled = weighted_merge(
        low_bwcn,
        low_b2k,
        N_LOW25_BWCN,
        N_LOW25_B2K,
        out_name="low25_coupled"
    )

    full_dict = {
        "MERRA2": full_merra.values,
        "COUPLED": full_coupled.values,
        "NOCOUPL": full_nocoupl.values,
    }

    low25_dict = {
        "MERRA2": low_merra.values,
        "COUPLED": low_coupled.values,
        "NOCOUPL": low_nocoupl.values,
    }

    plev = full_merra["plev"].values
    x = full_merra["time_index"].values

    # 额外打印一些诊断信息，方便核验
    print(f"\n[{var_key}] RAW DIRECT LOAD")
    print(f"  full  BWCN shape   : {full_bwcn.shape}")
    print(f"  full  B2000WCN shape: {full_b2k.shape}")
    print(f"  full  NOCOUPL shape : {full_nocoupl.shape}")
    print(f"  low25 BWCN shape   : {low_bwcn.shape}")
    print(f"  low25 B2000WCN shape: {low_b2k.shape}")
    print(f"  low25 NOCOUPL shape : {low_nocoupl.shape}")
    print(f"  full coupled weights : BWCN={N_FULL_BWCN}, B2000WCN={N_FULL_B2K}")
    print(f"  low25 coupled weights: BWCN={N_LOW25_BWCN}, B2000WCN={N_LOW25_B2K}")

    return full_dict, low25_dict, plev, x


# ============================================================
# Main Plot Routine
# ============================================================
def plot_diff_low25_vs_clim(var_key):
    full_dict, low25_dict, plev, x = load_data(var_key)
    datasets = ["MERRA2", "COUPLED", "NOCOUPL"]

    diff_dict = {d: low25_dict[d] - full_dict[d] for d in datasets}
    all_diffs = np.concatenate([diff_dict[d].ravel() for d in datasets])
    levels_diff = robust_symmetric_levels(all_diffs, nlev=31)

    fig, axes = plt.subplots(
        1, 3,
        figsize=(15.0, 5.0),
        sharex=True,
        sharey=True,
        constrained_layout=True
    )

    cf = None
    for i, dname in enumerate(datasets):
        ax = axes[i]
        z_diff = diff_dict[dname]
        z_clim = full_dict[dname]

        cf = ax.contourf(
            x, plev, z_diff,
            levels=levels_diff,
            cmap=PLOT_CFG_DIFF[var_key]["cmap"],
            extend="both",
            alpha=0.95
        )

        cs = ax.contour(
            x, plev, z_clim,
            levels=10,
            colors="k",
            linewidths=0.5,
            alpha=0.7
        )
        ax.clabel(cs, inline=True, fontsize=7, fmt="%.2g")

        if var_key in ["U", "EPFLUX"]:
            try:
                ax.contour(x, plev, z_clim, levels=[0], colors="k", linewidths=1.2)
            except Exception:
                pass

        ax.set_yscale("log")
        ax.invert_yaxis()
        ax.set_ylim(100, 1)
        ax.set_yticks(plev)
        ax.yaxis.set_major_formatter(ScalarFormatter())
        ax.set_xticks(XTICKS)
        ax.set_xticklabels(XTICK_LABELS)
        ax.grid(True, linestyle="--", linewidth=0.3, alpha=0.3)
        ax.set_title(f"{dname}", fontweight="bold")

    axes[0].set_ylabel("Pressure (hPa)")
    cbar = fig.colorbar(cf, ax=axes, pad=0.02, shrink=0.85)
    cbar.set_label(PLOT_CFG_DIFF[var_key]["cbar_label"])
    fig.suptitle(PLOT_CFG_DIFF[var_key]["title"], fontsize=15, y=1.05)

    out_file = os.path.join(PLOT_OUT, f"{var_key}_Anomaly_Low25_minus_Clim_OctSep_raw_direct.png")
    plt.savefig(out_file, dpi=300, bbox_inches="tight")
    plt.show(fig)
    print(f"  [SAVED] {out_file}")


if __name__ == "__main__":
    print("=" * 70)
    print(" Plotting Low25 - Climatology using ONLY raw per-experiment files")
    print("=" * 70)

    for var in ["U", "T", "O3", "EPFLUX"]:
        print(f"\nProcessing {var} ...")
        plot_diff_low25_vs_clim(var)

    print("\n✅ 所有异常图已生成完毕！")

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import glob
import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
import numba as nb
import warnings

warnings.filterwarnings("ignore")

# ============================================================
# Paths & Config
# ============================================================
ROOT = "/mnt/soclim0/public_data/weiji"
TXT_ROOT = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"

PATHS = {
    "BWCN": os.path.join(ROOT, "BWCN"),
    "B2000WCN": os.path.join(ROOT, "B2000WCN001002"),
    "NOCOUPL": os.path.join(ROOT, "B2000WCN_NOCOUPL001002"),
    "MERRA2": os.path.join(ROOT, "MERRA2_Processed"),
}

PREFIX_MAP = {
    "BWCN": "BWCN",
    "B2000WCN": "B2000WCN",
    "NOCOUPL": "B2000WCN_NOCOUPL",
    "MERRA2": "MERRA2",
}

OUT_ROOT = "/home/weiji/restart_exam/code/20260226B2000WCN_BWCN_MERRA2_COMPARE/RESULT"
PLOT_OUT = os.path.join(OUT_ROOT, "plots_extreme_single_year_panels")
os.makedirs(PLOT_OUT, exist_ok=True)

# 气候态权重
N_FULL_BWCN = 24
N_FULL_B2K = 209

TARGET_PLEV_HPA = np.array([100.0, 70.0, 50.0, 40.0, 30.0, 20.0, 10.0, 7.0, 5.0, 4.0, 3.0, 2.0, 1.0], dtype=np.float64)
TARGET_PLEV_PA = TARGET_PLEV_HPA * 100.0

MW_AIR = 28.9644
MW_O3 = 47.9982
MOL_MASS_RATIO = MW_AIR / MW_O3

VAR_SPECS = {
    "U": {"var_name": "U", "scale_model": 1.0, "scale_merra": 1.0, "cmap": "RdBu_r", "cbar": "Diff: U (m/s)", "title": "Anomaly (Most Extreme Year - Clim): Zonal Wind at 60°N"},
    "T": {"var_name": "T", "scale_model": 1.0, "scale_merra": 1.0, "cmap": "RdBu_r", "cbar": "Diff: T (K)", "title": "Anomaly (Most Extreme Year - Clim): Polar-Cap Temp (60–90°N)"},
    "O3": {"var_name": "O3", "scale_model": 1e6, "scale_merra": MOL_MASS_RATIO * 1e6, "cmap": "PiYG", "cbar": "Diff: O$_3$ (ppmv)", "title": "Anomaly (Most Extreme Year - Clim): Polar-Cap O$_3$ (60–90°N)"},
    "EPFLUX": {"var_name": "Fz", "scale_model": 1.0, "scale_merra": 1.0, "cmap": "PuOr_r", "cbar": "Diff: Fz (native)", "title": "Anomaly (Most Extreme Year - Clim): EP Flux Fz"},
}

XTICKS = [0, 31, 61, 92, 123, 151, 182, 212, 243, 273, 304, 335]
XTICK_LABELS = ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep"]
mpl.rc("xtick", direction="out", labelsize=9)
mpl.rc("ytick", direction="out", labelsize=9)
mpl.rc("axes", titlesize=11, labelsize=10)

# ============================================================
# Core Helpers
# ============================================================
def get_extreme_year_from_txt(exp):
    folder_map = {"MERRA2": "MERRA2", "BWCN": "BWCN", "NOCOUPL": "B2000WCN_NOCOUPL"}
    txt_path = os.path.join(TXT_ROOT, folder_map[exp], "low25pct_years_30_70hPa.txt")
    
    fallbacks = {"MERRA2": 2020, "BWCN": 8, "NOCOUPL": 95}
    
    if os.path.exists(txt_path):
        years = np.loadtxt(txt_path, dtype=int)
        extreme_y = int(years) if years.ndim == 0 else int(years[0])
        print(f"  [TXT] 成功读取 {exp} 极端年: {extreme_y} (来自TXT文件)")
        return extreme_y
    else:
        print(f"  [WARN] 找不到 {exp} 的TXT文件，使用默认极值年: {fallbacks[exp]}")
        return fallbacks[exp]

def physical_year_to_file_year(exp_name, y_phys):
    if exp_name == "NOCOUPL":
        if 1 <= y_phys <= 103: return y_phys
        elif 104 <= y_phys <= 204: return y_phys + 1
    return y_phys

def drop_feb29(time_vals):
    keep = []
    for i, tt in enumerate(time_vals):
        y, m, d = (tt.year, tt.month, tt.day) if hasattr(tt, "year") else [int(x) for x in np.datetime_as_string(tt, unit="D").split("-")]
        if not (m == 2 and d == 29): keep.append(i)
    return np.array(keep)

def extract_2d_values(da):
    """
    强制挤掉大小为1的无用维度（如平均后的lat），并对齐为 (time, plev)
    """
    da = da.squeeze() # 剔除类似 (365, 13, 1) 中的那个 1
    t_dim = [d for d in da.dims if "time" in d.lower() or da.sizes[d] == 365][0]
    p_dim = [d for d in da.dims if "lev" in d.lower() or da.sizes[d] != 365][0]
    return da.transpose(t_dim, p_dim).values

@nb.njit(parallel=True)
def fast_logp_interp_flat(p_flat, v_flat, target_p):
    n_profs = p_flat.shape[0]
    nlev = p_flat.shape[1]
    ntarg = len(target_p)
    out = np.empty((n_profs, ntarg), dtype=np.float32)
    ltp = np.log(target_p)
    for i in nb.prange(n_profs):
        p, v = p_flat[i], v_flat[i]
        valid = 0
        for k in range(nlev):
            if p[k] > 0 and not np.isnan(p[k]) and not np.isnan(v[k]): valid += 1
        if valid < 2:
            for t_idx in range(ntarg): out[i, t_idx] = np.nan
            continue
        p_valid, v_valid = np.empty(valid, dtype=np.float64), np.empty(valid, dtype=np.float64)
        idx = 0
        for k in range(nlev):
            if p[k] > 0 and not np.isnan(p[k]) and not np.isnan(v[k]):
                p_valid[idx], v_valid[idx] = p[k], v[k]
                idx += 1
        order = np.argsort(p_valid)
        p_sorted, v_sorted = p_valid[order], v_valid[order]
        lp = np.log(p_sorted)
        out_i = np.interp(ltp, lp, v_sorted)
        slope_top = (v_sorted[1] - v_sorted[0]) / (lp[1] - lp[0])
        slope_bottom = (v_sorted[-1] - v_sorted[-2]) / (lp[-1] - lp[-2])
        for t_idx in range(ntarg):
            if ltp[t_idx] < lp[0]: out[i, t_idx] = v_sorted[0] + slope_top * (ltp[t_idx] - lp[0])
            elif ltp[t_idx] > lp[-1]: out[i, t_idx] = v_sorted[-1] + slope_bottom * (ltp[t_idx] - lp[-1])
            else: out[i, t_idx] = out_i[t_idx]
    return out

# ============================================================
# Loading Logic
# ============================================================
def load_merra_single_year(var_key, target_year):
    spec = VAR_SPECS[var_key]
    
    if var_key == "EPFLUX":
        # MERRA2 EPFLUX 也采用单文件模式读取
        ep_dir = os.path.join(PATHS["MERRA2"], "EPflux_daily")
        files = glob.glob(os.path.join(ep_dir, "*.nc"))
        if not files:
            raise FileNotFoundError(f"找不到 MERRA2 EPFLUX 单文件，路径: {ep_dir}")
            
        ds = xr.open_dataset(files[0], decode_times=True)
        da = ds[spec["var_name"]]
        
        if "plev" not in da.dims: 
            da = da.rename({"level": "plev"} if "level" in da.dims else {"lev": "plev"})
        if np.nanmax(da.plev.values) > 2000: 
            da = da.assign_coords(plev=da.plev / 100.0)
            
        years, months, days = da["time"].dt.year.values, da["time"].dt.month.values, da["time"].dt.day.values
        keep_mask = ~((months == 2) & (days == 29))
        da = da.isel(time=keep_mask)
        
        v_prev = da.isel(time=(da["time"].dt.year.values[keep_mask] == target_year - 1))
        v_curr = da.isel(time=(da["time"].dt.year.values[keep_mask] == target_year))
        
        stitched = xr.concat([v_prev[273:], v_curr[:273]], dim="time") 
        stitched = stitched.interp(plev=TARGET_PLEV_HPA, kwargs={"fill_value": "extrapolate"})
        return -stitched * spec["scale_merra"]  # 统一翻转
        
    else:
        # U / T / O3 使用按年份读取
        subdir = var_key
        f_prev_list = glob.glob(os.path.join(PATHS["MERRA2"], subdir, f"*{target_year-1}*.nc"))
        f_curr_list = glob.glob(os.path.join(PATHS["MERRA2"], subdir, f"*{target_year}*.nc"))
        
        if not f_prev_list: raise FileNotFoundError(f"Missing MERRA2 {var_key} file for {target_year-1}")
        if not f_curr_list: raise FileNotFoundError(f"Missing MERRA2 {var_key} file for {target_year}")
        
        ds_prev = xr.open_dataset(f_prev_list[0], decode_times=True)
        da_prev = ds_prev[spec["var_name"]].isel(time=drop_feb29(ds_prev.time.values)[273:])
        
        ds_curr = xr.open_dataset(f_curr_list[0], decode_times=True)
        da_curr = ds_curr[spec["var_name"]].isel(time=drop_feb29(ds_curr.time.values)[:273])
        
        da_stitched = xr.concat([da_prev, da_curr], dim="time")
        
        if "plev" not in da_stitched.dims:
            da_stitched = da_stitched.rename({"lev": "plev"} if "lev" in da_stitched.dims else {"level": "plev"})
        if np.nanmax(da_stitched.plev.values) > 2000:
            da_stitched = da_stitched.assign_coords(plev=da_stitched.plev / 100.0)
        
        da_stitched = da_stitched.interp(plev=TARGET_PLEV_HPA, kwargs={"fill_value": "extrapolate"})
        return da_stitched * spec["scale_merra"]

def load_waccm_single_year(exp_name, var_key, target_year_phys):
    spec = VAR_SPECS[var_key]
    
    if var_key == "EPFLUX":
        ep_dir = os.path.join(PATHS[exp_name], "EPflux_daily")
        files = glob.glob(os.path.join(ep_dir, "*.nc"))
        if not files: raise FileNotFoundError(f"找不到 {exp_name} EPFLUX 单文件，路径: {ep_dir}")
        
        ds = xr.open_dataset(files[0], decode_times=True)
        da = ds[spec["var_name"]]
        if "plev" not in da.dims: da = da.rename({"level": "plev", "lev": "plev"}[True])
        if np.nanmax(da.plev.values) > 2000: da = da.assign_coords(plev=da.plev / 100.0)
        
        years, months, days = da["time"].dt.year.values, da["time"].dt.month.values, da["time"].dt.day.values
        keep_mask = ~((months == 2) & (days == 29))
        da = da.isel(time=keep_mask)
        
        v_prev = da.isel(time=(da["time"].dt.year.values[keep_mask] == target_year_phys - 1))
        v_curr = da.isel(time=(da["time"].dt.year.values[keep_mask] == target_year_phys))
        
        stitched = -xr.concat([v_prev[273:], v_curr[:273]], dim="time") 
        stitched = stitched.interp(plev=TARGET_PLEV_HPA, kwargs={"fill_value": "extrapolate"})
        return stitched
        
    else:
        def get_half(y_phys, part):
            y_file = physical_year_to_file_year(exp_name, y_phys)
            f_path_list = glob.glob(os.path.join(PATHS[exp_name], var_key, f"*{y_file:04d}*.nc"))
            if not f_path_list: raise FileNotFoundError(f"找不到文件 {exp_name} {var_key} {y_file}")
            
            ds = xr.open_dataset(f_path_list[0], decode_times=True)
            da, ps = ds[spec["var_name"]], ds["PS"]
            
            keep_idx = drop_feb29(ds.time.values)
            sel = np.arange(273, 365) if part == "prev" else np.arange(0, 273)
            
            da_kept = da.isel(time=keep_idx[sel]).values.astype(np.float32) * spec["scale_model"]
            ps_kept = ps.isel(time=keep_idx[sel]).values.astype(np.float32)
            
            hyam, hybm, p0 = ds["hyam"].values.astype(np.float32), ds["hybm"].values.astype(np.float32), np.float32(ds["P0"].values)
            nt, nlev, nlat, nlon = da_kept.shape
            p_native = hyam[None, :, None, None] * p0 + hybm[None, :, None, None] * ps_kept[:, None, :, :]
            
            var_flat = da_kept.transpose(0, 2, 3, 1).reshape(-1, nlev).astype(np.float64)
            p_flat = p_native.transpose(0, 2, 3, 1).reshape(-1, nlev).astype(np.float64)
            out_flat = fast_logp_interp_flat(p_flat, var_flat, TARGET_PLEV_PA)
            var_interp_4d = out_flat.reshape(nt, nlat, nlon, len(TARGET_PLEV_PA)).transpose(0, 3, 1, 2)
            
            lat_name = "lat" if "lat" in ds.coords else "latitude"
            lon_name = "lon" if "lon" in ds.coords else "longitude"
            coords = {"time": np.arange(len(sel)), "plev": TARGET_PLEV_HPA, "lat": ds[lat_name].values, "lon": ds[lon_name].values}
            return xr.DataArray(var_interp_4d, coords=coords, dims=["time", "plev", "lat", "lon"])
            
        return xr.concat([get_half(target_year_phys - 1, "prev"), get_half(target_year_phys, "curr")], dim="time")

def process_spatial_reduction(da, var_key):
    if "lat" not in da.dims: return da
    if "lon" in da.dims: da = da.mean(dim="lon")
    if var_key == "U": return da.sel(lat=60.0, method="nearest")
    if var_key in ["T", "O3"]:
        lat_min, lat_max = 60.0, 90.0
        lat_vals = da["lat"].values
        if lat_vals[0] > lat_vals[-1]: lat_min, lat_max = lat_max, lat_min
        da_band = da.sel(lat=slice(lat_min, lat_max))
        weights = np.cos(np.deg2rad(da_band["lat"]))
        return da_band.weighted(weights).mean(dim="lat")
    return da

def get_clim_da(exp_name, var_key):
    prefix = PREFIX_MAP[exp_name]
    f_path = os.path.join(PATHS[exp_name], f"{prefix}_{var_key}_climatology_1to100hPa.nc")
    da = process_spatial_reduction(xr.open_dataset(f_path)["clim_octsep"], var_key)
    return -da if var_key == "EPFLUX" else da

# ============================================================
# Main Plotting
# ============================================================
def plot_extreme_vs_clim(var_key):
    print(f"\n[{var_key}] 读取数据 & 气候态基准...")
    spec = VAR_SPECS[var_key]
    
    # 提取极端年份
    year_merra = get_extreme_year_from_txt("MERRA2")
    year_bwcn = get_extreme_year_from_txt("BWCN")  
    year_nocoupl = get_extreme_year_from_txt("NOCOUPL")
    
    # 加载极端单年并转化为严格的 (365, 13) Numpy Array
    da_merra_ext = extract_2d_values(process_spatial_reduction(load_merra_single_year(var_key, year_merra), var_key))
    da_coupled_ext = extract_2d_values(process_spatial_reduction(load_waccm_single_year("BWCN", var_key, year_bwcn), var_key))
    da_nocoupl_ext = extract_2d_values(process_spatial_reduction(load_waccm_single_year("NOCOUPL", var_key, year_nocoupl), var_key))
    
    # 提取气候态并合并 COUPLED
    clim_merra_da = get_clim_da("MERRA2", var_key)
    clim_bwcn_da = get_clim_da("BWCN", var_key)
    clim_b2k_da = get_clim_da("B2000WCN", var_key)
    clim_nocoupl_da = get_clim_da("NOCOUPL", var_key)
    
    # BWCN + B2000WCN 加权合并作为 COUPLED 基准气候态
    clim_coupled_da = (clim_bwcn_da * N_FULL_BWCN + clim_b2k_da * N_FULL_B2K) / (N_FULL_BWCN + N_FULL_B2K)
    
    # 转化为严格的 (365, 13) Numpy Array
    clim_merra = extract_2d_values(clim_merra_da)
    clim_coupled = extract_2d_values(clim_coupled_da)
    clim_nocoupl = extract_2d_values(clim_nocoupl_da)
    
    plev = TARGET_PLEV_HPA
    x = np.arange(365)
    
    # 计算差异
    diff_merra = da_merra_ext - clim_merra
    diff_coupled = da_coupled_ext - clim_coupled
    diff_nocoupl = da_nocoupl_ext - clim_nocoupl

    # 统一色标（使用所有数据计算，确保 2 面板和 3 面板色标完全一致）
    all_diffs = np.concatenate([diff_merra.ravel(), diff_coupled.ravel(), diff_nocoupl.ravel()])
    vmax = np.nanpercentile(np.abs(all_diffs), 98)
    levels_diff = np.linspace(-vmax, vmax, 31)

    # 定义数据集列表
    dataset_merra = (f"MERRA2 (Ext: {year_merra})", diff_merra, clim_merra)
    dataset_coupled = (f"Interactive O3 WACCM4 (Ext from BWCN: {year_bwcn:04d})", diff_coupled, clim_coupled)
    dataset_nocoupl = (f"Prescribe O3 WACCM4 (Ext: {year_nocoupl:04d})", diff_nocoupl, clim_nocoupl)

    datasets_3_panels = [dataset_merra, dataset_coupled, dataset_nocoupl]
    datasets_2_panels = [dataset_merra, dataset_coupled]

    # --- 内部绘图函数 ---
    def draw_and_save(datasets, panel_count, suffix):
        fig, axes = plt.subplots(1, panel_count, figsize=(5.0 * panel_count, 5.0), sharex=True, sharey=True, constrained_layout=True)
        if panel_count == 1: axes = [axes]

        cf = None
        for i, (title, z_diff, z_clim) in enumerate(datasets):
            ax = axes[i]
            cf = ax.contourf(x, plev, z_diff.T, levels=levels_diff, cmap=spec["cmap"], extend="both", alpha=0.95)
            cs = ax.contour(x, plev, z_clim.T, levels=10, colors="k", linewidths=0.5, alpha=0.7)
            ax.clabel(cs, inline=True, fontsize=7, fmt="%.2g")
            
            if var_key in ["U", "EPFLUX"]:
                try: ax.contour(x, plev, z_clim.T, levels=[0], colors="k", linewidths=1.2)
                except Exception: pass
                
            ax.set_yscale("log")
            ax.invert_yaxis()
            ax.set_ylim(100, 1)
            ax.set_yticks(TARGET_PLEV_HPA)
            ax.yaxis.set_major_formatter(ScalarFormatter())
            ax.set_xticks(XTICKS)
            ax.set_xticklabels(XTICK_LABELS)
            ax.grid(True, linestyle="--", linewidth=0.3, alpha=0.3)
            ax.set_title(title, fontweight="bold")

        axes[0].set_ylabel("Pressure (hPa)")
        cbar = fig.colorbar(cf, ax=axes, pad=0.02, shrink=0.85)
        cbar.set_label(spec["cbar"])
        fig.suptitle(spec["title"], fontsize=15, y=1.05)

        out_file = os.path.join(PLOT_OUT, f"{var_key}_ExtremeSingleYear_{suffix}_Anomaly.png")
        plt.savefig(out_file, dpi=300, bbox_inches="tight")
        plt.show(block=False)
        plt.show(fig)
        print(f"  [SAVED] {out_file}")

    # 分别生成 3面板 和 2面板
    draw_and_save(datasets_3_panels, 3, "3Panels")
    draw_and_save(datasets_2_panels, 2, "2Panels")


if __name__ == "__main__":
    print("=" * 80)
    print(" Plotting Most Extreme Year vs Climatology (Auto TXT Read & Both 2/3 Panels)")
    print("=" * 80)
    
    for var in ["U", "T", "O3", "EPFLUX"]:
        plot_extreme_vs_clim(var)
        
    print("\n✅ 所有的 2面板 和 3面板 极端年对比图已生成完毕！")

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
import glob
import gc
import numpy as np
import xarray as xr
import numba as nb

ROOT = "/mnt/soclim0/public_data/weiji"
TXT_ROOT = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"

PATHS = {
    "BWCN": os.path.join(ROOT, "BWCN"),
    "B2000WCN": os.path.join(ROOT, "B2000WCN001002"),
    "NOCOUPL": os.path.join(ROOT, "B2000WCN_NOCOUPL001002"),
}

PREFIX_MAP = {
    "BWCN": "BWCN",
    "B2000WCN": "B2000WCN",
    "NOCOUPL": "B2000WCN_NOCOUPL",
}

TXT_FOLDER_MAP = {
    "BWCN": "BWCN",
    "B2000WCN": "B2000WCN",
    "NOCOUPL": "B2000WCN_NOCOUPL",
}

DROP_YEARS_MODEL = {104}

TARGET_PLEV_HPA = np.array(
    [100.0, 70.0, 50.0, 40.0, 30.0, 20.0, 10.0, 7.0, 5.0, 4.0, 3.0, 2.0, 1.0],
    dtype=np.float64
)
TARGET_PLEV_PA = TARGET_PLEV_HPA * 100.0

EXP_NAME = "NOCOUPL"
TEST_TIME_INDEX = 150
TEST_PLEV_HPA = 5.0
TEST_LAT = 75.0
TEST_LON = 0.0
CHECK_POLAR_CAP = True

def parse_file_year(path):
    m_all = re.findall(r'(?<!\d)(\d{4})(?!\d)', os.path.basename(path))
    return int(m_all[-1]) if m_all else None

def detect_coord_name(ds, candidates):
    for c in candidates:
        if c in ds.coords or c in ds.dims:
            return c
    return None

def get_ymd_from_time_value(tt):
    if hasattr(tt, "year") and hasattr(tt, "month"):
        return int(tt.year), int(tt.month), int(tt.day)
    y, m, d = np.datetime_as_string(tt, unit="D").split("-")
    return int(y), int(m), int(d)

def drop_feb29_and_get_time_parts(time_vals):
    keep_idx, years, months, days = [], [], [], []
    for i, tt in enumerate(time_vals):
        y, m, d = get_ymd_from_time_value(tt)
        if not (m == 2 and d == 29):
            keep_idx.append(i)
            years.append(y)
            months.append(m)
            days.append(d)
    return np.array(keep_idx, dtype=int), np.array(years, dtype=int), np.array(months, dtype=int), np.array(days, dtype=int)

def load_low25_years(exp_name):
    txt_folder = TXT_FOLDER_MAP[exp_name]
    txt_path = os.path.join(TXT_ROOT, txt_folder, "low25pct_years_30_70hPa.txt")
    years = np.loadtxt(txt_path, dtype=int)
    if np.ndim(years) == 0:
        years = np.array([int(years)])
    return years.tolist(), txt_path

def get_low25_nc_path(exp_name):
    prefix = PREFIX_MAP[exp_name]
    return os.path.join(PATHS[exp_name], f"{prefix}_O3_LOW25_climatology_1to100hPa.nc")

def find_exact_model_year_file(exp_name, y):
    files = sorted(glob.glob(os.path.join(PATHS[exp_name], "O3", "*.nc")))
    matched = [f for f in files if parse_file_year(f) == y]
    if len(matched) != 1:
        print(f"[WARN] {exp_name} year={y} matched {len(matched)} files")
        return None
    return matched[0]

@nb.njit(parallel=True)
def fast_logp_interp_flat(p_flat, v_flat, target_p):
    n_profs = p_flat.shape[0]
    nlev = p_flat.shape[1]
    ntarg = len(target_p)
    out = np.empty((n_profs, ntarg), dtype=np.float32)
    ltp = np.log(target_p)

    for i in nb.prange(n_profs):
        p = p_flat[i]
        v = v_flat[i]

        valid = 0
        for k in range(nlev):
            if p[k] > 0 and not np.isnan(p[k]) and not np.isnan(v[k]):
                valid += 1

        if valid < 2:
            for t_idx in range(ntarg):
                out[i, t_idx] = np.nan
            continue

        p_valid = np.empty(valid, dtype=np.float64)
        v_valid = np.empty(valid, dtype=np.float64)

        idx = 0
        for k in range(nlev):
            if p[k] > 0 and not np.isnan(p[k]) and not np.isnan(v[k]):
                p_valid[idx] = p[k]
                v_valid[idx] = v[k]
                idx += 1

        order = np.argsort(p_valid)
        p_sorted = p_valid[order]
        v_sorted = v_valid[order]

        lp = np.log(p_sorted)
        out_i = np.interp(ltp, lp, v_sorted)

        slope_top = (v_sorted[1] - v_sorted[0]) / (lp[1] - lp[0])
        slope_bottom = (v_sorted[-1] - v_sorted[-2]) / (lp[-1] - lp[-2])

        for t_idx in range(ntarg):
            if ltp[t_idx] < lp[0]:
                out[i, t_idx] = v_sorted[0] + slope_top * (ltp[t_idx] - lp[0])
            elif ltp[t_idx] > lp[-1]:
                out[i, t_idx] = v_sorted[-1] + slope_bottom * (ltp[t_idx] - lp[-1])
            else:
                out[i, t_idx] = out_i[t_idx]

    return out

def load_one_year_interp(exp_name, y):
    if y in DROP_YEARS_MODEL:
        return None

    f = find_exact_model_year_file(exp_name, y)
    if f is None:
        return None

    ds = xr.open_dataset(f, decode_times=True)

    phys_year = parse_file_year(f)
    if exp_name in ["B2000WCN", "NOCOUPL"] and phys_year >= 105:
        old_times = ds.time.values
        new_times = [t.replace(year=t.year + 103) for t in old_times]
        ds = ds.assign_coords(time=new_times)

    lat_name = detect_coord_name(ds, ["lat", "latitude"])
    lon_name = detect_coord_name(ds, ["lon", "longitude"])

    da = ds["O3"].transpose("time", "lev", lat_name, lon_name)
    ps = ds["PS"].transpose("time", lat_name, lon_name)

    hyam = ds["hyam"].values.astype(np.float32)
    hybm = ds["hybm"].values.astype(np.float32)
    p0 = np.float32(ds["P0"].values)

    keep_idx, _, _, _ = drop_feb29_and_get_time_parts(ds["time"].values)
    if len(keep_idx) != 365:
        ds.close()
        return None

    da_kept = da.isel(time=keep_idx).values.astype(np.float32) * np.float32(1e6)
    ps_kept = ps.isel(time=keep_idx).values.astype(np.float32)

    nt, nlev, nlat, nlon = da_kept.shape
    p_native = hyam[None, :, None, None] * p0 + hybm[None, :, None, None] * ps_kept[:, None, :, :]

    var_flat = da_kept.transpose(0, 2, 3, 1).reshape(-1, nlev).astype(np.float64)
    p_flat = p_native.transpose(0, 2, 3, 1).reshape(-1, nlev).astype(np.float64)
    out_flat = fast_logp_interp_flat(p_flat, var_flat, TARGET_PLEV_PA)

    out_4d = out_flat.reshape(nt, nlat, nlon, len(TARGET_PLEV_PA)).transpose(0, 3, 1, 2)

    lat_vals = ds[lat_name].values
    lon_vals = ds[lon_name].values
    ds.close()
    gc.collect()

    return xr.DataArray(
        out_4d,
        dims=("time", "plev", "lat", "lon"),
        coords={
            "time": np.arange(365),
            "plev": TARGET_PLEV_HPA.astype(np.float32),
            "lat": lat_vals.astype(np.float32),
            "lon": lon_vals.astype(np.float32),
        },
        name="O3_interp_ppmv"
    )

def build_one_seasonal_year(exp_name, y):
    if y in DROP_YEARS_MODEL or (y - 1) in DROP_YEARS_MODEL:
        print(f"[SKIP] seasonal year {y}: bridge-year related")
        return None

    prev_da = load_one_year_interp(exp_name, y - 1)
    curr_da = load_one_year_interp(exp_name, y)

    if prev_da is None or curr_da is None:
        print(f"[SKIP] seasonal year {y}: prev or curr missing")
        return None

    prev_part = prev_da.isel(time=slice(273, None))
    curr_part = curr_da.isel(time=slice(0, 273))
    stitched = xr.concat([prev_part, curr_part], dim="time")

    if stitched.sizes["time"] != 365:
        print(f"[WARN] seasonal year {y} stitched length = {stitched.sizes['time']}")
        return None

    stitched = stitched.rename({"time": "time_index"})
    stitched = stitched.assign_coords(time_index=np.arange(365))
    return stitched

def polar_cap_reduce(da):
    if "lon" in da.dims:
        da = da.mean(dim="lon")

    lat_vals = da["lat"].values
    lat_min, lat_max = 60.0, 90.0
    if lat_vals[0] > lat_vals[-1]:
        lat_min, lat_max = lat_max, lat_min

    da_band = da.sel(lat=slice(lat_min, lat_max))
    weights = np.cos(np.deg2rad(da_band["lat"]))
    return da_band.weighted(weights).mean(dim="lat")

def main():
    years, txt_path = load_low25_years(EXP_NAME)
    print(f"[INFO] txt path = {txt_path}")
    print(f"[INFO] years = {years}")

    low25_nc = get_low25_nc_path(EXP_NAME)
    print(f"[INFO] low25 nc = {low25_nc}")

    with xr.open_dataset(low25_nc) as ds:
        clim_os = ds["clim_octsep"].load()

    file_point_val = clim_os.sel(
        plev=TEST_PLEV_HPA, lat=TEST_LAT, lon=TEST_LON, method="nearest"
    ).isel(time_index=TEST_TIME_INDEX).item()

    clim_polar = polar_cap_reduce(clim_os)
    file_polar_val = clim_polar.sel(
        plev=TEST_PLEV_HPA, method="nearest"
    ).isel(time_index=TEST_TIME_INDEX).item()

    point_vals = []
    polar_vals = []
    used_years = []

    for y in years:
        print(f"\n[BUILD] seasonal year {y}")
        da_season = build_one_seasonal_year(EXP_NAME, y)
        if da_season is None:
            continue

        used_years.append(y)

        v_point = da_season.sel(
            plev=TEST_PLEV_HPA, lat=TEST_LAT, lon=TEST_LON, method="nearest"
        ).isel(time_index=TEST_TIME_INDEX).item()
        point_vals.append(v_point)

        da_polar = polar_cap_reduce(da_season)
        v_polar = da_polar.sel(
            plev=TEST_PLEV_HPA, method="nearest"
        ).isel(time_index=TEST_TIME_INDEX).item()
        polar_vals.append(v_polar)

        print(f"  point = {v_point:.8f} ppmv")
        print(f"  polar = {v_polar:.8f} ppmv")

        del da_season
        gc.collect()

    raw_point_mean = np.nanmean(point_vals)
    raw_polar_mean = np.nanmean(polar_vals)

    print("\n" + "=" * 80)
    print("[POINT CHECK]")
    print(f"raw recomputed mean = {raw_point_mean:.10f}")
    print(f"file low25 value    = {file_point_val:.10f}")
    print(f"difference          = {raw_point_mean - file_point_val:.10e}")

    print("\n[POLAR-CAP CHECK]")
    print(f"raw recomputed mean = {raw_polar_mean:.10f}")
    print(f"file low25 value    = {file_polar_val:.10f}")
    print(f"difference          = {raw_polar_mean - file_polar_val:.10e}")

    print(f"\n[INFO] used years ({len(used_years)}) = {used_years}")

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
import glob
import gc
import numpy as np
import xarray as xr
import numba as nb

# ============================================================
# Config
# ============================================================
ROOT = "/mnt/soclim0/public_data/weiji"

PATHS = {
    "BWCN": os.path.join(ROOT, "BWCN"),
    "B2000WCN": os.path.join(ROOT, "B2000WCN001002"),
    "NOCOUPL": os.path.join(ROOT, "B2000WCN_NOCOUPL001002"),
}

PREFIX_MAP = {
    "BWCN": "BWCN",
    "B2000WCN": "B2000WCN",
    "NOCOUPL": "B2000WCN_NOCOUPL",
}

DROP_YEARS_MODEL = {104}

TARGET_PLEV_HPA = np.array(
    [100.0, 70.0, 50.0, 40.0, 30.0, 20.0, 10.0, 7.0, 5.0, 4.0, 3.0, 2.0, 1.0],
    dtype=np.float64
)
TARGET_PLEV_PA = TARGET_PLEV_HPA * 100.0

# ============================================================
# What to test
# ============================================================
EXP_NAME = "NOCOUPL"      # "BWCN", "B2000WCN", "NOCOUPL"
VAR_KEY = "O3"

TEST_TIME_INDEX = 150
TEST_PLEV_HPA = 70.0
TEST_LAT = 75.0
TEST_LON = 0.0
CHECK_POLAR_CAP = True

# ============================================================
# Helpers
# ============================================================
def parse_file_year(path):
    m_all = re.findall(r'(?<!\d)(\d{4})(?!\d)', os.path.basename(path))
    return int(m_all[-1]) if m_all else None

def detect_coord_name(ds, candidates):
    for c in candidates:
        if c in ds.coords or c in ds.dims:
            return c
    return None

def get_ymd_from_time_value(tt):
    if hasattr(tt, "year") and hasattr(tt, "month"):
        return int(tt.year), int(tt.month), int(tt.day)
    y, m, d = np.datetime_as_string(tt, unit="D").split("-")
    return int(y), int(m), int(d)

def drop_feb29_and_get_time_parts(time_vals):
    keep_idx, years, months, days = [], [], [], []
    for i, tt in enumerate(time_vals):
        y, m, d = get_ymd_from_time_value(tt)
        if not (m == 2 and d == 29):
            keep_idx.append(i)
            years.append(y)
            months.append(m)
            days.append(d)
    return (
        np.array(keep_idx, dtype=int),
        np.array(years, dtype=int),
        np.array(months, dtype=int),
        np.array(days, dtype=int),
    )

def get_full_nc_path(exp_name):
    prefix = PREFIX_MAP[exp_name]
    return os.path.join(PATHS[exp_name], f"{prefix}_O3_climatology_1to100hPa.nc")

def find_exact_model_year_file(exp_name, y):
    files = sorted(glob.glob(os.path.join(PATHS[exp_name], "O3", "*.nc")))
    matched = [f for f in files if parse_file_year(f) == y]
    if len(matched) != 1:
        print(f"[WARN] {exp_name} year={y} matched {len(matched)} files")
        return None
    return matched[0]

def list_available_model_years(exp_name):
    files = sorted(glob.glob(os.path.join(PATHS[exp_name], "O3", "*.nc")))
    years = []
    for f in files:
        y = parse_file_year(f)
        if y is not None and y not in DROP_YEARS_MODEL:
            years.append(y)
    return sorted(years)

@nb.njit(parallel=True)
def fast_logp_interp_flat(p_flat, v_flat, target_p):
    n_profs = p_flat.shape[0]
    nlev = p_flat.shape[1]
    ntarg = len(target_p)
    out = np.empty((n_profs, ntarg), dtype=np.float32)
    ltp = np.log(target_p)

    for i in nb.prange(n_profs):
        p = p_flat[i]
        v = v_flat[i]

        valid = 0
        for k in range(nlev):
            if p[k] > 0 and not np.isnan(p[k]) and not np.isnan(v[k]):
                valid += 1

        if valid < 2:
            for t_idx in range(ntarg):
                out[i, t_idx] = np.nan
            continue

        p_valid = np.empty(valid, dtype=np.float64)
        v_valid = np.empty(valid, dtype=np.float64)

        idx = 0
        for k in range(nlev):
            if p[k] > 0 and not np.isnan(p[k]) and not np.isnan(v[k]):
                p_valid[idx] = p[k]
                v_valid[idx] = v[k]
                idx += 1

        order = np.argsort(p_valid)
        p_sorted = p_valid[order]
        v_sorted = v_valid[order]

        lp = np.log(p_sorted)
        out_i = np.interp(ltp, lp, v_sorted)

        slope_top = (v_sorted[1] - v_sorted[0]) / (lp[1] - lp[0])
        slope_bottom = (v_sorted[-1] - v_sorted[-2]) / (lp[-1] - lp[-2])

        for t_idx in range(ntarg):
            if ltp[t_idx] < lp[0]:
                out[i, t_idx] = v_sorted[0] + slope_top * (ltp[t_idx] - lp[0])
            elif ltp[t_idx] > lp[-1]:
                out[i, t_idx] = v_sorted[-1] + slope_bottom * (ltp[t_idx] - lp[-1])
            else:
                out[i, t_idx] = out_i[t_idx]

    return out

def load_one_year_interp(exp_name, y):
    """
    把某个物理年文件读进来，插值到 TARGET_PLEV_HPA，输出:
    (365, plev, lat, lon)
    """
    if y in DROP_YEARS_MODEL:
        return None

    f = find_exact_model_year_file(exp_name, y)
    if f is None:
        return None

    ds = xr.open_dataset(f, decode_times=True)

    phys_year = parse_file_year(f)
    if exp_name in ["B2000WCN", "NOCOUPL"] and phys_year >= 105:
        old_times = ds.time.values
        new_times = [t.replace(year=t.year + 103) for t in old_times]
        ds = ds.assign_coords(time=new_times)

    lat_name = detect_coord_name(ds, ["lat", "latitude"])
    lon_name = detect_coord_name(ds, ["lon", "longitude"])

    da = ds["O3"].transpose("time", "lev", lat_name, lon_name)
    ps = ds["PS"].transpose("time", lat_name, lon_name)

    hyam = ds["hyam"].values.astype(np.float32)
    hybm = ds["hybm"].values.astype(np.float32)
    p0 = np.float32(ds["P0"].values)

    keep_idx, _, _, _ = drop_feb29_and_get_time_parts(ds["time"].values)
    if len(keep_idx) != 365:
        print(f"[WARN] {exp_name} year={y} kept days = {len(keep_idx)}, not 365")
        ds.close()
        return None

    da_kept = da.isel(time=keep_idx).values.astype(np.float32) * np.float32(1e6)  # ppmv
    ps_kept = ps.isel(time=keep_idx).values.astype(np.float32)

    nt, nlev, nlat, nlon = da_kept.shape
    p_native = hyam[None, :, None, None] * p0 + hybm[None, :, None, None] * ps_kept[:, None, :, :]

    var_flat = da_kept.transpose(0, 2, 3, 1).reshape(-1, nlev).astype(np.float64)
    p_flat = p_native.transpose(0, 2, 3, 1).reshape(-1, nlev).astype(np.float64)
    out_flat = fast_logp_interp_flat(p_flat, var_flat, TARGET_PLEV_PA)

    out_4d = out_flat.reshape(nt, nlat, nlon, len(TARGET_PLEV_PA)).transpose(0, 3, 1, 2)

    lat_vals = ds[lat_name].values
    lon_vals = ds[lon_name].values
    ds.close()
    gc.collect()

    return xr.DataArray(
        out_4d,
        dims=("dayofyear", "plev", "lat", "lon"),
        coords={
            "dayofyear": np.arange(1, 366),
            "plev": TARGET_PLEV_HPA.astype(np.float32),
            "lat": lat_vals.astype(np.float32),
            "lon": lon_vals.astype(np.float32),
        },
        name="O3_interp_ppmv"
    )

def build_full_climatology_from_raw(exp_name):
    years = list_available_model_years(exp_name)
    print(f"[INFO] available years ({len(years)}) = {years[:20]} ... {years[-20:]}")

    sum_da = None
    count = 0

    for y in years:
        print(f"[BUILD] full year {y}")
        da_year = load_one_year_interp(exp_name, y)
        if da_year is None:
            continue

        if sum_da is None:
            sum_da = xr.zeros_like(da_year, dtype=np.float64)

        sum_da = sum_da + da_year.astype(np.float64)
        count += 1

        del da_year
        gc.collect()

    if count == 0:
        raise RuntimeError("No valid years accumulated for full climatology.")

    full_jandec = (sum_da / count).astype(np.float32)

    # rotate Jan-Dec -> Oct-Sep
    full_octsep_data = np.concatenate(
        [full_jandec.values[273:, ...], full_jandec.values[:273, ...]],
        axis=0
    )

    full_octsep = xr.DataArray(
        full_octsep_data,
        dims=("time_index", "plev", "lat", "lon"),
        coords={
            "time_index": np.arange(365),
            "plev": full_jandec["plev"].values,
            "lat": full_jandec["lat"].values,
            "lon": full_jandec["lon"].values,
        },
        name="full_octsep_raw_recomputed"
    )

    return full_octsep, count

def polar_cap_reduce(da):
    if "lon" in da.dims:
        da = da.mean(dim="lon")

    lat_vals = da["lat"].values
    lat_min, lat_max = 60.0, 90.0
    if lat_vals[0] > lat_vals[-1]:
        lat_min, lat_max = lat_max, lat_min

    da_band = da.sel(lat=slice(lat_min, lat_max))
    weights = np.cos(np.deg2rad(da_band["lat"]))
    return da_band.weighted(weights).mean(dim="lat")

# ============================================================
# Main
# ============================================================
def main():
    print("=" * 80)
    print(f"LIGHTWEIGHT DEBUG for FULL O3 climatology | EXP={EXP_NAME}")
    print("=" * 80)

    full_nc = get_full_nc_path(EXP_NAME)
    print(f"[INFO] full nc = {full_nc}")

    with xr.open_dataset(full_nc) as ds:
        clim_os_file = ds["clim_octsep"].load()

    print(f"[INFO] full clim dims = {clim_os_file.dims}, shape = {clim_os_file.shape}")

    # --------------------------
    # file value
    # --------------------------
    file_point_val = clim_os_file.sel(
        plev=TEST_PLEV_HPA,
        lat=TEST_LAT,
        lon=TEST_LON,
        method="nearest"
    ).isel(time_index=TEST_TIME_INDEX).item()

    clim_polar_file = polar_cap_reduce(clim_os_file)
    file_polar_val = clim_polar_file.sel(
        plev=TEST_PLEV_HPA,
        method="nearest"
    ).isel(time_index=TEST_TIME_INDEX).item()

    # --------------------------
    # raw recompute full climatology
    # --------------------------
    clim_os_raw, used_count = build_full_climatology_from_raw(EXP_NAME)

    raw_point_val = clim_os_raw.sel(
        plev=TEST_PLEV_HPA,
        lat=TEST_LAT,
        lon=TEST_LON,
        method="nearest"
    ).isel(time_index=TEST_TIME_INDEX).item()

    clim_polar_raw = polar_cap_reduce(clim_os_raw)
    raw_polar_val = clim_polar_raw.sel(
        plev=TEST_PLEV_HPA,
        method="nearest"
    ).isel(time_index=TEST_TIME_INDEX).item()

    print("\n" + "=" * 80)
    print("[POINT CHECK]")
    print(f"requested plev       = {TEST_PLEV_HPA} hPa")
    print(f"requested time_index = {TEST_TIME_INDEX}")
    print(f"requested lat/lon    = {TEST_LAT}, {TEST_LON}")
    print(f"raw recomputed value = {raw_point_val:.10f} ppmv")
    print(f"file full value      = {file_point_val:.10f} ppmv")
    print(f"difference           = {raw_point_val - file_point_val:.10e} ppmv")

    print("\n[POLAR-CAP CHECK]")
    print(f"requested plev       = {TEST_PLEV_HPA} hPa")
    print(f"requested time_index = {TEST_TIME_INDEX}")
    print(f"raw recomputed value = {raw_polar_val:.10f} ppmv")
    print(f"file full value      = {file_polar_val:.10f} ppmv")
    print(f"difference           = {raw_polar_val - file_polar_val:.10e} ppmv")

    print(f"\n[INFO] used full years count = {used_count}")
    print("\nDone.")

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
import gc
import glob
import warnings
import numpy as np
import xarray as xr
import numba as nb
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

warnings.filterwarnings("ignore")

# ============================================================
# Paths & Config
# ============================================================
ROOT = "/mnt/soclim0/public_data/weiji"

PATHS = {
    "BWCN": os.path.join(ROOT, "BWCN"),
    "B2000WCN": os.path.join(ROOT, "B2000WCN001002"),
    "NOCOUPL": os.path.join(ROOT, "B2000WCN_NOCOUPL001002"),
    "MERRA2": os.path.join(ROOT, "MERRA2_Processed"),
}

PREFIX_MAP = {
    "BWCN": "BWCN",
    "B2000WCN": "B2000WCN",
    "NOCOUPL": "B2000WCN_NOCOUPL",
    "MERRA2": "MERRA2",
}

OUT_ROOT = "/home/weiji/restart_exam/code/20260226B2000WCN_BWCN_MERRA2_COMPARE/RESULT"
DEBUG_OUT = os.path.join(OUT_ROOT, "debug_bwcn0008_single_year")
NC_OUT = os.path.join(DEBUG_OUT, "data")
PLOT_OUT = os.path.join(DEBUG_OUT, "plots")
os.makedirs(NC_OUT, exist_ok=True)
os.makedirs(PLOT_OUT, exist_ok=True)

# ------------------------------
# Debug target: only BWCN year 0008
# Seasonal year = Oct(0007) ~ Sep(0008)
# ------------------------------
DEBUG_EXP = "BWCN"
DEBUG_YEAR = 8

# ------------------------------
# Full coupled weights
# 请按你最终确认值修改
# ------------------------------
N_FULL_BWCN = 24
N_FULL_B2K = 209

# ------------------------------
# Pressure grid
# ------------------------------
TARGET_PLEV_HPA = np.array(
    [100.0, 70.0, 50.0, 40.0, 30.0, 20.0, 10.0, 7.0, 5.0, 4.0, 3.0, 2.0, 1.0],
    dtype=np.float64
)
TARGET_PLEV_PA = TARGET_PLEV_HPA * 100.0

DROP_YEARS_MODEL = {104}

MW_AIR = 28.9644
MW_O3 = 47.9982
MOL_MASS_RATIO = MW_AIR / MW_O3

VAR_SPECS = {
    "U": {
        "var_name": "U",
        "scale_factor_model": 1.0,
        "plot_unit": "m/s",
        "long_name": "Zonal wind",
    },
    "T": {
        "var_name": "T",
        "scale_factor_model": 1.0,
        "plot_unit": "K",
        "long_name": "Temperature",
    },
    "O3": {
        "var_name": "O3",
        "scale_factor_model": 1e6,   # mol/mol -> ppmv
        "plot_unit": "ppmv",
        "long_name": "Ozone mixing ratio",
    },
    "EPFLUX": {
        "var_name": "Fz",
        "scale_factor_model": 1.0,
        "plot_unit": "native",
        "long_name": "Vertical EP flux (Fz)",
    },
}

XTICKS = [0, 31, 61, 92, 123, 151, 182, 212, 243, 273, 304, 335]
XTICK_LABELS = ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep"]

mpl.rc("xtick", direction="out", labelsize=9)
mpl.rc("ytick", direction="out", labelsize=9)
mpl.rc("axes", titlesize=11, labelsize=10)
mpl.rc("font", family="sans-serif")

PLOT_CFG_DIFF = {
    "U": {
        "cmap": "RdBu_r",
        "cbar_label": "Diff: U (m/s)",
        "title": "DEBUG COUPLED anomaly: BWCN seasonal year 0008 - COUPLED full climatology | U at 60°N",
    },
    "T": {
        "cmap": "RdBu_r",
        "cbar_label": "Diff: T (K)",
        "title": "DEBUG COUPLED anomaly: BWCN seasonal year 0008 - COUPLED full climatology | Polar-Cap T (60–90°N)",
    },
    "O3": {
        "cmap": "PiYG",
        "cbar_label": "Diff: O$_3$ (ppmv)",
        "title": "DEBUG COUPLED anomaly: BWCN seasonal year 0008 - COUPLED full climatology | Polar-Cap O$_3$ (60–90°N)",
    },
    "EPFLUX": {
        "cmap": "PuOr_r",
        "cbar_label": "Diff: Fz (native)",
        "title": "DEBUG COUPLED anomaly: BWCN seasonal year 0008 - COUPLED full climatology | EP Flux Fz",
    },
}

# ============================================================
# Numba interp
# ============================================================
@nb.njit(parallel=True)
def fast_logp_interp_flat(p_flat, v_flat, target_p):
    n_profs = p_flat.shape[0]
    nlev = p_flat.shape[1]
    ntarg = len(target_p)
    out = np.empty((n_profs, ntarg), dtype=np.float32)
    ltp = np.log(target_p)

    for i in nb.prange(n_profs):
        p = p_flat[i]
        v = v_flat[i]

        valid = 0
        for k in range(nlev):
            if p[k] > 0 and not np.isnan(p[k]) and not np.isnan(v[k]):
                valid += 1

        if valid < 2:
            for t_idx in range(ntarg):
                out[i, t_idx] = np.nan
            continue

        p_valid = np.empty(valid, dtype=np.float64)
        v_valid = np.empty(valid, dtype=np.float64)

        idx = 0
        for k in range(nlev):
            if p[k] > 0 and not np.isnan(p[k]) and not np.isnan(v[k]):
                p_valid[idx] = p[k]
                v_valid[idx] = v[k]
                idx += 1

        order = np.argsort(p_valid)
        p_sorted = p_valid[order]
        v_sorted = v_valid[order]

        lp = np.log(p_sorted)
        out_i = np.interp(ltp, lp, v_sorted)

        slope_top = (v_sorted[1] - v_sorted[0]) / (lp[1] - lp[0])
        slope_bottom = (v_sorted[-1] - v_sorted[-2]) / (lp[-1] - lp[-2])

        for t_idx in range(ntarg):
            if ltp[t_idx] < lp[0]:
                out[i, t_idx] = v_sorted[0] + slope_top * (ltp[t_idx] - lp[0])
            elif ltp[t_idx] > lp[-1]:
                out[i, t_idx] = v_sorted[-1] + slope_bottom * (ltp[t_idx] - lp[-1])
            else:
                out[i, t_idx] = out_i[t_idx]

    return out

# ============================================================
# Helpers
# ============================================================
def parse_file_year(path):
    m_all = re.findall(r'(?<!\d)(\d{4})(?!\d)', os.path.basename(path))
    return int(m_all[-1]) if m_all else None

def detect_coord_name(ds, candidates):
    for c in candidates:
        if c in ds.coords or c in ds.dims:
            return c
    return None

def detect_main_var(ds, preferred=None):
    if preferred is not None and preferred in ds.data_vars:
        return preferred
    for v in ds.data_vars:
        if "time" in ds[v].dims:
            return v
    raise ValueError("Cannot detect main variable.")

def get_ymd_from_time_value(tt):
    if hasattr(tt, "year") and hasattr(tt, "month"):
        return int(tt.year), int(tt.month), int(tt.day)
    y, m, d = np.datetime_as_string(tt, unit="D").split("-")
    return int(y), int(m), int(d)

def drop_feb29_and_get_time_parts(time_vals):
    keep_idx, years, months, days = [], [], [], []
    for i, tt in enumerate(time_vals):
        y, m, d = get_ymd_from_time_value(tt)
        if not (m == 2 and d == 29):
            keep_idx.append(i)
            years.append(y)
            months.append(m)
            days.append(d)
    return (
        np.array(keep_idx, dtype=int),
        np.array(years, dtype=int),
        np.array(months, dtype=int),
        np.array(days, dtype=int),
    )

def get_full_nc_path(exp_name, var_key):
    prefix = PREFIX_MAP[exp_name]
    return os.path.join(PATHS[exp_name], f"{prefix}_{var_key}_climatology_1to100hPa.nc")

def get_debug_low25_nc_path(var_key):
    return os.path.join(NC_OUT, f"COUPLED_DEBUG_BWCN0008_{var_key}_LOW25_climatology_1to100hPa.nc")

def weighted_merge(da1, da2, w1, w2, out_name="merged"):
    if da1.shape != da2.shape:
        raise ValueError(f"Shape mismatch: {da1.shape} vs {da2.shape}")
    out = (da1 * w1 + da2 * w2) / (w1 + w2)
    out.name = out_name
    return out

def robust_symmetric_levels(values, nlev=31):
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.linspace(-1, 1, nlev)
    vmax = np.nanpercentile(np.abs(x), 98)
    if not np.isfinite(vmax) or vmax == 0:
        vmax = np.nanmax(np.abs(x))
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0
    return np.linspace(-vmax, vmax, nlev)

def process_spatial_reduction(da, var_key):
    """
    U      -> 60N
    T/O3   -> 60-90N cos(lat)-weighted mean
    EPFLUX -> keep as plev x time
    """
    if "lat" not in da.dims:
        return da

    if "lon" in da.dims:
        da = da.mean(dim="lon")

    if var_key == "U":
        return da.sel(lat=60.0, method="nearest")

    if var_key in ["T", "O3"]:
        lat_min, lat_max = 60.0, 90.0
        lat_vals = da["lat"].values
        if lat_vals[0] > lat_vals[-1]:
            lat_min, lat_max = lat_max, lat_min
        da_band = da.sel(lat=slice(lat_min, lat_max))
        weights = np.cos(np.deg2rad(da_band["lat"]))
        return da_band.weighted(weights).mean(dim="lat")

    return da

# ============================================================
# Hybrid vars (U/T/O3): build BWCN year 0008 seasonal-year file
# ============================================================
def find_exact_model_year_file(exp_name, var_key, y):
    files = sorted(glob.glob(os.path.join(PATHS[exp_name], var_key, "*.nc")))
    matched = [f for f in files if parse_file_year(f) == y]
    if len(matched) != 1:
        raise FileNotFoundError(f"{exp_name} {var_key} year={y} matched {len(matched)} files")
    return matched[0]

def load_model_year_interp(exp_name, var_key, y):
    if y in DROP_YEARS_MODEL:
        raise ValueError(f"Bridge year {y} should not be used.")

    spec = VAR_SPECS[var_key]
    f = find_exact_model_year_file(exp_name, var_key, y)

    ds = xr.open_dataset(f, decode_times=True)

    phys_year = parse_file_year(f)
    if exp_name in ["B2000WCN", "NOCOUPL"] and phys_year >= 105:
        old_times = ds.time.values
        new_times = [t.replace(year=t.year + 103) for t in old_times]
        ds = ds.assign_coords(time=new_times)

    lat_name = detect_coord_name(ds, ["lat", "latitude"])
    lon_name = detect_coord_name(ds, ["lon", "longitude"])

    da = ds[spec["var_name"]].transpose("time", "lev", lat_name, lon_name)
    ps = ds["PS"].transpose("time", lat_name, lon_name)

    hyam = ds["hyam"].values.astype(np.float32)
    hybm = ds["hybm"].values.astype(np.float32)
    p0 = np.float32(ds["P0"].values)

    keep_idx, _, _, _ = drop_feb29_and_get_time_parts(ds["time"].values)
    if len(keep_idx) != 365:
        ds.close()
        raise ValueError(f"{exp_name} {var_key} year={y}: kept days={len(keep_idx)}, not 365")

    da_kept = da.isel(time=keep_idx).values.astype(np.float32) * np.float32(spec["scale_factor_model"])
    ps_kept = ps.isel(time=keep_idx).values.astype(np.float32)

    nt, nlev, nlat, nlon = da_kept.shape
    p_native = hyam[None, :, None, None] * p0 + hybm[None, :, None, None] * ps_kept[:, None, :, :]

    var_flat = da_kept.transpose(0, 2, 3, 1).reshape(-1, nlev).astype(np.float64)
    p_flat = p_native.transpose(0, 2, 3, 1).reshape(-1, nlev).astype(np.float64)
    out_flat = fast_logp_interp_flat(p_flat, var_flat, TARGET_PLEV_PA)

    out_4d = out_flat.reshape(nt, nlat, nlon, len(TARGET_PLEV_PA)).transpose(0, 3, 1, 2)

    lat_vals = ds[lat_name].values.astype(np.float32)
    lon_vals = ds[lon_name].values.astype(np.float32)
    ds.close()
    gc.collect()

    return xr.DataArray(
        out_4d,
        dims=("time", "plev", "lat", "lon"),
        coords={
            "time": np.arange(365),
            "plev": TARGET_PLEV_HPA.astype(np.float32),
            "lat": lat_vals,
            "lon": lon_vals,
        },
        name=f"{var_key}_debug_bwcn0008"
    )

def build_debug_low25_hybrid_single_year(var_key):
    prev_y = DEBUG_YEAR - 1
    curr_y = DEBUG_YEAR

    print(f"[DEBUG LOW25] {var_key}: building seasonal year Oct({prev_y:04d})-Sep({curr_y:04d}) from BWCN")

    prev_da = load_model_year_interp(DEBUG_EXP, var_key, prev_y)
    curr_da = load_model_year_interp(DEBUG_EXP, var_key, curr_y)

    prev_part = prev_da.isel(time=slice(273, None))   # Oct-Dec
    curr_part = curr_da.isel(time=slice(0, 273))      # Jan-Sep
    stitched = xr.concat([prev_part, curr_part], dim="time")

    if stitched.sizes["time"] != 365:
        raise ValueError(f"{var_key}: stitched seasonal year length = {stitched.sizes['time']}")

    stitched = stitched.rename({"time": "time_index"})
    stitched = stitched.assign_coords(time_index=np.arange(365))

    # save as debug low25 nc
    clim_os = stitched.transpose("plev", "lat", "lon", "time_index").astype(np.float32)
    # for convenience also provide Jan-Dec version by inverse rotate
    arr_os = clim_os.values
    arr_jd = np.concatenate([arr_os[..., 92:], arr_os[..., :92]], axis=-1).astype(np.float32)

    clim_jd = xr.DataArray(
        arr_jd,
        dims=("plev", "lat", "lon", "dayofyear"),
        coords={
            "plev": clim_os["plev"].values,
            "lat": clim_os["lat"].values,
            "lon": clim_os["lon"].values,
            "dayofyear": np.arange(1, 366),
        },
        name="clim_jandec"
    )
    clim_os_da = xr.DataArray(
        arr_os,
        dims=("plev", "lat", "lon", "time_index"),
        coords={
            "plev": clim_os["plev"].values,
            "lat": clim_os["lat"].values,
            "lon": clim_os["lon"].values,
            "time_index": np.arange(365),
        },
        name="clim_octsep"
    )

    ds_out = xr.Dataset({"clim_jandec": clim_jd, "clim_octsep": clim_os_da})
    ds_out.attrs.update({
        "experiment": "COUPLED_DEBUG_BWCN0008",
        "description": f"Debug single-season LOW25 climatology built ONLY from BWCN seasonal year Oct({prev_y:04d})-Sep({curr_y:04d})",
        "variable": var_key,
        "long_name": VAR_SPECS[var_key]["long_name"],
        "plot_unit": VAR_SPECS[var_key]["plot_unit"],
        "source": "BWCN only",
    })

    out_nc = get_debug_low25_nc_path(var_key)
    ds_out.to_netcdf(out_nc)
    print(f"  [SAVED DEBUG LOW25 NC] {out_nc}")

    return clim_os_da

# ============================================================
# EPFLUX: build BWCN year 0008 seasonal-year file
# ============================================================
def find_epflux_file(exp_name):
    files = sorted(glob.glob(os.path.join(PATHS[exp_name], "EPflux_daily", "*.nc")))
    if not files:
        raise FileNotFoundError(f"No EPFLUX file for {exp_name}")
    return files[0]

def build_debug_low25_epflux_single_year():
    prev_y = DEBUG_YEAR - 1
    curr_y = DEBUG_YEAR

    print(f"[DEBUG LOW25] EPFLUX: building seasonal year Oct({prev_y:04d})-Sep({curr_y:04d}) from BWCN")

    path = find_epflux_file(DEBUG_EXP)
    ds = xr.open_dataset(path, decode_times=True)

    vname = detect_main_var(ds, preferred=VAR_SPECS["EPFLUX"]["var_name"])
    da = ds[vname]

    p_name = detect_coord_name(ds, ["plev", "lev", "level"])
    da = da.rename({p_name: "plev"}).transpose("time", "plev")

    src = da["plev"].values.astype(np.float64)
    if np.nanmax(src) > 2000:
        src /= 100.0
        da = da.assign_coords(plev=("plev", src))

    da = da.isel(plev=((src >= 1.0) & (src <= 100.0)))
    native_plev_hpa = da["plev"].values.astype(np.float64)

    keep_idx, years, _, _ = drop_feb29_and_get_time_parts(da["time"].values)
    da = da.isel(time=keep_idx)
    vals = da.values.astype(np.float32)

    mask_prev = (years == prev_y)
    mask_curr = (years == curr_y)

    if mask_prev.sum() != 365 or mask_curr.sum() != 365:
        ds.close()
        raise ValueError(
            f"EPFLUX seasonal-year pieces invalid: prev={mask_prev.sum()} days, curr={mask_curr.sum()} days"
        )

    prev_part = vals[mask_prev][273:]   # Oct-Dec
    curr_part = vals[mask_curr][:273]   # Jan-Sep
    stitched_native = np.concatenate([prev_part, curr_part], axis=0)

    if stitched_native.shape[0] != 365:
        ds.close()
        raise ValueError(f"EPFLUX stitched length = {stitched_native.shape[0]}")

    ds.close()
    gc.collect()

    # native plev -> target plev
    stitched_target = np.full((365, len(TARGET_PLEV_HPA)), np.nan, dtype=np.float32)
    lp_native = np.log(native_plev_hpa * 100.0)
    lp_target = np.log(TARGET_PLEV_PA)

    for d in range(365):
        stitched_target[d, :] = np.interp(lp_target, lp_native, stitched_native[d, :]).astype(np.float32)

    clim_os_da = xr.DataArray(
        stitched_target.T,
        dims=("plev", "time_index"),
        coords={
            "plev": TARGET_PLEV_HPA.astype(np.float32),
            "time_index": np.arange(365),
        },
        name="clim_octsep"
    )

    arr_os = clim_os_da.values
    arr_jd = np.concatenate([arr_os[..., 92:], arr_os[..., :92]], axis=-1).astype(np.float32)

    clim_jd = xr.DataArray(
        arr_jd,
        dims=("plev", "dayofyear"),
        coords={
            "plev": clim_os_da["plev"].values,
            "dayofyear": np.arange(1, 366),
        },
        name="clim_jandec"
    )

    ds_out = xr.Dataset({"clim_jandec": clim_jd, "clim_octsep": clim_os_da})
    ds_out.attrs.update({
        "experiment": "COUPLED_DEBUG_BWCN0008",
        "description": f"Debug single-season LOW25 climatology built ONLY from BWCN seasonal year Oct({prev_y:04d})-Sep({curr_y:04d})",
        "variable": "EPFLUX",
        "long_name": VAR_SPECS["EPFLUX"]["long_name"],
        "plot_unit": VAR_SPECS["EPFLUX"]["plot_unit"],
        "source": "BWCN only",
    })

    out_nc = get_debug_low25_nc_path("EPFLUX")
    ds_out.to_netcdf(out_nc)
    print(f"  [SAVED DEBUG LOW25 NC] {out_nc}")

    return clim_os_da

# ============================================================
# Full coupled from raw full climatology files (not compare)
# ============================================================
def load_one_full_field(exp_name, var_key):
    path = get_full_nc_path(exp_name, var_key)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing full climatology file: {path}")

    with xr.open_dataset(path) as ds:
        da = ds["clim_octsep"].load()

    if var_key == "EPFLUX":
        # unify sign convention with your plotting chain
        da = -da

    return da

def load_debug_low25_field(var_key):
    path = get_debug_low25_nc_path(var_key)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing debug low25 file: {path}")

    with xr.open_dataset(path) as ds:
        da = ds["clim_octsep"].load()

    if var_key == "EPFLUX":
        da = -da

    return da

def build_coupled_full(var_key):
    full_bwcn = load_one_full_field("BWCN", var_key)
    full_b2k = load_one_full_field("B2000WCN", var_key)
    full_coupled = weighted_merge(
        full_bwcn, full_b2k,
        N_FULL_BWCN, N_FULL_B2K,
        out_name=f"COUPLED_full_{var_key}"
    )
    return full_coupled

# ============================================================
# Plot only COUPLED
# ============================================================
def plot_coupled_debug_anomaly(var_key):
    full_coupled = build_coupled_full(var_key)
    low25_debug = load_debug_low25_field(var_key)

    # same spatial reduction on both sides
    full_plot = process_spatial_reduction(full_coupled, var_key)
    low_plot = process_spatial_reduction(low25_debug, var_key)

    z_diff = (low_plot - full_plot).values
    z_clim = full_plot.values

    plev = full_plot["plev"].values
    x = full_plot["time_index"].values

    levels_diff = robust_symmetric_levels(z_diff.ravel(), nlev=31)

    fig, ax = plt.subplots(figsize=(6.2, 5.0), constrained_layout=True)

    cf = ax.contourf(
        x, plev, z_diff,
        levels=levels_diff,
        cmap=PLOT_CFG_DIFF[var_key]["cmap"],
        extend="both",
        alpha=0.95
    )

    cs = ax.contour(
        x, plev, z_clim,
        levels=10,
        colors="k",
        linewidths=0.5,
        alpha=0.7
    )
    ax.clabel(cs, inline=True, fontsize=7, fmt="%.2g")

    if var_key in ["U", "EPFLUX"]:
        try:
            ax.contour(x, plev, z_clim, levels=[0], colors="k", linewidths=1.2)
        except Exception:
            pass

    ax.set_yscale("log")
    ax.invert_yaxis()
    ax.set_ylim(100, 1)
    ax.set_yticks(plev)
    ax.yaxis.set_major_formatter(ScalarFormatter())
    ax.set_xticks(XTICKS)
    ax.set_xticklabels(XTICK_LABELS)
    ax.grid(True, linestyle="--", linewidth=0.3, alpha=0.3)
    ax.set_ylabel("Pressure (hPa)")
    ax.set_title("COUPLED", fontweight="bold")

    cbar = fig.colorbar(cf, ax=ax, pad=0.02, shrink=0.88)
    cbar.set_label(PLOT_CFG_DIFF[var_key]["cbar_label"])
    fig.suptitle(PLOT_CFG_DIFF[var_key]["title"], fontsize=13, y=1.03)

    out_png = os.path.join(PLOT_OUT, f"COUPLED_DEBUG_BWCN0008_{var_key}_anomaly_vs_full.png")
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.show(fig)
    print(f"  [SAVED PLOT] {out_png}")

# ============================================================
# Main
# ============================================================
def main():
    print("=" * 90)
    print("DEBUG PROGRAM: COUPLED built ONLY from BWCN seasonal year 0008 for LOW25")
    print("=" * 90)
    print(f"Debug source seasonal year: Oct({DEBUG_YEAR-1:04d}) - Sep({DEBUG_YEAR:04d})")
    print(f"Full coupled weights: BWCN={N_FULL_BWCN}, B2000WCN={N_FULL_B2K}")
    print(f"Output dir: {DEBUG_OUT}")

    # 1) build debug LOW25 nc from BWCN year 0008 only
    for var_key in ["U", "T", "O3"]:
        print("\n" + "-" * 70)
        print(f"[STEP 1] Build debug LOW25 nc for {var_key}")
        print("-" * 70)
        build_debug_low25_hybrid_single_year(var_key)

    print("\n" + "-" * 70)
    print("[STEP 1] Build debug LOW25 nc for EPFLUX")
    print("-" * 70)
    build_debug_low25_epflux_single_year()

    # 2) plot anomaly vs full coupled
    for var_key in ["U", "T", "O3", "EPFLUX"]:
        print("\n" + "-" * 70)
        print(f"[STEP 2] Plot COUPLED debug anomaly for {var_key}")
        print("-" * 70)
        plot_coupled_debug_anomaly(var_key)

    print("\n✅ Done. Check:")
    print(f"  NCs  -> {NC_OUT}")
    print(f"  PNGs -> {PLOT_OUT}")

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
import glob
import numpy as np
import xarray as xr

ROOT = "/mnt/soclim0/public_data/weiji"
TXT_ROOT = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"

EXP_CONFIG = {
    "B2000WCN": {
        "data_dir": os.path.join(ROOT, "B2000WCN001002", "O3"),
        "pattern": "B2000WCN.sample.cam.h3.*.O3.nc",
        "txt_dir": os.path.join(TXT_ROOT, "B2000WCN"),
        "run2_file_year_min": 105,
        "run2_file_year_max": 210,
        "offset": 103,
    },
    "NOCOUPL": {
        "data_dir": os.path.join(ROOT, "B2000WCN_NOCOUPL001002", "O3"),
        "pattern": "B2000WCN.NOCOUPL.sample.cam.h3.*.O3.nc",
        "txt_dir": os.path.join(TXT_ROOT, "B2000WCN_NOCOUPL"),
        "run2_file_year_min": 105,
        "run2_file_year_max": 205,
        "offset": 103,
    },
}


def parse_file_year(path):
    base = os.path.basename(path)
    nums = re.findall(r"(?<!\d)(\d{4})(?!\d)", base)
    if not nums:
        raise ValueError(f"Cannot parse year from file name: {base}")
    return int(nums[-1])


def get_time_years(ds):
    years = np.array([int(t.year) for t in ds.time.values], dtype=int)
    return years


def summarize_years(years):
    uniq = np.unique(years)
    return int(uniq.min()), int(uniq.max()), len(uniq)


def load_txt_years(txt_path):
    if not os.path.exists(txt_path):
        return []
    arr = np.loadtxt(txt_path, dtype=int)
    if np.ndim(arr) == 0:
        return [int(arr)]
    return [int(x) for x in arr.tolist()]


def inspect_txt_generation_mapping(exp_name, cfg):
    """
    模拟 low25 txt 生成代码的时间轴语义：
    - run001: 不偏移
    - run002: time.year + 103
    - 再看最终物理年覆盖
    """
    print("=" * 100)
    print(f"[TXT-SIDE] Inspecting {exp_name}")
    print("=" * 100)

    files = sorted(glob.glob(os.path.join(cfg["data_dir"], cfg["pattern"])))
    if not files:
        print(f"[ERROR] No files found: {cfg['data_dir']}")
        return None

    rows = []
    physical_year_to_file = {}

    for f in files:
        fy = parse_file_year(f)
        with xr.open_dataset(f, decode_times=True) as ds:
            raw_years = get_time_years(ds)

        raw_min, raw_max, _ = summarize_years(raw_years)

        if fy >= cfg["run2_file_year_min"]:
            shifted_years = raw_years + cfg["offset"]
            shifted = True
        else:
            shifted_years = raw_years.copy()
            shifted = False

        phy_min, phy_max, _ = summarize_years(shifted_years)

        rows.append({
            "file_year": fy,
            "shifted": shifted,
            "raw_min": raw_min,
            "raw_max": raw_max,
            "phy_min": phy_min,
            "phy_max": phy_max,
            "path": f,
        })

        for y in np.unique(shifted_years):
            physical_year_to_file.setdefault(int(y), []).append({
                "file_year": fy,
                "path": f,
                "shifted": shifted,
                "raw_years": (raw_min, raw_max),
                "physical_years": (phy_min, phy_max),
            })

    for r in rows[:10]:
        print(
            f"file_year={r['file_year']:4d} | shifted={str(r['shifted']):5s} | "
            f"raw={r['raw_min']:4d}-{r['raw_max']:4d} | "
            f"physical={r['phy_min']:4d}-{r['phy_max']:4d} | "
            f"{os.path.basename(r['path'])}"
        )
    if len(rows) > 10:
        print(f"... ({len(rows)} files total)")

    txt_path = os.path.join(cfg["txt_dir"], "low25pct_years_30_70hPa.txt")
    txt_years = load_txt_years(txt_path)

    print(f"\n[TXT years from file] {txt_path}")
    print(txt_years)

    print("\n[Physical year coverage in TXT-side mapping for low25 years]")
    for y in txt_years:
        hits = physical_year_to_file.get(y, [])
        if not hits:
            print(f"  year {y}: NOT FOUND in TXT-side physical timeline")
        else:
            print(f"  year {y}:")
            for h in hits:
                print(
                    f"    <- file_year={h['file_year']:4d}, shifted={h['shifted']}, "
                    f"raw={h['raw_years'][0]}-{h['raw_years'][1]}, "
                    f"physical={h['physical_years'][0]}-{h['physical_years'][1]}, "
                    f"{os.path.basename(h['path'])}"
                )

    return {
        "rows": rows,
        "physical_year_to_file": physical_year_to_file,
        "txt_years": txt_years,
    }


def get_model_hybrid_part_debug(cfg, y, part):
    """
    完全模拟 low25 climatology 代码里的 get_model_hybrid_part() 的文件查找和时间偏移逻辑
    这里只做年份映射调试，不做变量插值。
    """
    files = glob.glob(os.path.join(cfg["data_dir"], f"*{y:04d}*.nc"))
    files = sorted(files)

    if not files:
        return {
            "target_y": y,
            "part": part,
            "found": False,
            "path": None,
            "file_year": None,
            "raw_year_range": None,
            "physical_year_range": None,
            "selected_day_count": None,
            "selected_unique_years": None,
        }

    f = files[0]
    fy = parse_file_year(f)

    with xr.open_dataset(f, decode_times=True) as ds:
        raw_years = np.array([int(t.year) for t in ds.time.values], dtype=int)

        if fy >= cfg["run2_file_year_min"]:
            shifted_years = raw_years + cfg["offset"]
            shifted = True
        else:
            shifted_years = raw_years.copy()
            shifted = False

        # 完全模拟原代码 drop_feb29 后再取 prev/curr
        keep_idx = []
        months = []
        days = []
        for i, t in enumerate(ds.time.values):
            m = int(t.month)
            d = int(t.day)
            if not (m == 2 and d == 29):
                keep_idx.append(i)
                months.append(m)
                days.append(d)

        keep_idx = np.array(keep_idx, dtype=int)
        shifted_years_kept = shifted_years[keep_idx]
        months = np.array(months, dtype=int)
        days = np.array(days, dtype=int)

        if len(keep_idx) != 365:
            return {
                "target_y": y,
                "part": part,
                "found": True,
                "path": f,
                "file_year": fy,
                "shifted": shifted,
                "raw_year_range": (int(raw_years.min()), int(raw_years.max())),
                "physical_year_range": (int(shifted_years.min()), int(shifted_years.max())),
                "selected_day_count": f"BAD_{len(keep_idx)}",
                "selected_unique_years": None,
            }

        if part == "prev":
            sel = np.arange(273, 365)
        elif part == "curr":
            sel = np.arange(0, 273)
        else:
            raise ValueError("part must be 'prev' or 'curr'")

        sel_years = shifted_years_kept[sel]
        sel_months = months[sel]
        sel_days = days[sel]

        return {
            "target_y": y,
            "part": part,
            "found": True,
            "path": f,
            "file_year": fy,
            "shifted": shifted,
            "raw_year_range": (int(raw_years.min()), int(raw_years.max())),
            "physical_year_range": (int(shifted_years.min()), int(shifted_years.max())),
            "selected_day_count": len(sel),
            "selected_unique_years": sorted(np.unique(sel_years).tolist()),
            "selected_first_date_like": f"Y{int(sel_years[0]):04d}-{int(sel_months[0]):02d}-{int(sel_days[0]):02d}",
            "selected_last_date_like": f"Y{int(sel_years[-1]):04d}-{int(sel_months[-1]):02d}-{int(sel_days[-1]):02d}",
        }


def inspect_climatology_mapping(exp_name, cfg, txt_years):
    """
    模拟 low25 climatology 代码对 txt 年份的逐年读取。
    """
    print("\n" + "=" * 100)
    print(f"[CLIM-SIDE] Inspecting {exp_name}")
    print("=" * 100)

    results = []

    for y in txt_years:
        prev_info = get_model_hybrid_part_debug(cfg, y - 1, "prev")
        curr_info = get_model_hybrid_part_debug(cfg, y, "curr")

        results.append((y, prev_info, curr_info))

        print(f"\n[target physical year from TXT] y = {y}")
        print("  prev side:")
        print(f"    found={prev_info['found']}")
        if prev_info["found"]:
            print(f"    file={os.path.basename(prev_info['path'])}")
            print(f"    file_year={prev_info['file_year']}")
            print(f"    shifted={prev_info['shifted']}")
            print(f"    raw_year_range={prev_info['raw_year_range']}")
            print(f"    physical_year_range={prev_info['physical_year_range']}")
            print(f"    selected_unique_years={prev_info['selected_unique_years']}")
            print(f"    selected_range={prev_info.get('selected_first_date_like')} -> {prev_info.get('selected_last_date_like')}")

        print("  curr side:")
        print(f"    found={curr_info['found']}")
        if curr_info["found"]:
            print(f"    file={os.path.basename(curr_info['path'])}")
            print(f"    file_year={curr_info['file_year']}")
            print(f"    shifted={curr_info['shifted']}")
            print(f"    raw_year_range={curr_info['raw_year_range']}")
            print(f"    physical_year_range={curr_info['physical_year_range']}")
            print(f"    selected_unique_years={curr_info['selected_unique_years']}")
            print(f"    selected_range={curr_info.get('selected_first_date_like')} -> {curr_info.get('selected_last_date_like')}")

    return results


def compare_txt_and_clim(exp_name, txt_info, clim_results):
    """
    判断 txt-side 物理年 y 和 clim-side 实际读取是否一致。
    对于目标 y，理论上应当：
      prev = 物理年 y-1 的 Oct-Dec
      curr = 物理年 y 的 Jan-Sep
    """
    print("\n" + "=" * 100)
    print(f"[FINAL CHECK] {exp_name}")
    print("=" * 100)

    bad = []

    for y, prev_info, curr_info in clim_results:
        prev_ok = (
            prev_info["found"]
            and prev_info["selected_unique_years"] == [y - 1]
            and prev_info.get("selected_first_date_like", "").startswith(f"Y{y-1:04d}-10-")
        )
        curr_ok = (
            curr_info["found"]
            and curr_info["selected_unique_years"] == [y]
            and curr_info.get("selected_first_date_like", "").startswith(f"Y{y:04d}-01-")
        )

        txt_has_y = y in txt_info["physical_year_to_file"]

        status = txt_has_y and prev_ok and curr_ok

        print(
            f"y={y:4d} | txt_has_y={txt_has_y} | prev_ok={prev_ok} | curr_ok={curr_ok} | overall={status}"
        )

        if not status:
            bad.append({
                "y": y,
                "txt_has_y": txt_has_y,
                "prev_info": prev_info,
                "curr_info": curr_info,
            })

    print("\n" + "-" * 100)
    if bad:
        print(f"[BAD CASES] {len(bad)} problematic years found:")
        for b in bad:
            print(f"\n  y = {b['y']}")
            print(f"    txt_has_y = {b['txt_has_y']}")
            print(f"    prev_info = {b['prev_info']}")
            print(f"    curr_info = {b['curr_info']}")
    else:
        print("[OK] All low25 years are mapped consistently between TXT-side and CLIM-side.")

    return bad


def main():
    for exp_name, cfg in EXP_CONFIG.items():
        txt_info = inspect_txt_generation_mapping(exp_name, cfg)
        if txt_info is None:
            continue

        txt_years = txt_info["txt_years"]
        clim_results = inspect_climatology_mapping(exp_name, cfg, txt_years)
        compare_txt_and_clim(exp_name, txt_info, clim_results)


if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import numpy as np
import xarray as xr

ROOT = "/mnt/soclim0/public_data/weiji"

FILES = {
    "B2000WCN": os.path.join(ROOT, "B2000WCN001002", "EPflux_daily", "EPFLUX_206yr_Daily_Series_Full.nc"),
    "NOCOUPL": os.path.join(ROOT, "B2000WCN_NOCOUPL001002", "EPflux_daily", "EPFLUX_205yr_Daily_Series_Full.nc"),
}

def safe_get_ymd(tt):
    """
    尽量兼容:
    - cftime
    - python datetime
    - numpy.datetime64
    """
    # cftime / datetime-like
    if hasattr(tt, "year") and hasattr(tt, "month") and hasattr(tt, "day"):
        return int(tt.year), int(tt.month), int(tt.day), "attr"

    # numpy.datetime64
    try:
        s = np.datetime_as_string(tt, unit="D")
        y, m, d = s.split("-")
        return int(y), int(m), int(d), "np.datetime_as_string"
    except Exception:
        pass

    # 最后兜底
    s = str(tt)
    if "T" in s:
        s = s.split("T")[0]
    elif " " in s:
        s = s.split(" ")[0]
    y, m, d = s.split("-")
    return int(y), int(m), int(d), "str_fallback"


def inspect_one(exp, path):
    print("=" * 120)
    print(f"[EPFLUX TIME DEBUG] {exp}")
    print("=" * 120)
    print(f"FILE: {path}")

    if not os.path.exists(path):
        print("❌ file not found")
        return

    # ------------------------------------------------------------
    # A. decode_times=True
    # ------------------------------------------------------------
    print("\n" + "-" * 100)
    print("[A] open_dataset(decode_times=True)")
    print("-" * 100)
    with xr.open_dataset(path, decode_times=True) as ds:
        print(ds)

        time = ds["time"]
        vals = time.values

        print("\n[time basic]")
        print("dtype:", vals.dtype)
        print("shape:", vals.shape)
        print("type(time.values):", type(vals))
        if len(vals) > 0:
            print("type(time.values[0]):", type(vals[0]))
            print("time[0]:", vals[0])
            print("time[-1]:", vals[-1])

        print("\n[time attrs]")
        print("attrs:", dict(time.attrs))
        print("encoding:", dict(time.encoding))

        # sample parsing
        print("\n[sample safe_get_ymd]")
        sample_indices = [0, 1, 2, len(vals)//2, len(vals)-3, len(vals)-2, len(vals)-1]
        sample_indices = sorted(set([i for i in sample_indices if 0 <= i < len(vals)]))
        for i in sample_indices:
            try:
                y, m, d, method = safe_get_ymd(vals[i])
                print(f"  idx={i:6d}  raw={vals[i]!r}  ->  {y:04d}-{m:02d}-{d:02d}  via {method}")
            except Exception as e:
                print(f"  idx={i:6d}  raw={vals[i]!r}  ->  FAILED: {type(e).__name__}: {e}")

        # full parse
        print("\n[full parse test]")
        try:
            parsed = [safe_get_ymd(t) for t in vals]
            years = np.array([p[0] for p in parsed], dtype=int)
            months = np.array([p[1] for p in parsed], dtype=int)
            days = np.array([p[2] for p in parsed], dtype=int)
            methods = sorted(set(p[3] for p in parsed))
            print("✅ full parse success")
            print("methods used:", methods)
            print("year min/max:", years.min(), years.max())
            print("n unique years:", len(np.unique(years)))

            # counts
            keep_mask = ~((months == 2) & (days == 29))
            years_noleap = years[keep_mask]
            uniq = np.unique(years)
            bad = []
            for y in uniq:
                c_raw = int((years == y).sum())
                c_nl = int((years_noleap == y).sum())
                if c_nl != 365:
                    bad.append((int(y), c_raw, c_nl))
            print("bad noleap years:", bad[:20], f"{'...' if len(bad) > 20 else ''}")

            # duplicates
            tstr = np.array([f"{y:04d}-{m:02d}-{d:02d}" for y, m, d in zip(years, months, days)])
            uniq_t, counts = np.unique(tstr, return_counts=True)
            ndups = int((counts > 1).sum())
            print("duplicated timestamps:", ndups)

            # monotonic
            is_sorted = bool(np.all(tstr[:-1] <= tstr[1:]))
            print("monotonic sorted:", is_sorted)

            # around 103-105
            idx = np.where((years >= 103) & (years <= 105))[0]
            print("\n[sample around 103-105]")
            if len(idx) > 0:
                i0 = max(idx[0] - 3, 0)
                i1 = min(idx[0] + 8, len(vals))
                for j in range(i0, i1):
                    print(f"  {j:8d}  {tstr[j]}")
            else:
                print("  no year 103-105 found")

        except Exception as e:
            print("❌ full parse failed:", type(e).__name__, e)

        # dt accessor availability
        print("\n[xarray dt accessor test]")
        try:
            yy = time.dt.year.values
            mm = time.dt.month.values
            dd = time.dt.day.values
            print("✅ .dt works")
            print("year min/max via .dt:", int(np.nanmin(yy)), int(np.nanmax(yy)))
            print("sample .dt year:", yy[:5], "...", yy[-5:])
        except Exception as e:
            print("❌ .dt failed:", type(e).__name__, e)

    # ------------------------------------------------------------
    # B. decode_times=False
    # ------------------------------------------------------------
    print("\n" + "-" * 100)
    print("[B] open_dataset(decode_times=False)")
    print("-" * 100)
    with xr.open_dataset(path, decode_times=False) as ds0:
        print(ds0)

        time0 = ds0["time"]
        vals0 = time0.values

        print("\n[raw time basic]")
        print("dtype:", vals0.dtype)
        print("shape:", vals0.shape)
        print("type(time.values):", type(vals0))
        if len(vals0) > 0:
            print("type(time.values[0]):", type(vals0[0]))
            print("time[0]:", vals0[0])
            print("time[-1]:", vals0[-1])

        print("\n[raw time attrs]")
        print("attrs:", dict(time0.attrs))
        print("encoding:", dict(time0.encoding))

        # try manual CF decode
        print("\n[manual CF decode test]")
        try:
            decoded = xr.decode_cf(ds0)
            tdec = decoded["time"].values
            print("✅ xr.decode_cf(ds0) success")
            print("decoded type:", type(tdec))
            if len(tdec) > 0:
                print("decoded element type:", type(tdec[0]))
                print("decoded first:", tdec[0])
                print("decoded last :", tdec[-1])
        except Exception as e:
            print("❌ xr.decode_cf(ds0) failed:", type(e).__name__, e)

    print("\n")


def main():
    for exp, path in FILES.items():
        inspect_one(exp, path)


if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
import gc
import glob
import hashlib
import warnings
import numpy as np
import xarray as xr

warnings.filterwarnings("ignore")

try:
    from aostools_functions import ComputeEPfluxDiv
except ImportError:
    raise ImportError("请确保 aostools_functions.py 与此脚本在同一目录，且可导入 ComputeEPfluxDiv")

# ============================================================
# Paths
# ============================================================
ROOT = "/mnt/soclim0/public_data/weiji"

PATHS = {
    "BWCN": os.path.join(ROOT, "BWCN"),
    "B2000WCN": os.path.join(ROOT, "B2000WCN001002"),
    "NOCOUPL": os.path.join(ROOT, "B2000WCN_NOCOUPL001002"),
}

RAW_DIRS = {
    "BWCN": {
        "U": os.path.join(PATHS["BWCN"], "U"),
        "V": os.path.join(PATHS["BWCN"], "V"),
        "T": os.path.join(PATHS["BWCN"], "T"),
        "OMEGA": os.path.join(PATHS["BWCN"], "OMEGA"),
    },
    "B2000WCN": {
        "U": os.path.join(PATHS["B2000WCN"], "U"),
        "V": os.path.join(PATHS["B2000WCN"], "V"),
        "T": os.path.join(PATHS["B2000WCN"], "T"),
        "OMEGA": os.path.join(PATHS["B2000WCN"], "OMEGA"),
    },
    "NOCOUPL": {
        "U": os.path.join(PATHS["NOCOUPL"], "U"),
        "V": os.path.join(PATHS["NOCOUPL"], "V"),
        "T": os.path.join(PATHS["NOCOUPL"], "T"),
        "OMEGA": os.path.join(PATHS["NOCOUPL"], "OMEGA"),
    },
}

EPFLUX_FILES = {
    "BWCN": os.path.join(PATHS["BWCN"], "EPflux_daily", "EPFLUX_BWCN_Daily_24yr.nc"),
    "B2000WCN": os.path.join(PATHS["B2000WCN"], "EPflux_daily", "EPFLUX_206yr_Daily_Series_Full.nc"),
    "NOCOUPL": os.path.join(PATHS["NOCOUPL"], "EPflux_daily", "EPFLUX_205yr_Daily_Series_Full.nc"),
}

# ============================================================
# Pressure levels
# ============================================================
PLEV_STD_PA = np.array([
    10, 50, 100, 200, 300, 500, 1000, 2000, 3000, 5000, 7000, 10000, 15000,
    20000, 25000, 30000, 40000, 50000, 60000, 70000, 85000, 92500, 100000
], dtype=float)
PLEV_STD_HPA = PLEV_STD_PA / 100.0

TARGET_PLEV_HPA = np.array(
    [100.0, 70.0, 50.0, 40.0, 30.0, 20.0, 10.0, 7.0, 5.0, 4.0, 3.0, 2.0, 1.0],
    dtype=np.float64
)

LAT_RANGE = (40.0, 80.0)
DO_UBAR = False
WAVE = -1

YEAR_RE = re.compile(r"\.cam\.h3\.(\d{4})\.")

# ============================================================
# Helpers
# ============================================================
def md5_array(arr):
    arr = np.ascontiguousarray(arr.astype(np.float64))
    return hashlib.md5(arr.tobytes()).hexdigest()

def safe_get_ymd(tt):
    if hasattr(tt, "year") and hasattr(tt, "month") and hasattr(tt, "day"):
        return int(tt.year), int(tt.month), int(tt.day)
    s = np.datetime_as_string(tt, unit="D")
    y, m, d = s.split("-")
    return int(y), int(m), int(d)

def format_time_range(time_vals):
    if len(time_vals) == 0:
        return "None -> None"
    y1, m1, d1 = safe_get_ymd(time_vals[0])
    y2, m2, d2 = safe_get_ymd(time_vals[-1])
    return f"{y1:04d}-{m1:02d}-{d1:02d} -> {y2:04d}-{m2:02d}-{d2:02d}"

def compare_arrays(a, b, atol=1e-8, rtol=1e-8):
    if a.shape != b.shape:
        return {
            "shape_equal": False,
            "max_abs_diff": np.nan,
            "mean_abs_diff": np.nan,
            "allclose": False,
            "same_md5": False,
        }
    diff = np.abs(a - b)
    return {
        "shape_equal": True,
        "max_abs_diff": np.nanmax(diff),
        "mean_abs_diff": np.nanmean(diff),
        "allclose": np.allclose(a, b, atol=atol, rtol=rtol, equal_nan=True),
        "same_md5": (md5_array(a) == md5_array(b)),
    }

def shift_time_like_blockA(ds, offset_years=103):
    old_times = ds.time.values
    new_times = [t.replace(year=t.year + offset_years) for t in old_times]
    return ds.assign_coords(time=new_times)

def physical_year_to_file_year(exp_name, y_phys):
    y_phys = int(y_phys)

    if exp_name == "BWCN":
        return y_phys

    if exp_name == "B2000WCN":
        if 1 <= y_phys <= 103:
            return y_phys
        elif 104 <= y_phys <= 209:
            return y_phys + 1
        else:
            return None

    if exp_name == "NOCOUPL":
        if 1 <= y_phys <= 103:
            return y_phys
        elif 104 <= y_phys <= 204:
            return y_phys + 1
        else:
            return None

    return y_phys

def file_for(exp_name, var_name, file_year):
    y = f"{file_year:04d}"

    if exp_name == "BWCN":
        return os.path.join(RAW_DIRS[exp_name][var_name], f"BWCN.cam.h3.{y}.{var_name}.nc")
    if exp_name == "B2000WCN":
        return os.path.join(RAW_DIRS[exp_name][var_name], f"B2000WCN.sample.cam.h3.{y}.{var_name}.nc")
    if exp_name == "NOCOUPL":
        return os.path.join(RAW_DIRS[exp_name][var_name], f"B2000WCN.NOCOUPL.sample.cam.h3.{y}.{var_name}.nc")

    raise ValueError(exp_name)

def compute_pressure_mid(ds):
    return ds["hyam"] * ds["P0"] + ds["hybm"] * ds["PS"]

def interp_profile_logp_4d(v_hyb, p_hyb, p_tgt_pa):
    p_tgt_pa = np.asarray(p_tgt_pa, float)

    def _interp_col(vcol, pcol):
        m = np.isfinite(vcol) & np.isfinite(pcol) & (pcol > 0)
        if m.sum() < 2:
            return np.full(p_tgt_pa.shape, np.nan, float)
        p_use, v_use = pcol[m], vcol[m]
        idx = np.argsort(p_use)
        return np.interp(
            np.log(p_tgt_pa),
            np.log(p_use[idx]),
            v_use[idx],
            left=np.nan,
            right=np.nan
        )

    dask_gufunc_kwargs = {"output_sizes": {"plev": len(p_tgt_pa)}}
    out = xr.apply_ufunc(
        _interp_col, v_hyb, p_hyb,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[["plev"]],
        vectorize=True, dask="parallelized", output_dtypes=[float],
        dask_gufunc_kwargs=dask_gufunc_kwargs
    )
    return out.assign_coords(plev=("plev", p_tgt_pa))

def sel_latband(da, lat0=40., lat1=80., lat_name="lat"):
    lat = da[lat_name]
    dec = float(lat.values[0]) > float(lat.values[-1])
    slc = slice(lat1, lat0) if dec else slice(lat0, lat1)
    out = da.sel({lat_name: slc})
    if out.sizes.get(lat_name, 0) == 0:
        raise ValueError("Empty lat selection.")
    return out

def coslat_weighted_mean(da, lat_name="lat"):
    lat = da[lat_name]
    w = np.cos(np.deg2rad(lat))
    return da.weighted(w).mean(lat_name, skipna=True)

def interp_logp_2d_to_target(data_2d, src_plev_hpa, target_plev_hpa):
    src = np.asarray(src_plev_hpa, dtype=np.float64)
    tgt = np.asarray(target_plev_hpa, dtype=np.float64)
    arr = np.asarray(data_2d, dtype=np.float64)

    order = np.argsort(src)
    src = src[order]
    arr = arr[:, order]

    out = np.full((arr.shape[0], len(tgt)), np.nan, dtype=np.float64)
    lp_tgt = np.log(tgt)

    for i in range(arr.shape[0]):
        v = arr[i, :]
        m = np.isfinite(v) & np.isfinite(src) & (src > 0)
        if np.sum(m) < 2:
            continue
        s = src[m]
        vv = v[m]
        oo = np.argsort(s)
        s = s[oo]
        vv = vv[oo]
        out[i, :] = np.interp(lp_tgt, np.log(s), vv, left=np.nan, right=np.nan)

    return out

def to_13levels(data_2d, src_plev_hpa):
    return interp_logp_2d_to_target(data_2d, src_plev_hpa, TARGET_PLEV_HPA)

def jan1_jan10_mask(time_vals, year):
    return np.array([
        (safe_get_ymd(t)[0] == year and safe_get_ymd(t)[1] == 1 and 1 <= safe_get_ymd(t)[2] <= 10)
        for t in time_vals
    ], dtype=bool)

# ============================================================
# RAW recompute: only Jan 1-10
# ============================================================
def recompute_raw_epflux_jan10_to13(exp_name, target_y):
    fy = physical_year_to_file_year(exp_name, target_y)
    if fy is None:
        raise ValueError(f"Invalid physical year for {exp_name}: {target_y}")

    fU = file_for(exp_name, "U", fy)
    fV = file_for(exp_name, "V", fy)
    fT = file_for(exp_name, "T", fy)
    fW = file_for(exp_name, "OMEGA", fy)

    for fp in [fU, fV, fT, fW]:
        if not os.path.exists(fp):
            raise FileNotFoundError(fp)

    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)

    with xr.open_dataset(fU, decode_times=time_coder) as dsU, \
         xr.open_dataset(fV, decode_times=time_coder) as dsV, \
         xr.open_dataset(fT, decode_times=time_coder) as dsT, \
         xr.open_dataset(fW, decode_times=time_coder) as dsW:

        if exp_name in ["B2000WCN", "NOCOUPL"] and fy >= 105:
            dsU = shift_time_like_blockA(dsU, 103)
            dsV = shift_time_like_blockA(dsV, 103)
            dsT = shift_time_like_blockA(dsT, 103)
            dsW = shift_time_like_blockA(dsW, 103)

        time_vals_full = dsU["time"].values
        mask10 = jan1_jan10_mask(time_vals_full, target_y)
        if mask10.sum() != 10:
            raise ValueError(f"{exp_name} year={target_y} raw Jan1-10 count != 10, got {mask10.sum()}")

        dsU = dsU.isel(time=mask10)
        dsV = dsV.isel(time=mask10)
        dsT = dsT.isel(time=mask10)
        dsW = dsW.isel(time=mask10)

        p_mid = compute_pressure_mid(dsU)

        u_std = interp_profile_logp_4d(dsU["U"], p_mid, PLEV_STD_PA)
        v_std = interp_profile_logp_4d(dsV["V"], p_mid, PLEV_STD_PA)
        t_std = interp_profile_logp_4d(dsT["T"], p_mid, PLEV_STD_PA)
        w_std = interp_profile_logp_4d(dsW["OMEGA"] / 100.0, p_mid, PLEV_STD_PA)

        u_np = u_std.transpose("time", "plev", "lat", "lon").values
        v_np = v_std.transpose("time", "plev", "lat", "lon").values
        t_np = t_std.transpose("time", "plev", "lat", "lon").values
        w_np = w_std.transpose("time", "plev", "lat", "lon").values
        lat_np = dsU["lat"].values

        _, ep2, _, _ = ComputeEPfluxDiv(
            lat=lat_np,
            pres=PLEV_STD_HPA,
            u=u_np, v=v_np, t=t_np, w=w_np,
            do_ubar=DO_UBAR, wave=WAVE
        )

        da_ep2 = xr.DataArray(
            ep2,
            coords={"time": dsU["time"], "plev": PLEV_STD_HPA, "lat": lat_np},
            dims=("time", "plev", "lat"),
            name="Fz"
        )

        da_sub = sel_latband(da_ep2, LAT_RANGE[0], LAT_RANGE[1], "lat")
        da_mean = coslat_weighted_mean(da_sub).compute()   # (10, 23)

        data_23 = da_mean.values.astype(np.float64)
        time_vals = da_mean["time"].values
        data_13 = to_13levels(data_23, da_mean["plev"].values.astype(np.float64))

        out = {
            "physical_year": target_y,
            "file_year": fy,
            "sample_file": fU,
            "time": time_vals,
            "data_23": data_23,
            "data_13": data_13,
        }

        del u_std, v_std, t_std, w_std, da_ep2, da_sub, da_mean, u_np, v_np, t_np, w_np
        gc.collect()

        return out

# ============================================================
# PRECOMPUTED single-file: only Jan 1-10
# ============================================================
def get_precomputed_epflux_jan10_to13(exp_name, target_y):
    fp = EPFLUX_FILES[exp_name]
    if not os.path.exists(fp):
        raise FileNotFoundError(fp)

    ds = xr.open_dataset(fp, decode_times=True)
    try:
        da = ds["Fz"]

        if "lat" in da.dims:
            da = da.mean("lat", skipna=True)

        da = da.transpose("time", "plev")
        time_vals = da["time"].values
        mask10 = jan1_jan10_mask(time_vals, target_y)

        if mask10.sum() != 10:
            raise ValueError(f"{exp_name} precomputed year={target_y} Jan1-10 count != 10, got {mask10.sum()}")

        da10 = da.isel(time=mask10)
        data_23 = da10.values.astype(np.float64)
        time10 = da10["time"].values
        data_13 = to_13levels(data_23, da10["plev"].values.astype(np.float64))

        return {
            "time": time10,
            "data_23": data_23,
            "data_13": data_13,
        }
    finally:
        ds.close()

# ============================================================
# Main
# ============================================================
def main():
    # 你可以自己改测试年
    TEST_YEARS = {
        "BWCN": [8, 13],
        "B2000WCN": [2, 109],
        "NOCOUPL": [2, 106],
    }

    print("=" * 100)
    print("EPFLUX QUICK CONSISTENCY DEBUG: compare Jan 1-10 only, on unified 13 pressure levels")
    print("=" * 100)

    for exp in ["BWCN", "B2000WCN", "NOCOUPL"]:
        print("\n" + "-" * 100)
        print(f"[{exp}] raw recompute vs precomputed single-file (Jan 1-10 only, 13 levels)")
        print("-" * 100)

        for y in TEST_YEARS[exp]:
            print(f"\n[{exp}] target physical year = {y}")

            raw = recompute_raw_epflux_jan10_to13(exp, y)
            pre = get_precomputed_epflux_jan10_to13(exp, y)

            print("  [RAW Jan1-10]")
            print(f"    physical year = {y}")
            print(f"    mapped file_year = {raw['file_year']}")
            print(f"    sample file = {raw['sample_file']}")
            print(f"    range = {format_time_range(raw['time'])}")
            print(f"    shape = {raw['data_13'].shape}")
            print(f"    md5   = {md5_array(raw['data_13'])}")

            print("  [PRECOMPUTED Jan1-10]")
            print(f"    range = {format_time_range(pre['time'])}")
            print(f"    shape = {pre['data_13'].shape}")
            print(f"    md5   = {md5_array(pre['data_13'])}")

            cmpres = compare_arrays(raw["data_13"], pre["data_13"], atol=1e-8, rtol=1e-8)

            print("  [COMPARE]")
            print(f"    max abs diff  = {cmpres['max_abs_diff']:.6e}")
            print(f"    mean abs diff = {cmpres['mean_abs_diff']:.6e}")
            print(f"    allclose      = {cmpres['allclose']}")
            print(f"    same md5      = {cmpres['same_md5']}")

            gc.collect()

if __name__ == "__main__":
    main()